In [1]:
# Batch 0 / Cell 1 - Imports, project root, and output paths
import json
import math
import os
import re
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

NOTEBOOK_ID = "26_stable_diffusion_report_generation"
NOTEBOOK_TITLE = "Stable Diffusion Report Generation"
REPORT_SUBJECT = "Stable Diffusion restoration report"
REPORT_VERSION = "refactor_v1"

def find_project_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start).resolve()
    candidates = [start, *start.parents]

    root_markers = [
        Path("tools") / "build_project_inventory.py",
        Path("src") / "restoration_eval",
        Path("outputs"),
        Path("notebooks"),
    ]

    for candidate in candidates:
        marker_hits = sum((candidate / marker).exists() for marker in root_markers)
        if marker_hits >= 2:
            return candidate

    return start

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

OUTPUT_ROOT = PROJECT_ROOT / "outputs" / NOTEBOOK_ID

OUTPUT_DIRS = {
    "root": OUTPUT_ROOT,
    "inventory": OUTPUT_ROOT / "inventory",
    "validation": OUTPUT_ROOT / "validation",
    "metrics": OUTPUT_ROOT / "metrics",
    "analysis": OUTPUT_ROOT / "analysis",
    "figures": OUTPUT_ROOT / "figures",
    "reports": OUTPUT_ROOT / "reports",
    "manifests": OUTPUT_ROOT / "manifests",
    "assets": OUTPUT_ROOT / "assets",
    "report_assets": OUTPUT_ROOT / "assets" / "report",
    "aggregate_figures": OUTPUT_ROOT / "figures" / "aggregate",
    "candidate_panels": OUTPUT_ROOT / "figures" / "candidate_panels",
    "difference_map_panels": OUTPUT_ROOT / "figures" / "difference_map_panels",
}

for directory in OUTPUT_DIRS.values():
    directory.mkdir(parents=True, exist_ok=True)

BATCH0_INVENTORY_SNAPSHOT_PATH = OUTPUT_DIRS["inventory"] / "batch0_project_inventory_snapshot.csv"
BATCH0_VALIDATION_PATH = OUTPUT_DIRS["validation"] / "batch0_validation.csv"

STAGE_MANIFEST_PATH = OUTPUT_DIRS["manifests"] / "stable_diffusion_report_stage_manifest.json"
FINAL_ARTIFACT_INDEX_PATH = OUTPUT_DIRS["manifests"] / "stable_diffusion_report_artifact_index.csv"
FINAL_HANDOFF_MANIFEST_PATH = OUTPUT_DIRS["manifests"] / "stable_diffusion_report_handoff_manifest.json"

print("Notebook:", NOTEBOOK_ID)
print("Project root:", PROJECT_ROOT)
print("Output root:", OUTPUT_ROOT)
print("Batch 0 inventory snapshot:", BATCH0_INVENTORY_SNAPSHOT_PATH.relative_to(PROJECT_ROOT))
print("Batch 0 validation:", BATCH0_VALIDATION_PATH.relative_to(PROJECT_ROOT))

Notebook: 26_stable_diffusion_report_generation
Project root: D:\Masters\FH\Thesis\painting-restoration-eval
Output root: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\26_stable_diffusion_report_generation
Batch 0 inventory snapshot: outputs\26_stable_diffusion_report_generation\inventory\batch0_project_inventory_snapshot.csv
Batch 0 validation: outputs\26_stable_diffusion_report_generation\validation\batch0_validation.csv


In [2]:
# Batch 0 / Cell 2 - Shared utility functions
def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()

def rel(path_value: str | Path | None) -> str:
    if path_value is None:
        return ""

    path = Path(path_value)
    try:
        return str(path.resolve().relative_to(PROJECT_ROOT.resolve())).replace("\\", "/")
    except Exception:
        return str(path).replace("\\", "/")

def ensure_parent(path_value: str | Path) -> Path:
    path = Path(path_value)
    path.parent.mkdir(parents=True, exist_ok=True)
    return path

def read_json_if_exists(path_value: str | Path) -> dict:
    path = Path(path_value)
    if not path.is_file():
        return {}
    return json.loads(path.read_text(encoding="utf-8"))

def write_json(path_value: str | Path, payload: dict) -> None:
    path = ensure_parent(path_value)
    path.write_text(json.dumps(to_json_safe(payload), indent=2), encoding="utf-8")

def is_missing_scalar(value: Any) -> bool:
    if value is None:
        return True
    if isinstance(value, float):
        return math.isnan(value)
    if isinstance(value, (str, bool, int, Path, dict, list, tuple)):
        return False
    try:
        result = pd.isna(value)
        if isinstance(result, (bool, np.bool_)):
            return bool(result)
    except Exception:
        return False
    return False

def to_json_safe(value: Any) -> Any:
    if isinstance(value, Path):
        return rel(value)
    if isinstance(value, dict):
        return {str(key): to_json_safe(item) for key, item in value.items()}
    if isinstance(value, list):
        return [to_json_safe(item) for item in value]
    if isinstance(value, tuple):
        return [to_json_safe(item) for item in value]
    if isinstance(value, set):
        return sorted(to_json_safe(item) for item in value)
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        if np.isnan(value):
            return None
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if is_missing_scalar(value):
        return None
    return value

def bool_series(series: pd.Series) -> pd.Series:
    if series.dtype == bool:
        return series.fillna(False)

    normalized = series.fillna(False)
    if pd.api.types.is_numeric_dtype(normalized):
        return normalized.astype(float).ne(0)

    return (
        normalized.astype(str)
        .str.strip()
        .str.lower()
        .isin(["true", "1", "yes", "y", "passed", "pass", "ok"])
    )

def validation_row(
    check_name: str,
    actual: Any,
    expected: Any,
    passed: bool,
    failure_message: str,
    severity: str = "error",
) -> dict:
    return {
        "check_name": check_name,
        "severity": severity,
        "actual": json.dumps(to_json_safe(actual), ensure_ascii=False),
        "expected": json.dumps(to_json_safe(expected), ensure_ascii=False),
        "passed": bool(passed),
        "failure_message": "" if passed else failure_message,
        "checked_at_utc": utc_now_iso(),
    }

def safe_read_csv(path_value: str | Path, **kwargs) -> pd.DataFrame:
    path = Path(path_value)
    if not path.is_file():
        raise FileNotFoundError(f"CSV does not exist: {rel(path)}")
    return pd.read_csv(path, **kwargs)

def file_size_bytes(path_value: str | Path | None) -> int:
    if path_value is None:
        return 0
    path = Path(path_value)
    if path.is_file():
        return int(path.stat().st_size)
    if path.is_dir():
        return int(sum(file.stat().st_size for file in path.rglob("*") if file.is_file()))
    return 0

def csv_shape(path_value: str | Path) -> tuple[int | None, int | None]:
    path = Path(path_value)
    if not path.is_file():
        return None, None
    try:
        df = pd.read_csv(path)
        return int(len(df)), int(len(df.columns))
    except Exception:
        return None, None

def normalize_relative_path(value: Any) -> str:
    if is_missing_scalar(value):
        return ""
    return str(value).replace("\\", "/").lstrip("./")

def path_from_project(relative_path: str | Path) -> Path:
    path = Path(relative_path)
    if path.is_absolute():
        return path
    return PROJECT_ROOT / path

def display_path_table(paths: list[str | Path], title: str = "Paths") -> pd.DataFrame:
    rows = []
    for path_value in paths:
        path = path_from_project(path_value)
        rows.append(
            {
                "path": rel(path),
                "exists": path.exists(),
                "is_file": path.is_file(),
                "is_dir": path.is_dir(),
                "size_bytes": file_size_bytes(path),
            }
        )
    df = pd.DataFrame(rows)
    print(title)
    display(df)
    return df

print("Utility functions loaded.")

Utility functions loaded.


In [3]:
# Batch 0 / Cell 3 - Inventory discovery and snapshot
def inventory_candidate_paths() -> list[Path]:
    candidates = []

    preferred_paths = [
        PROJECT_ROOT / "outputs" / "inventory" / "project_file_inventory.csv",
        PROJECT_ROOT / "outputs" / "inventory" / "project_file_inventory_summary.csv",
    ]

    for path in preferred_paths:
        if path.is_file():
            candidates.append(path)

    upload_dir = PROJECT_ROOT / "upload"
    if upload_dir.is_dir():
        candidates.extend(sorted(upload_dir.glob("project_file_inventory*.csv")))
        candidates.extend(sorted(upload_dir.glob("outputs_inventory*.csv")))

    seen = set()
    unique_candidates = []
    for path in candidates:
        resolved = str(path.resolve())
        if resolved not in seen:
            seen.add(resolved)
            unique_candidates.append(path)

    return unique_candidates

def inspect_inventory_candidate(path: Path) -> dict:
    try:
        df = pd.read_csv(path)
        columns = list(df.columns)
        usable_path_column = "relative_path" in columns
        row_count = int(len(df))
        source_priority = 2 if rel(path) == "outputs/inventory/project_file_inventory.csv" else 1
        if "project_file_inventory" in path.name:
            source_priority += 1
        return {
            "path": path,
            "relative_path": rel(path),
            "exists": True,
            "readable": True,
            "row_count": row_count,
            "column_count": int(len(columns)),
            "columns": columns,
            "usable_path_column": usable_path_column,
            "source_priority": source_priority,
            "size_bytes": file_size_bytes(path),
            "last_modified_utc": datetime.fromtimestamp(path.stat().st_mtime, timezone.utc).isoformat(),
            "read_error": "",
        }
    except Exception as exc:
        return {
            "path": path,
            "relative_path": rel(path),
            "exists": path.exists(),
            "readable": False,
            "row_count": 0,
            "column_count": 0,
            "columns": [],
            "usable_path_column": False,
            "source_priority": 0,
            "size_bytes": file_size_bytes(path),
            "last_modified_utc": "",
            "read_error": str(exc),
        }

inventory_candidate_records = [
    inspect_inventory_candidate(path)
    for path in inventory_candidate_paths()
]

inventory_candidates_df = pd.DataFrame(inventory_candidate_records)

if inventory_candidates_df.empty:
    raise FileNotFoundError(
        "No project inventory CSV was found. Run tools/build_project_inventory.py first."
    )

usable_inventory_candidates_df = inventory_candidates_df.loc[
    bool_series(inventory_candidates_df["readable"])
    & bool_series(inventory_candidates_df["usable_path_column"])
].copy()

if usable_inventory_candidates_df.empty:
    display(inventory_candidates_df)
    raise RuntimeError("Inventory candidates were found, but none had a usable relative_path column.")

usable_inventory_candidates_df = usable_inventory_candidates_df.sort_values(
    ["row_count", "source_priority", "size_bytes", "last_modified_utc"],
    ascending=[False, False, False, False],
    kind="stable",
).reset_index(drop=True)

INVENTORY_SOURCE_PATH = Path(usable_inventory_candidates_df.loc[0, "path"])
inventory_raw_df = pd.read_csv(INVENTORY_SOURCE_PATH)

rename_map = {
    "last_modified_iso": "last_modified_utc",
    "LastWriteTime": "last_modified_utc",
    "depth": "path_depth",
    "csv_error": "csv_read_error",
    "image_error": "image_read_error",
    "type": "file_kind",
}

inventory_df = inventory_raw_df.rename(columns={key: value for key, value in rename_map.items() if key in inventory_raw_df.columns}).copy()

if "relative_path" not in inventory_df.columns:
    raise RuntimeError(f"Selected inventory has no relative_path column: {rel(INVENTORY_SOURCE_PATH)}")

inventory_df["relative_path"] = inventory_df["relative_path"].map(normalize_relative_path)

if "file_name" not in inventory_df.columns:
    inventory_df["file_name"] = inventory_df["relative_path"].map(lambda value: Path(value).name)

if "parent_dir" not in inventory_df.columns:
    inventory_df["parent_dir"] = inventory_df["relative_path"].map(lambda value: str(Path(value).parent).replace("\\", "/"))

if "extension" not in inventory_df.columns:
    inventory_df["extension"] = inventory_df["relative_path"].map(lambda value: Path(value).suffix.lower())

if "file_kind" not in inventory_df.columns:
    inventory_df["file_kind"] = inventory_df["extension"].map(
        lambda ext: "csv" if ext == ".csv" else "json" if ext == ".json" else "image" if ext in [".png", ".jpg", ".jpeg", ".webp"] else "other"
    )

for numeric_column in ["size_bytes", "csv_row_count", "csv_column_count", "image_width", "image_height", "path_depth"]:
    if numeric_column not in inventory_df.columns:
        inventory_df[numeric_column] = np.nan
    inventory_df[numeric_column] = pd.to_numeric(inventory_df[numeric_column], errors="coerce")

if "csv_columns" not in inventory_df.columns:
    inventory_df["csv_columns"] = ""

if "last_modified_utc" not in inventory_df.columns:
    inventory_df["last_modified_utc"] = ""

if "csv_read_error" not in inventory_df.columns:
    inventory_df["csv_read_error"] = ""

if "image_read_error" not in inventory_df.columns:
    inventory_df["image_read_error"] = ""

def notebook_number_from_path(relative_path: str) -> str:
    match = re.search(r"(?:^|/)(\d{2})_[^/]+", str(relative_path))
    return match.group(1) if match else ""

def stage_guess_from_path(relative_path: str) -> str:
    value = str(relative_path)
    match = re.search(r"(?:^|/)outputs/(\d{2}_[^/]+)/", value)
    if match:
        return match.group(1)
    match = re.search(r"(?:^|/)notebooks/(\d{2}_[^/]+)\.ipynb", value)
    if match:
        return match.group(1)
    return ""

def report_relevance_label(relative_path: str) -> str:
    value = str(relative_path).lower()

    if "26_stable_diffusion_report_generation" in value:
        return "current_notebook_output"
    if "21_stable_diffusion_restoration" in value:
        return "required_upstream_restoration"
    if "22_stable_diffusion_classical_metrics" in value:
        return "required_upstream_classical_metrics"
    if "23_stable_diffusion_difference_maps" in value:
        return "required_upstream_difference_maps"
    if "24_stable_diffusion_lpips_metrics" in value:
        return "required_upstream_lpips"
    if "25_stable_diffusion_feature_similarity" in value:
        return "required_upstream_feature_similarity"
    if "stable_diffusion" in value or "sdxl" in value or "diffusion" in value:
        return "stable_diffusion_related"
    return "other"

inventory_df["notebook_number_guess"] = inventory_df["relative_path"].map(notebook_number_from_path)
inventory_df["stage_guess"] = inventory_df["relative_path"].map(stage_guess_from_path)
inventory_df["report_relevance"] = inventory_df["relative_path"].map(report_relevance_label)
inventory_df["is_report_relevant"] = inventory_df["report_relevance"].ne("other")
inventory_df["inventory_source_path"] = rel(INVENTORY_SOURCE_PATH)
inventory_df["inventory_snapshot_created_at_utc"] = utc_now_iso()

preferred_column_order = [
    "inventory_snapshot_created_at_utc",
    "inventory_source_path",
    "relative_path",
    "file_name",
    "parent_dir",
    "extension",
    "file_kind",
    "report_relevance",
    "is_report_relevant",
    "notebook_number_guess",
    "stage_guess",
    "size_bytes",
    "last_modified_utc",
    "path_depth",
    "csv_row_count",
    "csv_column_count",
    "csv_columns",
    "csv_read_error",
    "image_width",
    "image_height",
    "image_mode",
    "image_read_error",
    "sha256_first_1mb",
]

ordered_columns = [column for column in preferred_column_order if column in inventory_df.columns]
remaining_columns = [column for column in inventory_df.columns if column not in ordered_columns]
batch0_inventory_snapshot_df = inventory_df[ordered_columns + remaining_columns].copy()

batch0_inventory_snapshot_df.to_csv(BATCH0_INVENTORY_SNAPSHOT_PATH, index=False)

print(f"Selected inventory source: {rel(INVENTORY_SOURCE_PATH)}")
print(f"Inventory source rows: {len(inventory_raw_df):,}")
print(f"Snapshot rows: {len(batch0_inventory_snapshot_df):,}")
print(f"Saved Batch 0 inventory snapshot: {rel(BATCH0_INVENTORY_SNAPSHOT_PATH)}")

print("\nInventory candidate ranking:")
display(
    usable_inventory_candidates_df[
        ["relative_path", "row_count", "column_count", "source_priority", "size_bytes", "last_modified_utc"]
    ]
)

print("\nReport-relevance counts:")
display(batch0_inventory_snapshot_df["report_relevance"].value_counts(dropna=False).rename_axis("report_relevance").reset_index(name="rows"))

Selected inventory source: outputs/inventory/project_file_inventory.csv
Inventory source rows: 10,914
Snapshot rows: 10,914
Saved Batch 0 inventory snapshot: outputs/26_stable_diffusion_report_generation/inventory/batch0_project_inventory_snapshot.csv

Inventory candidate ranking:


,relative_path,row_count,column_count,source_priority,size_bytes,last_modified_utc
0,outputs/inventory/project_file_inventory.csv,10914,17,3,3875287,2026-08-10T07:54:38.346996+00:00



Report-relevance counts:


,report_relevance,rows
0,required_upstream_difference_maps,6644
1,other,3088
2,stable_diffusion_related,956
3,required_upstream_feature_similarity,85
4,required_upstream_restoration,74
5,required_upstream_lpips,39
6,required_upstream_classical_metrics,28


In [4]:
# Batch 0 / Cell 4 - Input discovery helpers and expected upstream contract
def inventory_find(
    contains: str | list[str] | None = None,
    extension: str | None = None,
    relevance: str | list[str] | None = None,
    regex: str | None = None,
    case: bool = False,
    inventory: pd.DataFrame | None = None,
) -> pd.DataFrame:
    source_df = batch0_inventory_snapshot_df if inventory is None else inventory
    result_df = source_df.copy()

    path_series = result_df["relative_path"].astype(str)
    comparable_path_series = path_series if case else path_series.str.lower()

    if contains is not None:
        contains_values = [contains] if isinstance(contains, str) else list(contains)
        for value in contains_values:
            needle = str(value) if case else str(value).lower()
            result_df = result_df.loc[comparable_path_series.loc[result_df.index].str.contains(re.escape(needle), na=False)]

    if regex is not None:
        result_df = result_df.loc[path_series.loc[result_df.index].str.contains(regex, case=case, regex=True, na=False)]

    if extension is not None:
        normalized_extension = extension if str(extension).startswith(".") else f".{extension}"
        result_df = result_df.loc[result_df["extension"].astype(str).str.lower().eq(normalized_extension.lower())]

    if relevance is not None:
        relevance_values = [relevance] if isinstance(relevance, str) else list(relevance)
        result_df = result_df.loc[result_df["report_relevance"].isin(relevance_values)]

    return result_df.sort_values(["relative_path"], kind="stable").reset_index(drop=True)

def best_inventory_match(
    contains: str | list[str] | None = None,
    extension: str | None = None,
    relevance: str | list[str] | None = None,
    regex: str | None = None,
) -> str:
    matches_df = inventory_find(
        contains=contains,
        extension=extension,
        relevance=relevance,
        regex=regex,
    )

    if matches_df.empty:
        return ""

    ranked_df = matches_df.copy()
    ranked_df["csv_row_count_rank"] = pd.to_numeric(ranked_df["csv_row_count"], errors="coerce").fillna(-1)
    ranked_df["size_bytes_rank"] = pd.to_numeric(ranked_df["size_bytes"], errors="coerce").fillna(-1)

    ranked_df = ranked_df.sort_values(
        ["csv_row_count_rank", "size_bytes_rank", "relative_path"],
        ascending=[False, False, True],
        kind="stable",
    ).reset_index(drop=True)

    return str(ranked_df.loc[0, "relative_path"])

EXPECTED_UPSTREAM_INPUTS = [
    {
        "input_id": "sd_restoration_audit",
        "stage": "21_stable_diffusion_restoration",
        "role": "restoration audit with prompts, seeds, runtime, memory, output paths, and failures",
        "contains": ["outputs/21_stable_diffusion_restoration", "stable_diffusion_restoration_audit"],
        "extension": ".csv",
        "required_for_batch": "batch1",
    },
    {
        "input_id": "sd_prompt_policy",
        "stage": "21_stable_diffusion_restoration",
        "role": "prompt policy and generation settings",
        "contains": ["outputs/21_stable_diffusion_restoration", "stable_diffusion_prompt_policy"],
        "extension": ".json",
        "required_for_batch": "batch1",
    },
    {
        "input_id": "sd_prompt_ablation_or_comparison",
        "stage": "21_stable_diffusion_restoration",
        "role": "prompt/candidate sensitivity proxy evidence",
        "contains": ["outputs/21_stable_diffusion_restoration", "prompt"],
        "extension": ".csv",
        "required_for_batch": "batch2",
    },
    {
        "input_id": "sd_classical_metrics",
        "stage": "22_stable_diffusion_classical_metrics",
        "role": "classical full-reference metric rows",
        "contains": ["outputs/22_stable_diffusion_classical_metrics", "stable_diffusion_classical_metrics"],
        "extension": ".csv",
        "required_for_batch": "batch1",
    },
    {
        "input_id": "sd_classical_summary",
        "stage": "22_stable_diffusion_classical_metrics",
        "role": "classical metric summary",
        "contains": ["outputs/22_stable_diffusion_classical_metrics", "summary"],
        "extension": ".csv",
        "required_for_batch": "batch2",
    },
    {
        "input_id": "sd_difference_map_audit",
        "stage": "23_stable_diffusion_difference_maps",
        "role": "difference-map audit and visual evidence paths",
        "contains": ["outputs/23_stable_diffusion_difference_maps", "difference", "audit"],
        "extension": ".csv",
        "required_for_batch": "batch1",
    },
    {
        "input_id": "sd_difference_map_figure_manifest",
        "stage": "23_stable_diffusion_difference_maps",
        "role": "difference-map figure manifest",
        "contains": ["outputs/23_stable_diffusion_difference_maps", "figure_manifest"],
        "extension": ".csv",
        "required_for_batch": "batch4",
    },
    {
        "input_id": "sd_lpips_metrics",
        "stage": "24_stable_diffusion_lpips_metrics",
        "role": "LPIPS metric rows",
        "contains": ["outputs/24_stable_diffusion_lpips_metrics", "lpips", "metrics"],
        "extension": ".csv",
        "required_for_batch": "batch1",
    },
    {
        "input_id": "sd_lpips_selected_cases",
        "stage": "24_stable_diffusion_lpips_metrics",
        "role": "LPIPS selected cases",
        "contains": ["outputs/24_stable_diffusion_lpips_metrics", "selected"],
        "extension": ".csv",
        "required_for_batch": "batch3",
    },
    {
        "input_id": "sd_lpips_figure_manifest",
        "stage": "24_stable_diffusion_lpips_metrics",
        "role": "LPIPS figure manifest",
        "contains": ["outputs/24_stable_diffusion_lpips_metrics", "figure_manifest"],
        "extension": ".csv",
        "required_for_batch": "batch4",
    },
    {
        "input_id": "sd_feature_metrics",
        "stage": "25_stable_diffusion_feature_similarity",
        "role": "CLIP/DINOv2 feature-similarity metric rows",
        "contains": ["outputs/25_stable_diffusion_feature_similarity", "feature_similarity_metrics"],
        "extension": ".csv",
        "required_for_batch": "batch1",
    },
    {
        "input_id": "sd_feature_selected_cases",
        "stage": "25_stable_diffusion_feature_similarity",
        "role": "feature-similarity selected cases",
        "contains": ["outputs/25_stable_diffusion_feature_similarity", "selected_cases"],
        "extension": ".csv",
        "required_for_batch": "batch3",
    },
    {
        "input_id": "sd_feature_figure_manifest",
        "stage": "25_stable_diffusion_feature_similarity",
        "role": "feature-similarity figure manifest",
        "contains": ["outputs/25_stable_diffusion_feature_similarity", "figure_manifest"],
        "extension": ".csv",
        "required_for_batch": "batch4",
    },
    {
        "input_id": "sd_feature_embedding_manifest",
        "stage": "25_stable_diffusion_feature_similarity",
        "role": "feature embedding manifest for audit/handoff only",
        "contains": ["outputs/25_stable_diffusion_feature_similarity", "embedding_manifest"],
        "extension": ".csv",
        "required_for_batch": "batch6",
    },
    {
        "input_id": "sd_feature_embeddings_npz",
        "stage": "25_stable_diffusion_feature_similarity",
        "role": "feature embedding NPZ for audit/handoff only",
        "contains": ["outputs/25_stable_diffusion_feature_similarity", "embeddings"],
        "extension": ".npz",
        "required_for_batch": "batch6",
    },
]

expected_input_rows = []
for spec in EXPECTED_UPSTREAM_INPUTS:
    matches_df = inventory_find(
        contains=spec["contains"],
        extension=spec["extension"],
    )
    best_match = best_inventory_match(
        contains=spec["contains"],
        extension=spec["extension"],
    )

    expected_input_rows.append(
        {
            "input_id": spec["input_id"],
            "stage": spec["stage"],
            "role": spec["role"],
            "required_for_batch": spec["required_for_batch"],
            "extension": spec["extension"],
            "match_count": int(len(matches_df)),
            "best_match": best_match,
            "best_match_exists_on_disk": path_from_project(best_match).exists() if best_match else False,
            "candidate_matches": "; ".join(matches_df["relative_path"].head(8).astype(str).tolist()),
        }
    )

batch0_expected_inputs_df = pd.DataFrame(expected_input_rows)

print("Expected upstream input discovery preview:")
display(batch0_expected_inputs_df)

Expected upstream input discovery preview:


,input_id,stage,role,required_for_batch,extension,match_count,best_match,best_match_exists_on_disk,candidate_matches
0,sd_restoration_audit,21_stable_diffusion_restoration,"restoration audit with prompts, seeds, runtime...",batch1,.csv,1,outputs/21_stable_diffusion_restoration/stable...,True,outputs/21_stable_diffusion_restoration/stable...
1,sd_prompt_policy,21_stable_diffusion_restoration,prompt policy and generation settings,batch1,.json,1,outputs/21_stable_diffusion_restoration/stable...,True,outputs/21_stable_diffusion_restoration/stable...
2,sd_prompt_ablation_or_comparison,21_stable_diffusion_restoration,prompt/candidate sensitivity proxy evidence,batch2,.csv,15,outputs/21_stable_diffusion_restoration/analys...,True,outputs/21_stable_diffusion_restoration/analys...
3,sd_classical_metrics,22_stable_diffusion_classical_metrics,classical full-reference metric rows,batch1,.csv,14,outputs/22_stable_diffusion_classical_metrics/...,True,outputs/22_stable_diffusion_classical_metrics/...
4,sd_classical_summary,22_stable_diffusion_classical_metrics,classical metric summary,batch2,.csv,1,outputs/22_stable_diffusion_classical_metrics/...,True,outputs/22_stable_diffusion_classical_metrics/...
5,sd_difference_map_audit,23_stable_diffusion_difference_maps,difference-map audit and visual evidence paths,batch1,.csv,2,outputs/23_stable_diffusion_difference_maps/me...,True,outputs/23_stable_diffusion_difference_maps/me...
6,sd_difference_map_figure_manifest,23_stable_diffusion_difference_maps,difference-map figure manifest,batch4,.csv,1,outputs/23_stable_diffusion_difference_maps/fi...,True,outputs/23_stable_diffusion_difference_maps/fi...
7,sd_lpips_metrics,24_stable_diffusion_lpips_metrics,LPIPS metric rows,batch1,.csv,17,outputs/24_stable_diffusion_lpips_metrics/inve...,True,outputs/24_stable_diffusion_lpips_metrics/anal...
8,sd_lpips_selected_cases,24_stable_diffusion_lpips_metrics,LPIPS selected cases,batch3,.csv,1,outputs/24_stable_diffusion_lpips_metrics/anal...,True,outputs/24_stable_diffusion_lpips_metrics/anal...
9,sd_lpips_figure_manifest,24_stable_diffusion_lpips_metrics,LPIPS figure manifest,batch4,.csv,1,outputs/24_stable_diffusion_lpips_metrics/figu...,True,outputs/24_stable_diffusion_lpips_metrics/figu...


In [5]:
# Batch 0 / Cell 5 - Batch 0 validation and stage manifest
required_output_dirs = [
    OUTPUT_DIRS["inventory"],
    OUTPUT_DIRS["validation"],
    OUTPUT_DIRS["metrics"],
    OUTPUT_DIRS["analysis"],
    OUTPUT_DIRS["figures"],
    OUTPUT_DIRS["reports"],
    OUTPUT_DIRS["manifests"],
    OUTPUT_DIRS["assets"],
    OUTPUT_DIRS["report_assets"],
    OUTPUT_DIRS["aggregate_figures"],
    OUTPUT_DIRS["candidate_panels"],
    OUTPUT_DIRS["difference_map_panels"],
]

inventory_source_exists = INVENTORY_SOURCE_PATH.is_file()
inventory_snapshot_exists = BATCH0_INVENTORY_SNAPSHOT_PATH.is_file()
inventory_snapshot_rows, inventory_snapshot_columns = csv_shape(BATCH0_INVENTORY_SNAPSHOT_PATH)

expected_input_match_count = int(batch0_expected_inputs_df["match_count"].gt(0).sum())
expected_input_total = int(len(batch0_expected_inputs_df))

batch0_validation_rows = [
    validation_row(
        "project_root_exists",
        rel(PROJECT_ROOT),
        "directory exists",
        PROJECT_ROOT.is_dir(),
        "Project root does not exist.",
    ),
    validation_row(
        "output_root_created",
        rel(OUTPUT_ROOT),
        "directory exists",
        OUTPUT_ROOT.is_dir(),
        "Notebook output root was not created.",
    ),
    validation_row(
        "required_output_directories_created",
        [rel(path) for path in required_output_dirs],
        "all directories exist",
        all(path.is_dir() for path in required_output_dirs),
        "One or more required output directories were not created.",
    ),
    validation_row(
        "inventory_source_exists",
        rel(INVENTORY_SOURCE_PATH),
        "file exists",
        inventory_source_exists,
        "Selected project inventory source does not exist.",
    ),
    validation_row(
        "inventory_source_has_relative_path",
        "relative_path" in inventory_raw_df.columns,
        True,
        "relative_path" in inventory_raw_df.columns,
        "Selected inventory source does not have a relative_path column.",
    ),
    validation_row(
        "inventory_source_not_empty",
        len(inventory_raw_df),
        "> 0",
        len(inventory_raw_df) > 0,
        "Selected inventory source is empty.",
    ),
    validation_row(
        "inventory_snapshot_written",
        rel(BATCH0_INVENTORY_SNAPSHOT_PATH),
        "file exists",
        inventory_snapshot_exists,
        "Batch 0 inventory snapshot was not written.",
    ),
    validation_row(
        "inventory_snapshot_not_empty",
        inventory_snapshot_rows,
        "> 0",
        inventory_snapshot_rows is not None and inventory_snapshot_rows > 0,
        "Batch 0 inventory snapshot is empty or unreadable.",
    ),
    validation_row(
        "inventory_snapshot_has_core_columns",
        list(batch0_inventory_snapshot_df.columns),
        ["relative_path", "file_name", "extension", "report_relevance"],
        all(column in batch0_inventory_snapshot_df.columns for column in ["relative_path", "file_name", "extension", "report_relevance"]),
        "Batch 0 inventory snapshot is missing one or more core columns.",
    ),
    validation_row(
        "stable_diffusion_related_inventory_rows_present",
        int(batch0_inventory_snapshot_df["is_report_relevant"].sum()),
        "> 0",
        int(batch0_inventory_snapshot_df["is_report_relevant"].sum()) > 0,
        "Inventory snapshot did not identify any Stable Diffusion/report-relevant rows.",
    ),
    validation_row(
        "expected_upstream_inputs_discoverable_preview",
        f"{expected_input_match_count}/{expected_input_total}",
        "informational discovery preview only; Batch 1 performs hard validation",
        True,
        "",
        severity="info",
    ),
]

batch0_validation_df = pd.DataFrame(batch0_validation_rows)
batch0_validation_df.to_csv(BATCH0_VALIDATION_PATH, index=False)

batch0_error_checks_df = batch0_validation_df.loc[batch0_validation_df["severity"].eq("error")].copy()
batch0_passed = bool(bool_series(batch0_error_checks_df["passed"]).all())

stage_manifest = read_json_if_exists(STAGE_MANIFEST_PATH)
stage_manifest.update(
    {
        "notebook_id": NOTEBOOK_ID,
        "notebook_title": NOTEBOOK_TITLE,
        "report_subject": REPORT_SUBJECT,
        "report_version": REPORT_VERSION,
        "stage": "batch0_setup_paths_inventory_snapshot",
        "stage_status": "passed" if batch0_passed else "failed",
        "updated_at_utc": utc_now_iso(),
        "output_root": rel(OUTPUT_ROOT),
        "batch0": {
            "status": "passed" if batch0_passed else "failed",
            "inventory_source": rel(INVENTORY_SOURCE_PATH),
            "inventory_snapshot": rel(BATCH0_INVENTORY_SNAPSHOT_PATH),
            "validation": rel(BATCH0_VALIDATION_PATH),
            "inventory_source_rows": int(len(inventory_raw_df)),
            "inventory_snapshot_rows": int(inventory_snapshot_rows or 0),
            "inventory_snapshot_columns": int(inventory_snapshot_columns or 0),
            "stable_diffusion_related_rows": int(batch0_inventory_snapshot_df["is_report_relevant"].sum()),
            "expected_upstream_inputs_detected_preview": expected_input_match_count,
            "expected_upstream_inputs_total_preview": expected_input_total,
            "checks_passed": int(bool_series(batch0_error_checks_df["passed"]).sum()),
            "checks_total": int(len(batch0_error_checks_df)),
        },
        "planned_outputs": {
            "batch0_inventory_snapshot": rel(BATCH0_INVENTORY_SNAPSHOT_PATH),
            "batch0_validation": rel(BATCH0_VALIDATION_PATH),
            "final_artifact_index": rel(FINAL_ARTIFACT_INDEX_PATH),
            "final_handoff_manifest": rel(FINAL_HANDOFF_MANIFEST_PATH),
            "stage_manifest": rel(STAGE_MANIFEST_PATH),
        },
    }
)

write_json(STAGE_MANIFEST_PATH, stage_manifest)

print(f"Saved Batch 0 validation: {rel(BATCH0_VALIDATION_PATH)}")
print(f"Updated stage manifest: {rel(STAGE_MANIFEST_PATH)}")
print(f"Batch 0 error checks passed: {int(bool_series(batch0_error_checks_df['passed']).sum())} / {len(batch0_error_checks_df)}")

display(batch0_validation_df)

missing_preview_df = batch0_expected_inputs_df.loc[batch0_expected_inputs_df["match_count"].eq(0)].copy()
if not missing_preview_df.empty:
    print("Preview only: some expected upstream inputs were not discovered by the inventory. Batch 1 will validate these strictly.")
    display(missing_preview_df[["input_id", "stage", "role", "required_for_batch", "extension"]])

if not batch0_passed:
    display(batch0_validation_df.loc[~bool_series(batch0_validation_df["passed"]), ["check_name", "failure_message"]])
    raise RuntimeError("Batch 0 validation failed. Fix setup/inventory issues before running Batch 1.")

print("Batch 0 passed. Setup, paths, inventory snapshot, and shared utilities are ready.")

Saved Batch 0 validation: outputs/26_stable_diffusion_report_generation/validation/batch0_validation.csv
Updated stage manifest: outputs/26_stable_diffusion_report_generation/manifests/stable_diffusion_report_stage_manifest.json
Batch 0 error checks passed: 10 / 10


,check_name,severity,actual,expected,passed,failure_message,checked_at_utc
0,project_root_exists,error,""".""","""directory exists""",True,,2026-08-10T11:28:47.456929+00:00
1,output_root_created,error,"""outputs/26_stable_diffusion_report_generation""","""directory exists""",True,,2026-08-10T11:28:47.456929+00:00
2,required_output_directories_created,error,"[""outputs/26_stable_diffusion_report_generatio...","""all directories exist""",True,,2026-08-10T11:28:47.469279+00:00
3,inventory_source_exists,error,"""outputs/inventory/project_file_inventory.csv""","""file exists""",True,,2026-08-10T11:28:47.469279+00:00
4,inventory_source_has_relative_path,error,true,true,True,,2026-08-10T11:28:47.469279+00:00
5,inventory_source_not_empty,error,10914,"""> 0""",True,,2026-08-10T11:28:47.469279+00:00
6,inventory_snapshot_written,error,"""outputs/26_stable_diffusion_report_generation...","""file exists""",True,,2026-08-10T11:28:47.471825+00:00
7,inventory_snapshot_not_empty,error,10914,"""> 0""",True,,2026-08-10T11:28:47.471825+00:00
8,inventory_snapshot_has_core_columns,error,"[""inventory_snapshot_created_at_utc"", ""invento...","[""relative_path"", ""file_name"", ""extension"", ""r...",True,,2026-08-10T11:28:47.471825+00:00
9,stable_diffusion_related_inventory_rows_present,error,7826,"""> 0""",True,,2026-08-10T11:28:47.471825+00:00


Batch 0 passed. Setup, paths, inventory snapshot, and shared utilities are ready.


In [6]:
# Batch 1 / Cell 1 - Define upstream artifact contract
BATCH1_VALIDATION_PATH = OUTPUT_DIRS["validation"] / "batch1_input_validation.csv"

UPSTREAM_ARTIFACT_SPECS = [
    {
        "artifact_id": "sd_restoration_audit",
        "stage": "21_stable_diffusion_restoration",
        "kind": "csv",
        "required": True,
        "expected_rows": 945,
        "contains": [
            "outputs/21_stable_diffusion_restoration",
            "stable_diffusion_restoration_audit.csv",
        ],
        "role": "Primary restoration audit with prompts, seeds, runtime, memory, outputs, and failures.",
    },
    {
        "artifact_id": "sd_candidate_manifest",
        "stage": "21_stable_diffusion_restoration",
        "kind": "csv",
        "required": False,
        "expected_rows": 945,
        "contains": [
            "outputs/21_stable_diffusion_restoration",
            "stable_diffusion_candidate_manifest.csv",
        ],
        "role": "Candidate manifest retained for traceability if needed.",
    },
    {
        "artifact_id": "sd_prompt_policy",
        "stage": "21_stable_diffusion_restoration",
        "kind": "json",
        "required": True,
        "expected_rows": None,
        "contains": [
            "outputs/21_stable_diffusion_restoration",
            "stable_diffusion_prompt_policy.json",
        ],
        "role": "Prompt policy and generation settings.",
    },
    {
        "artifact_id": "sd_prompt_ablation_rows",
        "stage": "21_stable_diffusion_restoration",
        "kind": "csv",
        "required": False,
        "expected_rows": 600,
        "contains": [
            "outputs/21_stable_diffusion_restoration/analysis",
            "stable_diffusion_prompt_ablation_analysis_rows.csv",
        ],
        "role": "Prompt/candidate sensitivity proxy evidence.",
    },
    {
        "artifact_id": "sd_prompt_comparison_pairs",
        "stage": "21_stable_diffusion_restoration",
        "kind": "csv",
        "required": False,
        "expected_rows": 480,
        "contains": [
            "outputs/21_stable_diffusion_restoration/analysis",
            "stable_diffusion_prompt_comparison_pairs.csv",
        ],
        "role": "Prompt comparison pair evidence for preliminary sensitivity reporting.",
    },
    {
        "artifact_id": "sd_classical_metrics",
        "stage": "22_stable_diffusion_classical_metrics",
        "kind": "csv",
        "required": True,
        "expected_rows": 5294,
        "contains": [
            "outputs/22_stable_diffusion_classical_metrics/metrics",
            "stable_diffusion_classical_metrics.csv",
        ],
        "role": "Classical full-reference metric rows.",
    },
    {
        "artifact_id": "sd_classical_summary",
        "stage": "22_stable_diffusion_classical_metrics",
        "kind": "csv",
        "required": True,
        "expected_rows": 135,
        "contains": [
            "outputs/22_stable_diffusion_classical_metrics/analysis",
            "stable_diffusion_classical_metrics_summary.csv",
        ],
        "role": "Classical metric summary table.",
    },
    {
        "artifact_id": "sd_classical_selected_cases",
        "stage": "22_stable_diffusion_classical_metrics",
        "kind": "csv",
        "required": False,
        "expected_rows": 11,
        "contains": [
            "outputs/22_stable_diffusion_classical_metrics/analysis",
            "stable_diffusion_classical_metric_selected_cases.csv",
        ],
        "role": "Representative classical metric cases.",
    },
    {
        "artifact_id": "sd_classical_figure_manifest",
        "stage": "22_stable_diffusion_classical_metrics",
        "kind": "csv",
        "required": False,
        "expected_rows": 11,
        "contains": [
            "outputs/22_stable_diffusion_classical_metrics/figures",
            "stable_diffusion_classical_metric_figure_manifest.csv",
        ],
        "role": "Classical metric figure manifest.",
    },
    {
        "artifact_id": "sd_difference_map_audit",
        "stage": "23_stable_diffusion_difference_maps",
        "kind": "csv",
        "required": True,
        "expected_rows": 945,
        "contains": [
            "outputs/23_stable_diffusion_difference_maps/metrics",
            "stable_diffusion_difference_map_audit.csv",
        ],
        "role": "Difference-map audit and image evidence paths.",
    },
    {
        "artifact_id": "sd_difference_map_summary",
        "stage": "23_stable_diffusion_difference_maps",
        "kind": "csv",
        "required": True,
        "expected_rows": 945,
        "contains": [
            "outputs/23_stable_diffusion_difference_maps/analysis",
            "stable_diffusion_difference_map_summary.csv",
        ],
        "role": "Difference-map summary rows.",
    },
    {
        "artifact_id": "sd_difference_map_selected_cases",
        "stage": "23_stable_diffusion_difference_maps",
        "kind": "csv",
        "required": False,
        "expected_rows": 11,
        "contains": [
            "outputs/23_stable_diffusion_difference_maps/analysis",
            "stable_diffusion_difference_map_selected_cases.csv",
        ],
        "role": "Representative difference-map cases.",
    },
    {
        "artifact_id": "sd_difference_map_figure_manifest",
        "stage": "23_stable_diffusion_difference_maps",
        "kind": "csv",
        "required": False,
        "expected_rows": 11,
        "contains": [
            "outputs/23_stable_diffusion_difference_maps/figures",
            "stable_diffusion_difference_map_figure_manifest.csv",
        ],
        "role": "Difference-map figure manifest.",
    },
    {
        "artifact_id": "sd_difference_map_image_manifest",
        "stage": "23_stable_diffusion_difference_maps",
        "kind": "csv",
        "required": False,
        "expected_rows": 945,
        "contains": [
            "outputs/23_stable_diffusion_difference_maps/manifests",
            "stable_diffusion_difference_map_image_manifest.csv",
        ],
        "role": "Difference-map image manifest.",
    },
    {
        "artifact_id": "sd_lpips_metrics",
        "stage": "24_stable_diffusion_lpips_metrics",
        "kind": "csv",
        "required": True,
        "expected_rows": 2724,
        "contains": [
            "outputs/24_stable_diffusion_lpips_metrics/metrics",
            "stable_diffusion_lpips_metrics.csv",
        ],
        "role": "LPIPS metric rows.",
    },
    {
        "artifact_id": "sd_lpips_summary",
        "stage": "24_stable_diffusion_lpips_metrics",
        "kind": "csv",
        "required": True,
        "expected_rows": 83,
        "contains": [
            "outputs/24_stable_diffusion_lpips_metrics/analysis",
            "stable_diffusion_lpips_summary.csv",
        ],
        "role": "LPIPS summary table.",
    },
    {
        "artifact_id": "sd_lpips_selected_cases",
        "stage": "24_stable_diffusion_lpips_metrics",
        "kind": "csv",
        "required": False,
        "expected_rows": 56,
        "contains": [
            "outputs/24_stable_diffusion_lpips_metrics/analysis",
            "stable_diffusion_lpips_selected_cases.csv",
        ],
        "role": "LPIPS representative cases.",
    },
    {
        "artifact_id": "sd_lpips_figure_manifest",
        "stage": "24_stable_diffusion_lpips_metrics",
        "kind": "csv",
        "required": False,
        "expected_rows": 19,
        "contains": [
            "outputs/24_stable_diffusion_lpips_metrics/figures",
            "stable_diffusion_lpips_figure_manifest.csv",
        ],
        "role": "LPIPS figure manifest.",
    },
    {
        "artifact_id": "sd_feature_metrics",
        "stage": "25_stable_diffusion_feature_similarity",
        "kind": "csv",
        "required": True,
        "expected_rows": 2724,
        "contains": [
            "outputs/25_stable_diffusion_feature_similarity/metrics",
            "stable_diffusion_feature_similarity_metrics.csv",
        ],
        "role": "CLIP/DINOv2 feature-similarity metric rows.",
    },
    {
        "artifact_id": "sd_feature_summary",
        "stage": "25_stable_diffusion_feature_similarity",
        "kind": "csv",
        "required": True,
        "expected_rows": 90,
        "contains": [
            "outputs/25_stable_diffusion_feature_similarity/analysis",
            "stable_diffusion_feature_similarity_summary.csv",
        ],
        "role": "Feature-similarity summary table.",
    },
    {
        "artifact_id": "sd_feature_selected_cases",
        "stage": "25_stable_diffusion_feature_similarity",
        "kind": "csv",
        "required": False,
        "expected_rows": 60,
        "contains": [
            "outputs/25_stable_diffusion_feature_similarity/analysis",
            "stable_diffusion_feature_similarity_selected_cases.csv",
        ],
        "role": "Feature-similarity representative cases.",
    },
    {
        "artifact_id": "sd_feature_figure_manifest",
        "stage": "25_stable_diffusion_feature_similarity",
        "kind": "csv",
        "required": False,
        "expected_rows": 65,
        "contains": [
            "outputs/25_stable_diffusion_feature_similarity/figures",
            "stable_diffusion_feature_similarity_figure_manifest.csv",
        ],
        "role": "Feature-similarity figure manifest.",
    },
    {
        "artifact_id": "sd_feature_embedding_manifest",
        "stage": "25_stable_diffusion_feature_similarity",
        "kind": "csv",
        "required": False,
        "expected_rows": 16344,
        "contains": [
            "outputs/25_stable_diffusion_feature_similarity/manifests",
            "stable_diffusion_feature_similarity_embedding_manifest.csv",
        ],
        "role": "Feature embedding manifest for handoff traceability.",
    },
    {
        "artifact_id": "sd_feature_embeddings_npz",
        "stage": "25_stable_diffusion_feature_similarity",
        "kind": "npz",
        "required": False,
        "expected_rows": None,
        "contains": [
            "outputs/25_stable_diffusion_feature_similarity/metrics",
            "stable_diffusion_feature_similarity_embeddings.npz",
        ],
        "role": "Feature embedding NPZ for handoff traceability.",
    },
]

UPSTREAM_FINAL_VALIDATION_SPECS = [
    {
        "validation_id": "sd_restoration_final_validation",
        "stage": "21_stable_diffusion_restoration",
        "expected_rows": 14,
        "contains": [
            "outputs/21_stable_diffusion_restoration/validation",
            "stable_diffusion_batch8_final_validation.csv",
        ],
    },
    {
        "validation_id": "sd_classical_metrics_final_validation",
        "stage": "22_stable_diffusion_classical_metrics",
        "expected_rows": 22,
        "contains": [
            "outputs/22_stable_diffusion_classical_metrics/validation",
            "stable_diffusion_classical_metrics_final_validation.csv",
        ],
    },
    {
        "validation_id": "sd_difference_maps_final_validation",
        "stage": "23_stable_diffusion_difference_maps",
        "expected_rows": 19,
        "contains": [
            "outputs/23_stable_diffusion_difference_maps/validation",
            "stable_diffusion_difference_maps_final_validation.csv",
        ],
    },
    {
        "validation_id": "sd_lpips_metrics_final_validation",
        "stage": "24_stable_diffusion_lpips_metrics",
        "expected_rows": 21,
        "contains": [
            "outputs/24_stable_diffusion_lpips_metrics/validation",
            "stable_diffusion_lpips_metrics_final_validation.csv",
        ],
    },
    {
        "validation_id": "sd_feature_similarity_final_validation",
        "stage": "25_stable_diffusion_feature_similarity",
        "expected_rows": 15,
        "contains": [
            "outputs/25_stable_diffusion_feature_similarity/validation",
            "stable_diffusion_feature_similarity_final_validation.csv",
        ],
    },
]

print(f"Configured upstream artifact specs: {len(UPSTREAM_ARTIFACT_SPECS)}")
print(f"Configured upstream final validation specs: {len(UPSTREAM_FINAL_VALIDATION_SPECS)}")

Configured upstream artifact specs: 24
Configured upstream final validation specs: 5


In [7]:
# Batch 1 / Cell 2 - Resolve and load upstream artifacts
def batch1_resolve_artifact(spec: dict) -> dict:
    matches_df = inventory_find(
        contains=spec["contains"],
        extension=f".{spec['kind']}" if spec["kind"] not in ["csv", "json", "npz"] else f".{spec['kind']}",
    )

    if matches_df.empty:
        return {
            **spec,
            "resolved_relative_path": "",
            "resolved_path": None,
            "match_count": 0,
            "inventory_csv_row_count": np.nan,
            "inventory_csv_column_count": np.nan,
            "inventory_size_bytes": np.nan,
            "exists_on_disk": False,
            "loaded": False,
            "load_error": "No matching inventory row found.",
        }

    ranked_df = matches_df.copy()
    ranked_df["csv_row_count_rank"] = pd.to_numeric(ranked_df["csv_row_count"], errors="coerce").fillna(-1)
    ranked_df["size_bytes_rank"] = pd.to_numeric(ranked_df["size_bytes"], errors="coerce").fillna(-1)

    ranked_df = ranked_df.sort_values(
        ["csv_row_count_rank", "size_bytes_rank", "relative_path"],
        ascending=[False, False, True],
        kind="stable",
    ).reset_index(drop=True)

    row = ranked_df.iloc[0]
    resolved_relative_path = str(row["relative_path"])
    resolved_path = path_from_project(resolved_relative_path)

    return {
        **spec,
        "resolved_relative_path": resolved_relative_path,
        "resolved_path": resolved_path,
        "match_count": int(len(matches_df)),
        "inventory_csv_row_count": row.get("csv_row_count", np.nan),
        "inventory_csv_column_count": row.get("csv_column_count", np.nan),
        "inventory_size_bytes": row.get("size_bytes", np.nan),
        "exists_on_disk": resolved_path.exists(),
        "loaded": False,
        "load_error": "",
    }

def batch1_load_artifact(resolved_spec: dict) -> tuple[Any, dict]:
    path = resolved_spec["resolved_path"]
    kind = resolved_spec["kind"]

    if path is None:
        resolved_spec["load_error"] = "Cannot load because no path was resolved."
        return None, resolved_spec

    if not path.exists():
        resolved_spec["load_error"] = f"Resolved path is not present on disk: {rel(path)}"
        return None, resolved_spec

    try:
        if kind == "csv":
            loaded_object = pd.read_csv(path)
            resolved_spec["actual_rows"] = int(len(loaded_object))
            resolved_spec["actual_columns"] = int(len(loaded_object.columns))
            resolved_spec["columns"] = list(loaded_object.columns)
        elif kind == "json":
            loaded_object = json.loads(path.read_text(encoding="utf-8"))
            resolved_spec["actual_rows"] = None
            resolved_spec["actual_columns"] = None
            resolved_spec["columns"] = list(loaded_object.keys()) if isinstance(loaded_object, dict) else []
        elif kind == "npz":
            loaded_object = np.load(path, allow_pickle=False)
            resolved_spec["actual_rows"] = None
            resolved_spec["actual_columns"] = None
            resolved_spec["columns"] = list(loaded_object.files)
        else:
            loaded_object = path
            resolved_spec["actual_rows"] = None
            resolved_spec["actual_columns"] = None
            resolved_spec["columns"] = []
        resolved_spec["loaded"] = True
        resolved_spec["load_error"] = ""
        return loaded_object, resolved_spec
    except Exception as exc:
        resolved_spec["loaded"] = False
        resolved_spec["load_error"] = str(exc)
        return None, resolved_spec

UPSTREAM_PATHS = {}
UPSTREAM_DATA = {}
UPSTREAM_LOAD_RECORDS = []

for spec in UPSTREAM_ARTIFACT_SPECS:
    resolved_spec = batch1_resolve_artifact(spec)
    loaded_object, resolved_spec = batch1_load_artifact(resolved_spec)

    artifact_id = resolved_spec["artifact_id"]
    UPSTREAM_PATHS[artifact_id] = resolved_spec["resolved_path"]
    UPSTREAM_DATA[artifact_id] = loaded_object
    UPSTREAM_LOAD_RECORDS.append(resolved_spec)

batch1_upstream_loads_df = pd.DataFrame(
    [
        {
            "artifact_id": record["artifact_id"],
            "stage": record["stage"],
            "kind": record["kind"],
            "required": bool(record["required"]),
            "role": record["role"],
            "resolved_relative_path": record["resolved_relative_path"],
            "match_count": int(record["match_count"]),
            "exists_on_disk": bool(record["exists_on_disk"]),
            "loaded": bool(record["loaded"]),
            "expected_rows": record["expected_rows"],
            "actual_rows": record.get("actual_rows"),
            "actual_columns": record.get("actual_columns"),
            "inventory_csv_row_count": record.get("inventory_csv_row_count"),
            "inventory_csv_column_count": record.get("inventory_csv_column_count"),
            "inventory_size_bytes": record.get("inventory_size_bytes"),
            "load_error": record.get("load_error", ""),
        }
        for record in UPSTREAM_LOAD_RECORDS
    ]
)

print("Upstream artifact load summary:")
display(batch1_upstream_loads_df)

Upstream artifact load summary:


,artifact_id,stage,kind,required,role,resolved_relative_path,match_count,exists_on_disk,loaded,expected_rows,actual_rows,actual_columns,inventory_csv_row_count,inventory_csv_column_count,inventory_size_bytes,load_error
0,sd_restoration_audit,21_stable_diffusion_restoration,csv,True,"Primary restoration audit with prompts, seeds,...",outputs/21_stable_diffusion_restoration/stable...,1,True,True,945.0,945.0,282.0,945.0,282.0,3512362,
1,sd_candidate_manifest,21_stable_diffusion_restoration,csv,False,Candidate manifest retained for traceability i...,outputs/21_stable_diffusion_restoration/stable...,1,True,True,945.0,945.0,224.0,945.0,224.0,2904218,
2,sd_prompt_policy,21_stable_diffusion_restoration,json,True,Prompt policy and generation settings.,outputs/21_stable_diffusion_restoration/stable...,1,True,True,NaN,NaN,NaN,NaN,NaN,6376,
3,sd_prompt_ablation_rows,21_stable_diffusion_restoration,csv,False,Prompt/candidate sensitivity proxy evidence.,outputs/21_stable_diffusion_restoration/analys...,1,True,True,600.0,600.0,297.0,600.0,297.0,2334656,
4,sd_prompt_comparison_pairs,21_stable_diffusion_restoration,csv,False,Prompt comparison pair evidence for preliminar...,outputs/21_stable_diffusion_restoration/analys...,1,True,True,480.0,480.0,29.0,480.0,29.0,492777,
5,sd_classical_metrics,22_stable_diffusion_classical_metrics,csv,True,Classical full-reference metric rows.,outputs/22_stable_diffusion_classical_metrics/...,1,True,True,5294.0,5294.0,58.0,5294.0,58.0,6288950,
6,sd_classical_summary,22_stable_diffusion_classical_metrics,csv,True,Classical metric summary table.,outputs/22_stable_diffusion_classical_metrics/...,1,True,True,135.0,135.0,24.0,135.0,24.0,25706,
7,sd_classical_selected_cases,22_stable_diffusion_classical_metrics,csv,False,Representative classical metric cases.,outputs/22_stable_diffusion_classical_metrics/...,1,True,True,11.0,11.0,65.0,11.0,65.0,15579,
8,sd_classical_figure_manifest,22_stable_diffusion_classical_metrics,csv,False,Classical metric figure manifest.,outputs/22_stable_diffusion_classical_metrics/...,1,True,True,11.0,11.0,19.0,11.0,19.0,5764,
9,sd_difference_map_audit,23_stable_diffusion_difference_maps,csv,True,Difference-map audit and image evidence paths.,outputs/23_stable_diffusion_difference_maps/me...,1,True,True,945.0,945.0,147.0,945.0,147.0,2476887,


In [8]:
# Batch 1 / Cell 3 - Resolve and validate prior final validation files
def batch1_resolve_validation(spec: dict) -> dict:
    matches_df = inventory_find(
        contains=spec["contains"],
        extension=".csv",
    )

    if matches_df.empty:
        return {
            **spec,
            "resolved_relative_path": "",
            "resolved_path": None,
            "match_count": 0,
            "exists_on_disk": False,
            "loaded": False,
            "actual_rows": None,
            "actual_columns": None,
            "passed_column": "",
            "checks_passed": 0,
            "checks_total": 0,
            "validation_passed": False,
            "load_error": "No matching final validation file found.",
        }

    ranked_df = matches_df.copy()
    ranked_df["csv_row_count_rank"] = pd.to_numeric(ranked_df["csv_row_count"], errors="coerce").fillna(-1)
    ranked_df["size_bytes_rank"] = pd.to_numeric(ranked_df["size_bytes"], errors="coerce").fillna(-1)

    ranked_df = ranked_df.sort_values(
        ["csv_row_count_rank", "size_bytes_rank", "relative_path"],
        ascending=[False, False, True],
        kind="stable",
    ).reset_index(drop=True)

    row = ranked_df.iloc[0]
    resolved_relative_path = str(row["relative_path"])
    resolved_path = path_from_project(resolved_relative_path)

    record = {
        **spec,
        "resolved_relative_path": resolved_relative_path,
        "resolved_path": resolved_path,
        "match_count": int(len(matches_df)),
        "exists_on_disk": resolved_path.exists(),
        "loaded": False,
        "actual_rows": None,
        "actual_columns": None,
        "passed_column": "",
        "checks_passed": 0,
        "checks_total": 0,
        "validation_passed": False,
        "load_error": "",
    }

    if not resolved_path.exists():
        record["load_error"] = f"Resolved final validation path is not present on disk: {rel(resolved_path)}"
        return record

    try:
        validation_df = pd.read_csv(resolved_path)
        record["loaded"] = True
        record["actual_rows"] = int(len(validation_df))
        record["actual_columns"] = int(len(validation_df.columns))

        if "passed" in validation_df.columns:
            passed_values = bool_series(validation_df["passed"])
            record["passed_column"] = "passed"
        elif "status" in validation_df.columns:
            passed_values = validation_df["status"].astype(str).str.strip().str.lower().isin(["passed", "pass", "ok", "true"])
            record["passed_column"] = "status"
        else:
            passed_values = pd.Series([False] * len(validation_df))
            record["passed_column"] = ""

        if "severity" in validation_df.columns:
            blocking_mask = ~validation_df["severity"].astype(str).str.lower().isin(["info", "warning"])
            passed_values_for_blocking = passed_values.loc[blocking_mask]
        else:
            passed_values_for_blocking = passed_values

        record["checks_passed"] = int(passed_values_for_blocking.sum())
        record["checks_total"] = int(len(passed_values_for_blocking))
        record["validation_passed"] = bool(len(passed_values_for_blocking) > 0 and passed_values_for_blocking.all())
        record["load_error"] = ""

        return record

    except Exception as exc:
        record["load_error"] = str(exc)
        return record

UPSTREAM_VALIDATION_PATHS = {}
UPSTREAM_FINAL_VALIDATION_RECORDS = []

for spec in UPSTREAM_FINAL_VALIDATION_SPECS:
    record = batch1_resolve_validation(spec)
    UPSTREAM_VALIDATION_PATHS[record["validation_id"]] = record["resolved_path"]
    UPSTREAM_FINAL_VALIDATION_RECORDS.append(record)

batch1_prior_validation_df = pd.DataFrame(
    [
        {
            "validation_id": record["validation_id"],
            "stage": record["stage"],
            "resolved_relative_path": record["resolved_relative_path"],
            "match_count": int(record["match_count"]),
            "exists_on_disk": bool(record["exists_on_disk"]),
            "loaded": bool(record["loaded"]),
            "expected_rows": record["expected_rows"],
            "actual_rows": record["actual_rows"],
            "actual_columns": record["actual_columns"],
            "passed_column": record["passed_column"],
            "checks_passed": record["checks_passed"],
            "checks_total": record["checks_total"],
            "validation_passed": bool(record["validation_passed"]),
            "load_error": record["load_error"],
        }
        for record in UPSTREAM_FINAL_VALIDATION_RECORDS
    ]
)

print("Prior final validation summary:")
display(batch1_prior_validation_df)

Prior final validation summary:


,validation_id,stage,resolved_relative_path,match_count,exists_on_disk,loaded,expected_rows,actual_rows,actual_columns,passed_column,checks_passed,checks_total,validation_passed,load_error
0,sd_restoration_final_validation,21_stable_diffusion_restoration,outputs/21_stable_diffusion_restoration/valida...,1,True,True,14,14,5,passed,14,14,True,
1,sd_classical_metrics_final_validation,22_stable_diffusion_classical_metrics,outputs/22_stable_diffusion_classical_metrics/...,1,True,True,22,22,5,passed,22,22,True,
2,sd_difference_maps_final_validation,23_stable_diffusion_difference_maps,outputs/23_stable_diffusion_difference_maps/va...,1,True,True,19,19,6,passed,19,19,True,
3,sd_lpips_metrics_final_validation,24_stable_diffusion_lpips_metrics,outputs/24_stable_diffusion_lpips_metrics/vali...,1,True,True,21,21,5,passed,21,21,True,
4,sd_feature_similarity_final_validation,25_stable_diffusion_feature_similarity,outputs/25_stable_diffusion_feature_similarity...,1,True,True,15,15,6,passed,15,15,True,


In [9]:
# Batch 1 / Cell 4 - Join-key and evidence-column checks
def existing_columns(df: pd.DataFrame | None, candidates: list[str]) -> list[str]:
    if df is None:
        return []
    return [column for column in candidates if column in df.columns]

def any_existing_column(df: pd.DataFrame | None, candidates: list[str]) -> bool:
    return bool(existing_columns(df, candidates))

def dataframe_columns(artifact_id: str) -> list[str]:
    obj = UPSTREAM_DATA.get(artifact_id)
    if isinstance(obj, pd.DataFrame):
        return list(obj.columns)
    return []

JOIN_KEY_CANDIDATES = [
    "case_id",
    "canonical_case_id",
    "source_case_id",
    "dataset",
    "mask_id",
    "mask_type",
    "damage_profile",
    "prompt_variant",
    "candidate_id",
    "restoration_id",
    "restored_path",
    "restored_image_path",
]

RESTORATION_EVIDENCE_COLUMNS = [
    "case_id",
    "canonical_case_id",
    "dataset",
    "mask_id",
    "mask_type",
    "prompt",
    "negative_prompt",
    "prompt_variant",
    "seed",
    "guidance_scale",
    "num_inference_steps",
    "runtime_seconds",
    "gpu_memory_allocated_mb",
    "gpu_memory_reserved_mb",
    "status",
    "failure_reason",
    "restored_path",
    "restored_image_path",
]

METRIC_VALUE_HINT_COLUMNS = [
    "metric_name",
    "metric_value",
    "region",
    "region_name",
    "comparison_name",
    "improvement",
    "improvement_abs",
    "improvement_pct",
]

FIGURE_PATH_HINT_COLUMNS = [
    "figure_path",
    "path",
    "relative_path",
    "panel_path",
    "image_path",
    "difference_map_path",
    "heatmap_path",
]

batch1_evidence_checks = []

for artifact_id in [
    "sd_restoration_audit",
    "sd_classical_metrics",
    "sd_difference_map_audit",
    "sd_lpips_metrics",
    "sd_feature_metrics",
]:
    df = UPSTREAM_DATA.get(artifact_id)
    columns = dataframe_columns(artifact_id)

    batch1_evidence_checks.append(
        {
            "artifact_id": artifact_id,
            "loaded_dataframe": isinstance(df, pd.DataFrame),
            "row_count": int(len(df)) if isinstance(df, pd.DataFrame) else 0,
            "column_count": int(len(columns)),
            "join_key_columns": existing_columns(df, JOIN_KEY_CANDIDATES),
            "join_key_column_count": len(existing_columns(df, JOIN_KEY_CANDIDATES)),
            "has_any_join_key": any_existing_column(df, JOIN_KEY_CANDIDATES),
            "metric_hint_columns": existing_columns(df, METRIC_VALUE_HINT_COLUMNS),
            "figure_hint_columns": existing_columns(df, FIGURE_PATH_HINT_COLUMNS),
        }
    )

batch1_evidence_checks_df = pd.DataFrame(batch1_evidence_checks)

restoration_audit_df = UPSTREAM_DATA.get("sd_restoration_audit")
restoration_evidence_columns = existing_columns(restoration_audit_df, RESTORATION_EVIDENCE_COLUMNS)

prompt_policy = UPSTREAM_DATA.get("sd_prompt_policy")
prompt_policy_loaded = isinstance(prompt_policy, dict) and len(prompt_policy) > 0

optional_loaded_count = int(
    batch1_upstream_loads_df.loc[
        ~batch1_upstream_loads_df["required"].astype(bool),
        "loaded",
    ].astype(bool).sum()
)

optional_total_count = int((~batch1_upstream_loads_df["required"].astype(bool)).sum())

print("Evidence-column checks:")
display(batch1_evidence_checks_df)

print("Restoration audit evidence columns detected:")
display(pd.DataFrame({"column": restoration_evidence_columns}))

print(f"Optional evidence loaded: {optional_loaded_count} / {optional_total_count}")

Evidence-column checks:


,artifact_id,loaded_dataframe,row_count,column_count,join_key_columns,join_key_column_count,has_any_join_key,metric_hint_columns,figure_hint_columns
0,sd_restoration_audit,True,945,282,"[case_id, source_case_id, mask_id, mask_type, ...",6,True,[],[]
1,sd_classical_metrics,True,5294,58,"[case_id, source_case_id, mask_id, mask_type, ...",6,True,[],[]
2,sd_difference_map_audit,True,945,147,"[case_id, source_case_id, mask_id, mask_type, ...",6,True,[],[]
3,sd_lpips_metrics,True,2724,66,"[case_id, source_case_id, mask_id, mask_type, ...",5,True,[],[]
4,sd_feature_metrics,True,2724,59,"[case_id, source_case_id, mask_id, mask_type]",4,True,[],[]


Restoration audit evidence columns detected:


,column
0,case_id
1,mask_id
2,mask_type
3,prompt
4,negative_prompt
5,guidance_scale
6,num_inference_steps
7,runtime_seconds
8,status
9,restored_path


Optional evidence loaded: 14 / 14


In [10]:
# Batch 1 / Cell 5 - Write Batch 1 validation
batch1_validation_rows = []

required_loads_df = batch1_upstream_loads_df.loc[batch1_upstream_loads_df["required"].astype(bool)].copy()
optional_loads_df = batch1_upstream_loads_df.loc[~batch1_upstream_loads_df["required"].astype(bool)].copy()

batch1_validation_rows.append(
    validation_row(
        "required_upstream_artifacts_resolved",
        int(required_loads_df["resolved_relative_path"].astype(str).ne("").sum()),
        int(len(required_loads_df)),
        bool(required_loads_df["resolved_relative_path"].astype(str).ne("").all()),
        "One or more required upstream artifacts could not be resolved from the inventory.",
    )
)

batch1_validation_rows.append(
    validation_row(
        "required_upstream_artifacts_exist_on_disk",
        int(required_loads_df["exists_on_disk"].astype(bool).sum()),
        int(len(required_loads_df)),
        bool(required_loads_df["exists_on_disk"].astype(bool).all()),
        "One or more required upstream artifacts were resolved but do not exist on disk.",
    )
)

batch1_validation_rows.append(
    validation_row(
        "required_upstream_artifacts_loaded",
        int(required_loads_df["loaded"].astype(bool).sum()),
        int(len(required_loads_df)),
        bool(required_loads_df["loaded"].astype(bool).all()),
        "One or more required upstream artifacts could not be loaded.",
    )
)

for _, row in batch1_upstream_loads_df.iterrows():
    expected_rows = row["expected_rows"]
    actual_rows = row["actual_rows"]
    required = bool(row["required"])
    loaded = bool(row["loaded"])

    if pd.isna(expected_rows):
        batch1_validation_rows.append(
            validation_row(
                f"artifact_loaded__{row['artifact_id']}",
                {
                    "path": row["resolved_relative_path"],
                    "loaded": loaded,
                    "required": required,
                },
                "loaded if required; optional files reported informationally",
                loaded if required else True,
                f"Artifact was not loaded: {row['load_error']}",
                severity="error" if required else "info",
            )
        )
    else:
        row_count_passed = loaded and int(actual_rows) == int(expected_rows)
        batch1_validation_rows.append(
            validation_row(
                f"artifact_row_count__{row['artifact_id']}",
                {
                    "path": row["resolved_relative_path"],
                    "actual_rows": None if pd.isna(actual_rows) else int(actual_rows),
                    "loaded": loaded,
                    "required": required,
                },
                int(expected_rows),
                row_count_passed if required else True,
                f"Artifact row count mismatch or artifact failed to load: {row['load_error']}",
                severity="error" if required else "info",
            )
        )

batch1_validation_rows.append(
    validation_row(
        "prior_final_validation_files_resolved",
        int(batch1_prior_validation_df["resolved_relative_path"].astype(str).ne("").sum()),
        int(len(batch1_prior_validation_df)),
        bool(batch1_prior_validation_df["resolved_relative_path"].astype(str).ne("").all()),
        "One or more prior final validation files could not be resolved from the inventory.",
    )
)

batch1_validation_rows.append(
    validation_row(
        "prior_final_validation_files_loaded",
        int(batch1_prior_validation_df["loaded"].astype(bool).sum()),
        int(len(batch1_prior_validation_df)),
        bool(batch1_prior_validation_df["loaded"].astype(bool).all()),
        "One or more prior final validation files could not be loaded.",
    )
)

batch1_validation_rows.append(
    validation_row(
        "prior_final_validations_passed",
        int(batch1_prior_validation_df["validation_passed"].astype(bool).sum()),
        int(len(batch1_prior_validation_df)),
        bool(batch1_prior_validation_df["validation_passed"].astype(bool).all()),
        "One or more upstream final validation files did not pass.",
    )
)

for _, row in batch1_prior_validation_df.iterrows():
    expected_rows = row["expected_rows"]
    actual_rows = row["actual_rows"]

    batch1_validation_rows.append(
        validation_row(
            f"prior_final_validation_row_count__{row['validation_id']}",
            {
                "path": row["resolved_relative_path"],
                "actual_rows": actual_rows,
                "checks_passed": int(row["checks_passed"]),
                "checks_total": int(row["checks_total"]),
                "validation_passed": bool(row["validation_passed"]),
            },
            int(expected_rows),
            bool(row["loaded"]) and int(actual_rows) == int(expected_rows),
            "Prior final validation row count does not match the expected completed notebook output.",
        )
    )

for _, row in batch1_evidence_checks_df.iterrows():
    batch1_validation_rows.append(
        validation_row(
            f"join_keys_available__{row['artifact_id']}",
            {
                "join_key_columns": row["join_key_columns"],
                "row_count": int(row["row_count"]),
                "column_count": int(row["column_count"]),
            },
            "at least one usable join/key identity column",
            bool(row["has_any_join_key"]),
            "Loaded artifact does not expose any expected join/key identity columns.",
        )
    )

batch1_validation_rows.append(
    validation_row(
        "restoration_audit_prompt_seed_runtime_memory_evidence_columns",
        restoration_evidence_columns,
        "prompt/seed/runtime/memory/failure evidence columns where available",
        len(restoration_evidence_columns) >= 8,
        "Restoration audit has too few detected evidence columns for a detailed report.",
    )
)

batch1_validation_rows.append(
    validation_row(
        "prompt_policy_loaded",
        list(prompt_policy.keys()) if isinstance(prompt_policy, dict) else type(prompt_policy).__name__,
        "non-empty JSON object",
        prompt_policy_loaded,
        "Prompt policy JSON was not loaded as a non-empty object.",
    )
)

batch1_validation_rows.append(
    validation_row(
        "optional_evidence_inventory_status",
        f"{optional_loaded_count}/{optional_total_count}",
        "optional files are useful but not blocking",
        True,
        "",
        severity="info",
    )
)

batch1_input_validation_df = pd.DataFrame(batch1_validation_rows)
batch1_input_validation_df.to_csv(BATCH1_VALIDATION_PATH, index=False)

batch1_error_checks_df = batch1_input_validation_df.loc[
    ~batch1_input_validation_df["severity"].astype(str).str.lower().isin(["info", "warning"])
].copy()

batch1_passed = bool(bool_series(batch1_error_checks_df["passed"]).all())

print(f"Saved Batch 1 validation: {rel(BATCH1_VALIDATION_PATH)}")
print(f"Batch 1 error checks passed: {int(bool_series(batch1_error_checks_df['passed']).sum())} / {len(batch1_error_checks_df)}")

display(batch1_input_validation_df)

if not batch1_passed:
    print("Failed Batch 1 checks:")
    display(
        batch1_input_validation_df.loc[
            ~bool_series(batch1_input_validation_df["passed"]),
            ["check_name", "severity", "actual", "expected", "failure_message"],
        ]
    )
    print("Required artifact load details:")
    display(required_loads_df.loc[~required_loads_df["loaded"].astype(bool)])
    print("Prior final validation details:")
    display(batch1_prior_validation_df.loc[~batch1_prior_validation_df["validation_passed"].astype(bool)])
    raise RuntimeError("Batch 1 validation failed. Fix missing/failed upstream inputs before running Batch 2.")

print("Batch 1 passed. Upstream Stable Diffusion artifacts and prior final validations are ready for report consolidation.")

Saved Batch 1 validation: outputs/26_stable_diffusion_report_generation/validation/batch1_input_validation.csv
Batch 1 error checks passed: 28 / 28


,check_name,severity,actual,expected,passed,failure_message,checked_at_utc
0,required_upstream_artifacts_resolved,error,10,10,True,,2026-08-10T11:28:49.466623+00:00
1,required_upstream_artifacts_exist_on_disk,error,10,10,True,,2026-08-10T11:28:49.473273+00:00
2,required_upstream_artifacts_loaded,error,10,10,True,,2026-08-10T11:28:49.473273+00:00
3,artifact_row_count__sd_restoration_audit,error,"{""path"": ""outputs/21_stable_diffusion_restorat...",945,True,,2026-08-10T11:28:49.473273+00:00
4,artifact_row_count__sd_candidate_manifest,info,"{""path"": ""outputs/21_stable_diffusion_restorat...",945,True,,2026-08-10T11:28:49.473273+00:00
5,artifact_loaded__sd_prompt_policy,error,"{""path"": ""outputs/21_stable_diffusion_restorat...","""loaded if required; optional files reported i...",True,,2026-08-10T11:28:49.473273+00:00
6,artifact_row_count__sd_prompt_ablation_rows,info,"{""path"": ""outputs/21_stable_diffusion_restorat...",600,True,,2026-08-10T11:28:49.473273+00:00
7,artifact_row_count__sd_prompt_comparison_pairs,info,"{""path"": ""outputs/21_stable_diffusion_restorat...",480,True,,2026-08-10T11:28:49.473273+00:00
8,artifact_row_count__sd_classical_metrics,error,"{""path"": ""outputs/22_stable_diffusion_classica...",5294,True,,2026-08-10T11:28:49.473273+00:00
9,artifact_row_count__sd_classical_summary,error,"{""path"": ""outputs/22_stable_diffusion_classica...",135,True,,2026-08-10T11:28:49.473273+00:00


Batch 1 passed. Upstream Stable Diffusion artifacts and prior final validations are ready for report consolidation.


In [11]:
# Batch 2 / Cell 1 - Helpers
import hashlib
import re
from typing import Any

BATCH2_REPORT_AUDIT_PATH = OUTPUT_DIRS["metrics"] / "stable_diffusion_report_audit.csv"

REPORT_REGION_KEYWORDS = [
    "full",
    "valid",
    "content",
    "mask",
    "crop",
    "bbox",
    "damage",
    "damaged",
    "undamaged",
    "restored",
]

CANDIDATE_KEY_CANDIDATES = [
    ["candidate_id"],
    ["stable_diffusion_candidate_id"],
    ["restoration_id"],
    ["generation_id"],
    ["output_id"],
    ["case_id", "candidate_id"],
    ["canonical_case_id", "candidate_id"],
    ["case_id", "prompt_variant", "seed"],
    ["canonical_case_id", "prompt_variant", "seed"],
    ["source_case_id", "mask_id", "prompt_variant", "seed"],
    ["case_id", "mask_id", "prompt_variant", "seed"],
    ["dataset", "case_id", "mask_id", "prompt_variant", "seed"],
    ["restored_path"],
    ["restored_image_path"],
    ["output_path"],
]

METRIC_VALUE_COLUMNS = [
    "metric_value",
    "value",
    "score",
    "mean_value",
    "similarity",
    "distance",
    "lpips",
    "psnr",
    "ssim",
    "mae",
    "rmse",
    "clip_similarity",
    "dinov2_similarity",
]

METRIC_LABEL_COLUMNS = [
    "comparison_name",
    "comparison",
    "metric_name",
    "metric",
    "region_name",
    "region",
    "mask_region",
    "aggregation",
    "feature_model",
    "embedding_model",
    "model_name",
]

PATH_COLUMN_PATTERNS = [
    "path",
    "file",
    "uri",
    "url",
    "panel",
    "figure",
    "image",
    "heatmap",
    "overlay",
    "difference",
    "diff",
]

def batch2_is_missing_like(value: Any) -> bool:
    if value is None:
        return True

    try:
        missing_value = pd.isna(value)
        if isinstance(missing_value, (bool, np.bool_)):
            if bool(missing_value):
                return True
    except Exception:
        pass

    if isinstance(value, str):
        return value.strip().lower() in ["", "nan", "none", "null", "<na>"]

    return False

def batch2_first_non_empty(series: pd.Series) -> Any:
    for value in series:
        if not batch2_is_missing_like(value):
            return value
    return np.nan

def batch2_clean_label(value: Any) -> str:
    text_value = str(value).strip().lower()
    text_value = re.sub(r"[^a-z0-9]+", "_", text_value)
    text_value = re.sub(r"_+", "_", text_value).strip("_")
    return text_value[:90] if text_value else "value"

def batch2_existing_columns(df: pd.DataFrame | None, candidates: list[str]) -> list[str]:
    if not isinstance(df, pd.DataFrame):
        return []
    return [column for column in candidates if column in df.columns]

def batch2_make_join_key(df: pd.DataFrame, key_columns: list[str]) -> pd.Series:
    if not key_columns:
        return pd.Series([f"row_{idx:06d}" for idx in range(len(df))], index=df.index)

    key_frame = df[key_columns].copy()
    for column in key_columns:
        key_frame[column] = key_frame[column].astype("string").fillna("<NA>").str.strip()

    return key_frame.agg("||".join, axis=1)

def batch2_choose_anchor_key(df: pd.DataFrame) -> list[str]:
    for key_columns in CANDIDATE_KEY_CANDIDATES:
        if all(column in df.columns for column in key_columns):
            key_series = batch2_make_join_key(df, key_columns)
            unique_count = int(key_series.nunique(dropna=False))
            if unique_count == len(df):
                return key_columns

    for key_columns in CANDIDATE_KEY_CANDIDATES:
        if all(column in df.columns for column in key_columns):
            return key_columns

    return []

def batch2_choose_join_key(anchor_df: pd.DataFrame, other_df: pd.DataFrame, anchor_key_columns: list[str]) -> list[str]:
    if all(column in other_df.columns for column in anchor_key_columns):
        return anchor_key_columns

    for key_columns in CANDIDATE_KEY_CANDIDATES:
        if all(column in anchor_df.columns and column in other_df.columns for column in key_columns):
            return key_columns

    shared_identity_columns = [
        column for column in [
            "case_id",
            "canonical_case_id",
            "source_case_id",
            "dataset",
            "mask_id",
            "mask_type",
            "damage_profile",
            "prompt_variant",
            "seed",
            "restored_path",
            "restored_image_path",
        ]
        if column in anchor_df.columns and column in other_df.columns
    ]

    return shared_identity_columns[:4]

def batch2_group_one_row_per_key(df: pd.DataFrame, key_columns: list[str]) -> pd.DataFrame:
    working_df = df.copy()
    working_df["__batch2_join_key"] = batch2_make_join_key(working_df, key_columns)

    non_key_columns = [
        column for column in working_df.columns
        if column not in key_columns and column != "__batch2_join_key"
    ]

    grouped_df = (
        working_df
        .groupby("__batch2_join_key", dropna=False, as_index=False)
        .agg({column: batch2_first_non_empty for column in key_columns + non_key_columns})
    )

    return grouped_df

def batch2_prefix_columns(df: pd.DataFrame, prefix: str, keep_columns: list[str]) -> pd.DataFrame:
    renamed_df = df.copy()
    rename_map = {
        column: f"{prefix}__{batch2_clean_label(column)}"
        for column in renamed_df.columns
        if column not in keep_columns
    }
    return renamed_df.rename(columns=rename_map)

def batch2_filter_report_regions(df: pd.DataFrame) -> tuple[pd.DataFrame, str]:
    region_columns = [
        column for column in df.columns
        if "region" in column.lower()
    ]

    if not region_columns:
        return df.copy(), "no_region_column"

    region_mask = pd.Series(False, index=df.index)
    for column in region_columns:
        comparable = df[column].astype(str).str.lower()
        for keyword in REPORT_REGION_KEYWORDS:
            region_mask = region_mask | comparable.str.contains(keyword, na=False)

    if int(region_mask.sum()) == 0:
        return df.copy(), "region_filter_empty_retained_all"

    return df.loc[region_mask].copy(), f"filtered_to_{int(region_mask.sum())}_rows"

def batch2_stable_candidate_id(join_key: str) -> str:
    digest = hashlib.sha1(str(join_key).encode("utf-8")).hexdigest()[:16]
    return f"sd_candidate_{digest}"

print(f"Batch 2 output path: {rel(BATCH2_REPORT_AUDIT_PATH)}")

Batch 2 output path: outputs/26_stable_diffusion_report_generation/metrics/stable_diffusion_report_audit.csv


In [12]:
# Batch 2 / Cell 2 - Anchor on restoration candidates
sd_restoration_audit_df = UPSTREAM_DATA.get("sd_restoration_audit")

if not isinstance(sd_restoration_audit_df, pd.DataFrame):
    raise RuntimeError("Batch 2 requires UPSTREAM_DATA['sd_restoration_audit'] from Batch 1.")

anchor_key_columns = batch2_choose_anchor_key(sd_restoration_audit_df)

if not anchor_key_columns:
    raise RuntimeError(
        "Could not identify candidate-level key columns in the Stable Diffusion restoration audit."
    )

batch2_report_audit_df = batch2_group_one_row_per_key(
    sd_restoration_audit_df,
    anchor_key_columns,
)

batch2_report_audit_df.insert(
    0,
    "report_candidate_id",
    batch2_report_audit_df["__batch2_join_key"].map(batch2_stable_candidate_id),
)

batch2_report_audit_df = batch2_report_audit_df.rename(
    columns={"__batch2_join_key": "report_candidate_key"}
)

anchor_candidate_count = int(len(batch2_report_audit_df))
source_restoration_rows = int(len(sd_restoration_audit_df))

print(f"Anchor key columns: {anchor_key_columns}")
print(f"Restoration audit source rows: {source_restoration_rows:,}")
print(f"Consolidated candidate rows: {anchor_candidate_count:,}")

display(batch2_report_audit_df.head())

Anchor key columns: ['candidate_id']
Restoration audit source rows: 945
Consolidated candidate rows: 945


,report_candidate_id,report_candidate_key,candidate_id,case_id,painting_id,mask_id,mask_type,generator_name,generator_version,generated_at_utc,...,after_cuda_max_memory_allocated_bytes,python_version,operating_system,processor,torch_version,cuda_available,cuda_device_count,cuda_device_name,diffusers_version,attempt_count
0,sd_candidate_bb26f917cfe8ca5a,sd__can__p001__loss_large__p00_generic__s2026_...,sd__can__p001__loss_large__p00_generic__s2026_...,p001_loss_large,p001,p001_loss_large,loss_large,canonical_damage_generator,2.0.0,2026-07-24T18:47:20.616478+00:00,...,2821253632,3.12.6,Windows-11-10.0.26200-SP0,"Intel64 Family 6 Model 165 Stepping 2, Genuine...",2.5.1+cu121,True,1,NVIDIA GeForce RTX 3060 Laptop GPU,0.27.2,1
1,sd_candidate_be57fcd9fc8fc503,sd__can__p001__loss_small__p00_generic__s2026_...,sd__can__p001__loss_small__p00_generic__s2026_...,p001_loss_small,p001,p001_loss_small,loss_small,canonical_damage_generator,2.0.0,2026-07-24T18:47:20.616478+00:00,...,2821253632,3.12.6,Windows-11-10.0.26200-SP0,"Intel64 Family 6 Model 165 Stepping 2, Genuine...",2.5.1+cu121,True,1,NVIDIA GeForce RTX 3060 Laptop GPU,0.27.2,1
2,sd_candidate_705cf95c6bc65a93,sd__can__p001__mixed_damage__p00_generic__s202...,sd__can__p001__mixed_damage__p00_generic__s202...,p001_mixed_damage,p001,p001_mixed_damage,mixed_damage,canonical_damage_generator,2.0.0,2026-07-24T18:47:20.616478+00:00,...,2821253632,3.12.6,Windows-11-10.0.26200-SP0,"Intel64 Family 6 Model 165 Stepping 2, Genuine...",2.5.1+cu121,True,1,NVIDIA GeForce RTX 3060 Laptop GPU,0.27.2,1
3,sd_candidate_af40c9a9179f955a,sd__can__p001__scratch_thin__p00_generic__s202...,sd__can__p001__scratch_thin__p00_generic__s202...,p001_scratch_thin,p001,p001_scratch_thin,scratch_thin,canonical_damage_generator,2.0.0,2026-07-24T18:47:20.616478+00:00,...,2821253632,3.12.6,Windows-11-10.0.26200-SP0,"Intel64 Family 6 Model 165 Stepping 2, Genuine...",2.5.1+cu121,True,1,NVIDIA GeForce RTX 3060 Laptop GPU,0.27.2,1
4,sd_candidate_a68370c87b7f46d1,sd__can__p001__zero__p00_generic__s2026__3be10...,sd__can__p001__zero__p00_generic__s2026__3be10...,p001_zero_control,p001,p001_zero_control,zero_control,canonical_damage_generator,2.0.0,2026-07-24T18:47:20.616478+00:00,...,2821253632,3.12.6,Windows-11-10.0.26200-SP0,"Intel64 Family 6 Model 165 Stepping 2, Genuine...",2.5.1+cu121,True,1,NVIDIA GeForce RTX 3060 Laptop GPU,0.27.2,1


In [13]:
# Batch 2 / Cell 3 - Aggregate metric tables to candidate level
def batch2_metric_wide_table(
    artifact_id: str,
    namespace: str,
    anchor_df: pd.DataFrame,
    anchor_key_columns: list[str],
) -> tuple[pd.DataFrame, dict]:
    source_df = UPSTREAM_DATA.get(artifact_id)

    record = {
        "artifact_id": artifact_id,
        "namespace": namespace,
        "loaded": isinstance(source_df, pd.DataFrame),
        "source_rows": int(len(source_df)) if isinstance(source_df, pd.DataFrame) else 0,
        "filtered_rows": 0,
        "join_key_columns": [],
        "value_mode": "",
        "wide_rows": 0,
        "wide_columns": 0,
        "metric_columns_created": 0,
        "status": "not_loaded",
    }

    if not isinstance(source_df, pd.DataFrame) or source_df.empty:
        return pd.DataFrame(), record

    join_key_columns = batch2_choose_join_key(anchor_df, source_df, anchor_key_columns)
    record["join_key_columns"] = join_key_columns

    if not join_key_columns:
        record["status"] = "no_join_key"
        return pd.DataFrame(), record

    filtered_df, region_status = batch2_filter_report_regions(source_df)
    record["filtered_rows"] = int(len(filtered_df))
    record["region_status"] = region_status

    working_df = filtered_df.copy()
    working_df["__batch2_join_key"] = batch2_make_join_key(working_df, join_key_columns)

    value_column = next(
        (column for column in METRIC_VALUE_COLUMNS if column in working_df.columns),
        None,
    )

    if value_column is not None:
        label_columns = [
            column for column in METRIC_LABEL_COLUMNS
            if column in working_df.columns and column != value_column
        ]

        if label_columns:
            working_df["__metric_label"] = working_df[label_columns].apply(
                lambda row: "__".join(
                    batch2_clean_label(value)
                    for value in row.tolist()
                    if not batch2_is_missing_like(value)
                ),
                axis=1,
            )
        else:
            working_df["__metric_label"] = batch2_clean_label(value_column)

        working_df["__metric_label"] = working_df["__metric_label"].replace("", batch2_clean_label(value_column))
        working_df["__metric_value"] = pd.to_numeric(working_df[value_column], errors="coerce")

        wide_df = (
            working_df
            .pivot_table(
                index="__batch2_join_key",
                columns="__metric_label",
                values="__metric_value",
                aggfunc="mean",
            )
            .reset_index()
        )

        wide_df.columns.name = None
        record["value_mode"] = f"long_value_column:{value_column}"

    else:
        metadata_columns = set(join_key_columns + METRIC_LABEL_COLUMNS + PATH_COLUMN_PATTERNS)
        numeric_columns = [
            column for column in working_df.columns
            if column not in metadata_columns
            and column != "__batch2_join_key"
            and pd.api.types.is_numeric_dtype(working_df[column])
        ]

        if not numeric_columns:
            record["status"] = "no_metric_value_columns"
            return pd.DataFrame(), record

        wide_df = (
            working_df
            .groupby("__batch2_join_key", dropna=False, as_index=False)[numeric_columns]
            .mean()
        )

        record["value_mode"] = "wide_numeric_columns"

    wide_df = batch2_prefix_columns(
        wide_df,
        namespace,
        keep_columns=["__batch2_join_key"],
    )

    record["wide_rows"] = int(len(wide_df))
    record["wide_columns"] = int(len(wide_df.columns))
    record["metric_columns_created"] = int(len(wide_df.columns) - 1)
    record["status"] = "built"

    return wide_df, record

metric_sources = [
    ("sd_classical_metrics", "classical"),
    ("sd_lpips_metrics", "lpips"),
    ("sd_feature_metrics", "feature"),
]

batch2_metric_tables = {}
batch2_metric_records = []

for artifact_id, namespace in metric_sources:
    metric_wide_df, metric_record = batch2_metric_wide_table(
        artifact_id=artifact_id,
        namespace=namespace,
        anchor_df=sd_restoration_audit_df,
        anchor_key_columns=anchor_key_columns,
    )

    batch2_metric_tables[artifact_id] = metric_wide_df
    batch2_metric_records.append(metric_record)

batch2_metric_records_df = pd.DataFrame(batch2_metric_records)

print("Metric aggregation summary:")
display(batch2_metric_records_df)

Metric aggregation summary:


,artifact_id,namespace,loaded,source_rows,filtered_rows,join_key_columns,value_mode,wide_rows,wide_columns,metric_columns_created,status,region_status
0,sd_classical_metrics,classical,True,5294,4443,[candidate_id],wide_numeric_columns,945,26,25,built,filtered_to_4443_rows
1,sd_lpips_metrics,lpips,True,2724,2724,[candidate_id],wide_numeric_columns,945,27,26,built,filtered_to_2724_rows
2,sd_feature_metrics,feature,True,2724,2724,"[case_id, source_case_id, mask_id, mask_type]",wide_numeric_columns,465,21,20,built,filtered_to_2724_rows


In [14]:
# Batch 2 / Cell 4 - Aggregate difference-map links
def batch2_unique_columns(columns: list[str]) -> list[str]:
    counts = {}
    unique_columns = []

    for column in columns:
        column = str(column)
        count = counts.get(column, 0)
        if count == 0:
            unique_columns.append(column)
        else:
            unique_columns.append(f"{column}__duplicate_{count}")
        counts[column] = count + 1

    return unique_columns


def batch2_difference_link_table(
    artifact_id: str,
    namespace: str,
    anchor_df: pd.DataFrame,
    anchor_key_columns: list[str],
) -> tuple[pd.DataFrame, dict]:
    source_df = UPSTREAM_DATA.get(artifact_id)

    record = {
        "artifact_id": artifact_id,
        "namespace": namespace,
        "loaded": isinstance(source_df, pd.DataFrame),
        "source_rows": int(len(source_df)) if isinstance(source_df, pd.DataFrame) else 0,
        "join_key_columns": [],
        "path_columns": [],
        "status_columns": [],
        "wide_rows": 0,
        "wide_columns": 0,
        "status": "not_loaded",
    }

    if not isinstance(source_df, pd.DataFrame) or source_df.empty:
        return pd.DataFrame(), record

    working_source_df = source_df.copy()
    working_source_df.columns = batch2_unique_columns(list(working_source_df.columns))

    join_key_columns = batch2_choose_join_key(anchor_df, working_source_df, anchor_key_columns)
    record["join_key_columns"] = join_key_columns

    if not join_key_columns:
        record["status"] = "no_join_key"
        return pd.DataFrame(), record

    path_columns = [
        column for column in working_source_df.columns
        if column not in join_key_columns
        and any(pattern in column.lower() for pattern in PATH_COLUMN_PATTERNS)
    ]

    useful_status_columns = [
        column for column in working_source_df.columns
        if column not in join_key_columns
        and any(pattern in column.lower() for pattern in ["status", "error", "failure"])
    ]

    selected_columns = []
    for column in join_key_columns + path_columns + useful_status_columns:
        if column in working_source_df.columns and column not in selected_columns:
            selected_columns.append(column)

    record["path_columns"] = path_columns
    record["status_columns"] = useful_status_columns

    if not path_columns and not useful_status_columns:
        record["status"] = "no_link_or_status_columns"
        return pd.DataFrame(), record

    working_df = working_source_df.loc[:, selected_columns].copy()
    working_df["__batch2_join_key"] = batch2_make_join_key(working_df, join_key_columns)

    aggregate_columns = [
        column for column in working_df.columns
        if column not in join_key_columns and column != "__batch2_join_key"
    ]

    grouped_rows = []

    for join_key, group_df in working_df.groupby("__batch2_join_key", dropna=False):
        row = {"__batch2_join_key": join_key}

        for column in aggregate_columns:
            row[column] = batch2_first_non_empty(group_df[column])

        grouped_rows.append(row)

    wide_df = pd.DataFrame(grouped_rows)

    if wide_df.empty:
        record["status"] = "empty_after_grouping"
        return pd.DataFrame(), record

    wide_df = batch2_prefix_columns(
        wide_df,
        namespace,
        keep_columns=["__batch2_join_key"],
    )

    record["wide_rows"] = int(len(wide_df))
    record["wide_columns"] = int(len(wide_df.columns))
    record["status"] = "built"

    return wide_df, record


batch2_difference_links_df, batch2_difference_record = batch2_difference_link_table(
    artifact_id="sd_difference_map_audit",
    namespace="difference_map",
    anchor_df=sd_restoration_audit_df,
    anchor_key_columns=anchor_key_columns,
)

print("Difference-map link aggregation summary:")
display(pd.DataFrame([batch2_difference_record]))

if not batch2_difference_links_df.empty:
    display(batch2_difference_links_df.head())

Difference-map link aggregation summary:


,artifact_id,namespace,loaded,source_rows,join_key_columns,path_columns,status_columns,wide_rows,wide_columns,status
0,sd_difference_map_audit,difference_map,True,945,[candidate_id],"[clean_path, damaged_path, restored_path, mask...","[status, error_map_version, error_vmin, error_...",945,60,built


,__batch2_join_key,difference_map__clean_path,difference_map__damaged_path,difference_map__restored_path,difference_map__mask_path,difference_map__damaged_error_map_path,difference_map__damaged_error_map_filename,difference_map__damaged_error_filename_length,difference_map__damaged_error_file_size_bytes,difference_map__restored_error_map_path,...,difference_map__damaged_error_boundary_mean,difference_map__damaged_error_boundary_std,difference_map__restored_error_boundary_mean,difference_map__restored_error_boundary_std,difference_map__damaged_error_outside_mask_mean,difference_map__damaged_error_outside_mask_std,difference_map__restored_error_outside_mask_mean,difference_map__restored_error_outside_mask_std,difference_map__damaged_error_mean_masked,difference_map__restored_error_mean_masked
0,sd__can__p001__loss_large__p00_generic__s2026_...,data/processed/clean/p001_clean.png,data/processed/masked/p001_loss_large_damaged.png,data/processed/restored/stable_diffusion/canon...,data/processed/masks/p001_loss_large_mask.png,outputs/23_stable_diffusion_difference_maps/ma...,dm_000001_der.png,17,104423,outputs/23_stable_diffusion_difference_maps/ma...,...,112.041573,113.267372,2.859372,5.360912,0.0,0.0,0.0,0.0,220.027664,12.347485
1,sd__can__p001__loss_small__p00_generic__s2026_...,data/processed/clean/p001_clean.png,data/processed/masked/p001_loss_small_damaged.png,data/processed/restored/stable_diffusion/canon...,data/processed/masks/p001_loss_small_mask.png,outputs/23_stable_diffusion_difference_maps/ma...,dm_000002_der.png,17,38471,outputs/23_stable_diffusion_difference_maps/ma...,...,111.966988,116.925644,2.246670,3.794370,0.0,0.0,0.0,0.0,232.473816,5.175668
2,sd__can__p001__mixed_damage__p00_generic__s202...,data/processed/clean/p001_clean.png,data/processed/masked/p001_mixed_damage_damage...,data/processed/restored/stable_diffusion/canon...,data/processed/masks/p001_mixed_damage_mask.png,outputs/23_stable_diffusion_difference_maps/ma...,dm_000003_der.png,17,93867,outputs/23_stable_diffusion_difference_maps/ma...,...,93.751656,111.918083,7.490422,21.631472,0.0,0.0,0.0,0.0,223.858749,15.197902
3,sd__can__p001__scratch_thin__p00_generic__s202...,data/processed/clean/p001_clean.png,data/processed/masked/p001_scratch_thin_damage...,data/processed/restored/stable_diffusion/canon...,data/processed/masks/p001_scratch_thin_mask.png,outputs/23_stable_diffusion_difference_maps/ma...,dm_000004_der.png,17,40850,outputs/23_stable_diffusion_difference_maps/ma...,...,80.732246,107.757797,21.895889,41.652641,0.0,0.0,0.0,0.0,221.437164,60.069550
4,sd__can__p001__zero__p00_generic__s2026__3be10...,data/processed/clean/p001_clean.png,data/processed/masked/p001_zero_control_damage...,data/processed/restored/stable_diffusion/canon...,data/processed/masks/p001_zero_control_mask.png,outputs/23_stable_diffusion_difference_maps/ma...,dm_000005_der.png,17,4023,outputs/23_stable_diffusion_difference_maps/ma...,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,NaN,NaN


In [15]:
# Batch 2 / Cell 5 - Join candidate audit pieces
def batch2_left_join_by_report_key(
    base_df: pd.DataFrame,
    addition_df: pd.DataFrame,
    source_name: str,
) -> tuple[pd.DataFrame, dict]:
    record = {
        "source_name": source_name,
        "source_rows": int(len(addition_df)) if isinstance(addition_df, pd.DataFrame) else 0,
        "source_columns": int(len(addition_df.columns)) if isinstance(addition_df, pd.DataFrame) else 0,
        "matched_candidate_rows": 0,
        "added_columns": 0,
        "status": "not_joined",
    }

    if not isinstance(addition_df, pd.DataFrame) or addition_df.empty:
        record["status"] = "empty"
        return base_df, record

    if "__batch2_join_key" not in addition_df.columns:
        record["status"] = "missing_join_key"
        return base_df, record

    addition_keys = set(addition_df["__batch2_join_key"].astype(str).tolist())
    base_join_key = base_df["report_candidate_key"].astype(str)
    record["matched_candidate_rows"] = int(base_join_key.isin(addition_keys).sum())

    joined_df = base_df.merge(
        addition_df,
        how="left",
        left_on="report_candidate_key",
        right_on="__batch2_join_key",
    )

    if "__batch2_join_key" in joined_df.columns:
        joined_df = joined_df.drop(columns=["__batch2_join_key"])

    record["added_columns"] = int(len(joined_df.columns) - len(base_df.columns))
    record["status"] = "joined"

    return joined_df, record

batch2_join_records = []

for artifact_id, metric_wide_df in batch2_metric_tables.items():
    batch2_report_audit_df, join_record = batch2_left_join_by_report_key(
        batch2_report_audit_df,
        metric_wide_df,
        artifact_id,
    )
    batch2_join_records.append(join_record)

batch2_report_audit_df, difference_join_record = batch2_left_join_by_report_key(
    batch2_report_audit_df,
    batch2_difference_links_df,
    "sd_difference_map_audit",
)
batch2_join_records.append(difference_join_record)

batch2_join_records_df = pd.DataFrame(batch2_join_records)

print("Batch 2 join summary:")
display(batch2_join_records_df)

print(f"Current report audit shape: {batch2_report_audit_df.shape[0]:,} rows x {batch2_report_audit_df.shape[1]:,} columns")
display(batch2_report_audit_df.head())

Batch 2 join summary:


,source_name,source_rows,source_columns,matched_candidate_rows,added_columns,status
0,sd_classical_metrics,945,26,945,25,joined
1,sd_lpips_metrics,945,27,945,26,joined
2,sd_feature_metrics,465,21,0,20,joined
3,sd_difference_map_audit,945,60,945,59,joined


Current report audit shape: 945 rows x 414 columns


,report_candidate_id,report_candidate_key,candidate_id,case_id,painting_id,mask_id,mask_type,generator_name,generator_version,generated_at_utc,...,difference_map__damaged_error_boundary_mean,difference_map__damaged_error_boundary_std,difference_map__restored_error_boundary_mean,difference_map__restored_error_boundary_std,difference_map__damaged_error_outside_mask_mean,difference_map__damaged_error_outside_mask_std,difference_map__restored_error_outside_mask_mean,difference_map__restored_error_outside_mask_std,difference_map__damaged_error_mean_masked,difference_map__restored_error_mean_masked
0,sd_candidate_bb26f917cfe8ca5a,sd__can__p001__loss_large__p00_generic__s2026_...,sd__can__p001__loss_large__p00_generic__s2026_...,p001_loss_large,p001,p001_loss_large,loss_large,canonical_damage_generator,2.0.0,2026-07-24T18:47:20.616478+00:00,...,112.041573,113.267372,2.859372,5.360912,0.0,0.0,0.0,0.0,220.027664,12.347485
1,sd_candidate_be57fcd9fc8fc503,sd__can__p001__loss_small__p00_generic__s2026_...,sd__can__p001__loss_small__p00_generic__s2026_...,p001_loss_small,p001,p001_loss_small,loss_small,canonical_damage_generator,2.0.0,2026-07-24T18:47:20.616478+00:00,...,111.966988,116.925644,2.246670,3.794370,0.0,0.0,0.0,0.0,232.473816,5.175668
2,sd_candidate_705cf95c6bc65a93,sd__can__p001__mixed_damage__p00_generic__s202...,sd__can__p001__mixed_damage__p00_generic__s202...,p001_mixed_damage,p001,p001_mixed_damage,mixed_damage,canonical_damage_generator,2.0.0,2026-07-24T18:47:20.616478+00:00,...,93.751656,111.918083,7.490422,21.631472,0.0,0.0,0.0,0.0,223.858749,15.197902
3,sd_candidate_af40c9a9179f955a,sd__can__p001__scratch_thin__p00_generic__s202...,sd__can__p001__scratch_thin__p00_generic__s202...,p001_scratch_thin,p001,p001_scratch_thin,scratch_thin,canonical_damage_generator,2.0.0,2026-07-24T18:47:20.616478+00:00,...,80.732246,107.757797,21.895889,41.652641,0.0,0.0,0.0,0.0,221.437164,60.069550
4,sd_candidate_a68370c87b7f46d1,sd__can__p001__zero__p00_generic__s2026__3be10...,sd__can__p001__zero__p00_generic__s2026__3be10...,p001_zero_control,p001,p001_zero_control,zero_control,canonical_damage_generator,2.0.0,2026-07-24T18:47:20.616478+00:00,...,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,NaN,NaN


In [16]:
# Batch 2 / Cell 6 - Reorder, write report audit, and validate
def batch2_columns_matching(df: pd.DataFrame, patterns: list[str]) -> list[str]:
    matched_columns = []
    for column in df.columns:
        lowered = column.lower()
        if any(pattern in lowered for pattern in patterns):
            matched_columns.append(column)
    return matched_columns

priority_columns = []

for column in ["report_candidate_id", "report_candidate_key"]:
    if column in batch2_report_audit_df.columns:
        priority_columns.append(column)

priority_columns.extend([
    column for column in anchor_key_columns
    if column in batch2_report_audit_df.columns and column not in priority_columns
])

prompt_columns = batch2_columns_matching(batch2_report_audit_df, ["prompt"])
seed_columns = batch2_columns_matching(batch2_report_audit_df, ["seed"])
runtime_columns = batch2_columns_matching(batch2_report_audit_df, ["runtime", "seconds", "duration", "elapsed"])
memory_columns = batch2_columns_matching(batch2_report_audit_df, ["memory", "vram", "gpu_memory", "allocated", "reserved"])
status_columns = batch2_columns_matching(batch2_report_audit_df, ["status", "failure", "error"])
path_columns = batch2_columns_matching(batch2_report_audit_df, ["path", "uri", "url"])
metric_columns = batch2_columns_matching(batch2_report_audit_df, ["classical__", "lpips__", "feature__"])
difference_columns = batch2_columns_matching(batch2_report_audit_df, ["difference_map__"])

ordered_columns = []
for column_group in [
    priority_columns,
    prompt_columns,
    seed_columns,
    runtime_columns,
    memory_columns,
    status_columns,
    path_columns,
    difference_columns,
    metric_columns,
]:
    for column in column_group:
        if column in batch2_report_audit_df.columns and column not in ordered_columns:
            ordered_columns.append(column)

remaining_columns = [
    column for column in batch2_report_audit_df.columns
    if column not in ordered_columns
]

batch2_report_audit_df = batch2_report_audit_df[ordered_columns + remaining_columns].copy()

BATCH2_REPORT_AUDIT_PATH.parent.mkdir(parents=True, exist_ok=True)
batch2_report_audit_df.to_csv(BATCH2_REPORT_AUDIT_PATH, index=False)

written_shape = csv_shape(BATCH2_REPORT_AUDIT_PATH)
metric_column_count = len(metric_columns)
difference_link_column_count = len(difference_columns)
prompt_column_count = len(prompt_columns)
seed_column_count = len(seed_columns)
runtime_column_count = len(runtime_columns)
memory_column_count = len(memory_columns)

batch2_validation_rows = [
    validation_row(
        "report_audit_written",
        rel(BATCH2_REPORT_AUDIT_PATH),
        "file exists",
        BATCH2_REPORT_AUDIT_PATH.is_file(),
        "Stable Diffusion report audit CSV was not written.",
    ),
    validation_row(
        "report_audit_row_count_matches_candidate_count",
        written_shape[0],
        anchor_candidate_count,
        written_shape[0] == anchor_candidate_count,
        "Report audit row count does not match candidate-level restoration audit count.",
    ),
    validation_row(
        "report_audit_candidate_ids_unique",
        int(batch2_report_audit_df["report_candidate_id"].nunique(dropna=False)),
        int(len(batch2_report_audit_df)),
        int(batch2_report_audit_df["report_candidate_id"].nunique(dropna=False)) == int(len(batch2_report_audit_df)),
        "Report candidate IDs are not unique.",
    ),
    validation_row(
        "prompt_columns_present",
        prompt_column_count,
        "> 0",
        prompt_column_count > 0,
        "No prompt columns were detected in the report audit.",
    ),
    validation_row(
        "seed_columns_present",
        seed_column_count,
        "> 0",
        seed_column_count > 0,
        "No seed columns were detected in the report audit.",
    ),
    validation_row(
        "runtime_columns_present",
        runtime_column_count,
        "> 0",
        runtime_column_count > 0,
        "No runtime/duration columns were detected in the report audit.",
    ),
    validation_row(
        "memory_columns_present",
        memory_column_count,
        "> 0",
        memory_column_count > 0,
        "No memory/GPU memory columns were detected in the report audit.",
    ),
    validation_row(
        "metric_columns_joined",
        metric_column_count,
        "> 0",
        metric_column_count > 0,
        "No classical/LPIPS/feature metric columns were joined into the report audit.",
    ),
    validation_row(
        "difference_map_columns_joined",
        difference_link_column_count,
        "> 0",
        difference_link_column_count > 0,
        "No difference-map link columns were joined into the report audit.",
    ),
]

batch2_validation_df = pd.DataFrame(batch2_validation_rows)
batch2_passed = bool(bool_series(batch2_validation_df["passed"]).all())

stage_manifest = read_json_if_exists(STAGE_MANIFEST_PATH)
stage_manifest.update(
    {
        "notebook_id": NOTEBOOK_ID,
        "notebook_title": NOTEBOOK_TITLE,
        "stage": "batch2_consolidated_report_audit",
        "stage_status": "passed" if batch2_passed else "failed",
        "updated_at_utc": utc_now_iso(),
        "batch2": {
            "status": "passed" if batch2_passed else "failed",
            "report_audit": rel(BATCH2_REPORT_AUDIT_PATH),
            "anchor_key_columns": anchor_key_columns,
            "source_restoration_rows": source_restoration_rows,
            "candidate_rows": anchor_candidate_count,
            "output_rows": int(written_shape[0] or 0),
            "output_columns": int(written_shape[1] or 0),
            "metric_columns": metric_column_count,
            "difference_map_columns": difference_link_column_count,
            "prompt_columns": prompt_column_count,
            "seed_columns": seed_column_count,
            "runtime_columns": runtime_column_count,
            "memory_columns": memory_column_count,
        },
    }
)
write_json(STAGE_MANIFEST_PATH, stage_manifest)

print(f"Saved Stable Diffusion report audit: {rel(BATCH2_REPORT_AUDIT_PATH)}")
print(f"Report audit shape: {batch2_report_audit_df.shape[0]:,} rows x {batch2_report_audit_df.shape[1]:,} columns")
print(f"Batch 2 checks passed: {int(bool_series(batch2_validation_df['passed']).sum())} / {len(batch2_validation_df)}")

display(batch2_validation_df)

if not batch2_passed:
    display(
        batch2_validation_df.loc[
            ~bool_series(batch2_validation_df["passed"]),
            ["check_name", "actual", "expected", "failure_message"],
        ]
    )
    raise RuntimeError("Batch 2 validation failed. Fix the consolidation issue before running Batch 3.")

display(batch2_report_audit_df.head())

print("Batch 2 passed. Consolidated Stable Diffusion report audit is ready for report-level analysis.")

Saved Stable Diffusion report audit: outputs/26_stable_diffusion_report_generation/metrics/stable_diffusion_report_audit.csv
Report audit shape: 945 rows x 414 columns
Batch 2 checks passed: 9 / 9


,check_name,severity,actual,expected,passed,failure_message,checked_at_utc
0,report_audit_written,error,"""outputs/26_stable_diffusion_report_generation...","""file exists""",True,,2026-08-10T11:29:00.293401+00:00
1,report_audit_row_count_matches_candidate_count,error,945,945,True,,2026-08-10T11:29:00.293401+00:00
2,report_audit_candidate_ids_unique,error,945,945,True,,2026-08-10T11:29:00.293401+00:00
3,prompt_columns_present,error,18,"""> 0""",True,,2026-08-10T11:29:00.293401+00:00
4,seed_columns_present,error,15,"""> 0""",True,,2026-08-10T11:29:00.293401+00:00
5,runtime_columns_present,error,6,"""> 0""",True,,2026-08-10T11:29:00.293401+00:00
6,memory_columns_present,error,6,"""> 0""",True,,2026-08-10T11:29:00.293401+00:00
7,metric_columns_joined,error,71,"""> 0""",True,,2026-08-10T11:29:00.293401+00:00
8,difference_map_columns_joined,error,59,"""> 0""",True,,2026-08-10T11:29:00.293401+00:00


,report_candidate_id,report_candidate_key,candidate_id,prompt_ablation_subset,prompt_policy_id,prompt_variant_id,prompt_template_name,prompt_variant_family,prompt_variant_order,prompt_template,...,outside_mask_max_abs_diff,python_version,operating_system,processor,torch_version,cuda_available,cuda_device_count,cuda_device_name,diffusers_version,attempt_count
0,sd_candidate_bb26f917cfe8ca5a,sd__can__p001__loss_large__p00_generic__s2026_...,sd__can__p001__loss_large__p00_generic__s2026_...,False,sd_inpaint_prompt_ablation_v1,p00_generic,generic_restoration,generic,0,restore the damaged or missing area of the pai...,...,0.0,3.12.6,Windows-11-10.0.26200-SP0,"Intel64 Family 6 Model 165 Stepping 2, Genuine...",2.5.1+cu121,True,1,NVIDIA GeForce RTX 3060 Laptop GPU,0.27.2,1
1,sd_candidate_be57fcd9fc8fc503,sd__can__p001__loss_small__p00_generic__s2026_...,sd__can__p001__loss_small__p00_generic__s2026_...,False,sd_inpaint_prompt_ablation_v1,p00_generic,generic_restoration,generic,0,restore the damaged or missing area of the pai...,...,0.0,3.12.6,Windows-11-10.0.26200-SP0,"Intel64 Family 6 Model 165 Stepping 2, Genuine...",2.5.1+cu121,True,1,NVIDIA GeForce RTX 3060 Laptop GPU,0.27.2,1
2,sd_candidate_705cf95c6bc65a93,sd__can__p001__mixed_damage__p00_generic__s202...,sd__can__p001__mixed_damage__p00_generic__s202...,False,sd_inpaint_prompt_ablation_v1,p00_generic,generic_restoration,generic,0,restore the damaged or missing area of the pai...,...,0.0,3.12.6,Windows-11-10.0.26200-SP0,"Intel64 Family 6 Model 165 Stepping 2, Genuine...",2.5.1+cu121,True,1,NVIDIA GeForce RTX 3060 Laptop GPU,0.27.2,1
3,sd_candidate_af40c9a9179f955a,sd__can__p001__scratch_thin__p00_generic__s202...,sd__can__p001__scratch_thin__p00_generic__s202...,False,sd_inpaint_prompt_ablation_v1,p00_generic,generic_restoration,generic,0,restore the damaged or missing area of the pai...,...,0.0,3.12.6,Windows-11-10.0.26200-SP0,"Intel64 Family 6 Model 165 Stepping 2, Genuine...",2.5.1+cu121,True,1,NVIDIA GeForce RTX 3060 Laptop GPU,0.27.2,1
4,sd_candidate_a68370c87b7f46d1,sd__can__p001__zero__p00_generic__s2026__3be10...,sd__can__p001__zero__p00_generic__s2026__3be10...,False,sd_inpaint_prompt_ablation_v1,p00_generic,generic_restoration,generic,0,restore the damaged or missing area of the pai...,...,NaN,3.12.6,Windows-11-10.0.26200-SP0,"Intel64 Family 6 Model 165 Stepping 2, Genuine...",2.5.1+cu121,True,1,NVIDIA GeForce RTX 3060 Laptop GPU,0.27.2,1


Batch 2 passed. Consolidated Stable Diffusion report audit is ready for report-level analysis.


In [17]:
# Batch 3 / Cell 1 - Helpers
from typing import Any
import re

BATCH3_SUMMARY_PATH = OUTPUT_DIRS["analysis"] / "stable_diffusion_report_summary.csv"
BATCH3_REPRESENTATIVE_PATH = OUTPUT_DIRS["analysis"] / "stable_diffusion_report_representative_candidates.csv"

def batch3_clean_text(value: Any) -> str:
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass
    return str(value).strip()

def batch3_clean_label(value: Any) -> str:
    text_value = batch3_clean_text(value).lower()
    text_value = re.sub(r"[^a-z0-9]+", "_", text_value)
    text_value = re.sub(r"_+", "_", text_value).strip("_")
    return text_value or "unknown"

def batch3_columns_matching(df: pd.DataFrame, patterns: list[str], exclude: list[str] | None = None) -> list[str]:
    exclude = exclude or []
    matched = []
    for column in df.columns:
        lowered = column.lower()
        if any(pattern in lowered for pattern in patterns) and not any(pattern in lowered for pattern in exclude):
            matched.append(column)
    return matched

def batch3_is_real_metric_column(column: str) -> bool:
    lowered = column.lower()

    blocked_tokens = [
        "path", "uri", "url", "status", "failure", "exception", "traceback",
        "index", "seed", "runtime", "seconds", "duration", "elapsed",
        "memory", "vram", "allocated", "reserved", "device",
        "area", "pixels", "width", "height", "x_min", "x_max", "y_min", "y_max",
        "bbox", "region", "mask_", "damage_", "prompt_variant_order",
        "num_inference_steps", "guidance_scale", "strength", "threshold",
        "row", "count", "case_id",
    ]

    metric_tokens = [
        "ssim", "psnr", "mae", "mse", "rmse", "lpips",
        "clip", "dino", "similarity", "distance", "improvement",
    ]

    return any(token in lowered for token in metric_tokens) and not any(token in lowered for token in blocked_tokens)

def batch3_numeric_metric_columns(df: pd.DataFrame) -> list[str]:
    numeric_columns = []
    for column in df.columns:
        if not batch3_is_real_metric_column(column):
            continue
        values = pd.to_numeric(df[column], errors="coerce")
        if int(values.notna().sum()) > 0:
            numeric_columns.append(column)
    return sorted(set(numeric_columns))

def batch3_higher_is_better(column: str) -> bool:
    lowered = column.lower()

    if any(token in lowered for token in ["improvement", "gain", "delta_positive", "better"]):
        return True

    if any(token in lowered for token in ["ssim", "psnr", "clip", "dino", "similarity", "cosine"]):
        return True

    if any(token in lowered for token in ["lpips", "mae", "rmse", "mse", "distance", "error"]):
        return False

    return True

def batch3_status_columns(df: pd.DataFrame) -> list[str]:
    columns = []
    for column in df.columns:
        lowered = column.lower()
        if any(token in lowered for token in ["error_map", "difference", "heatmap", "overlay", "path", "image", "figure"]):
            continue
        if (
            lowered.endswith("status")
            or "__status" in lowered
            or "run_status" in lowered
            or "candidate_status" in lowered
            or "failure_reason" in lowered
            or "error_message" in lowered
            or lowered.endswith("error")
            or lowered.endswith("failure")
        ):
            columns.append(column)
    return columns

def batch3_candidate_status(row: pd.Series, status_columns: list[str]) -> str:
    values = " | ".join(batch3_clean_text(row[column]).lower() for column in status_columns if column in row.index)

    if any(token in values for token in ["success", "completed", "done", "ok", "passed"]):
        return "successful"

    if any(token in values for token in ["fail", "exception", "timeout", "missing", "crash"]):
        return "failed_or_warning"

    if "error" in values and "error_map" not in values:
        return "failed_or_warning"

    return "unknown"

def batch3_rank_series(values: pd.Series, higher_is_better: bool) -> pd.Series:
    numeric_values = pd.to_numeric(values, errors="coerce")
    if int(numeric_values.notna().sum()) < 2:
        return pd.Series(np.nan, index=values.index)
    ranks = numeric_values.rank(pct=True, ascending=True)
    return ranks if higher_is_better else 1.0 - ranks

def batch3_visual_type_from_text(column: str, value: Any) -> str:
    column_text = batch3_clean_text(column).lower()
    value_text = batch3_clean_text(value).lower()
    text = f"{column_text} {value_text}"

    is_actual_restored_path = (
        column_text in {"restored_path", "restored_image_path", "difference_map__restored_path"}
        or "data/processed/restored/stable_diffusion" in value_text
    )

    is_visual_diagnostic = any(
        token in text
        for token in [
            "error_map",
            "signed_improvement",
            "improvement_map",
            "heatmap",
            "overlay",
            "difference_map__damaged_error",
            "difference_map__restored_error",
        ]
    )

    if is_visual_diagnostic:
        return "difference_map"

    if is_actual_restored_path:
        return "restored_candidate"

    if column_text in {"damaged_path", "difference_map__damaged_path"}:
        return "damaged_input"

    if column_text in {"clean_path", "difference_map__clean_path"}:
        return "clean_reference"

    if column_text in {"mask_path", "difference_map__mask_path"}:
        return "other_visual"

    if any(token in text for token in ["difference", "diff"]):
        return "difference_map"

    return "other_visual"

def batch3_has_visual_type(row: pd.Series, columns: list[str], visual_type: str) -> bool:
    for column in columns:
        value = row.get(column, "")
        if batch3_clean_text(value) and batch3_visual_type_from_text(column, value) == visual_type:
            return True
    return False

def batch3_first_existing(row: pd.Series, columns: list[str]) -> Any:
    for column in columns:
        if column in row.index:
            value = row[column]
            if batch3_clean_text(value):
                return value
    return np.nan

print(f"Batch 3 summary path: {rel(BATCH3_SUMMARY_PATH)}")
print(f"Batch 3 representative candidates path: {rel(BATCH3_REPRESENTATIVE_PATH)}")

Batch 3 summary path: outputs/26_stable_diffusion_report_generation/analysis/stable_diffusion_report_summary.csv
Batch 3 representative candidates path: outputs/26_stable_diffusion_report_generation/analysis/stable_diffusion_report_representative_candidates.csv


In [18]:
# Batch 3 / Cell 2 - Load Batch 2 audit and detect evidence columns
if "batch2_report_audit_df" in globals() and isinstance(batch2_report_audit_df, pd.DataFrame) and not batch2_report_audit_df.empty:
    batch3_report_audit_df = batch2_report_audit_df.copy()
else:
    batch3_report_audit_df = pd.read_csv(BATCH2_REPORT_AUDIT_PATH)

if batch3_report_audit_df.empty:
    raise RuntimeError("Batch 3 requires a non-empty Stable Diffusion report audit from Batch 2.")

candidate_id_column = "report_candidate_id" if "report_candidate_id" in batch3_report_audit_df.columns else batch3_report_audit_df.columns[0]

identity_columns = [
    column for column in [
        "report_candidate_id", "report_candidate_key", "case_id", "canonical_case_id",
        "source_case_id", "dataset", "mask_id", "mask_type", "damage_profile",
        "prompt_variant", "seed",
    ]
    if column in batch3_report_audit_df.columns
]

prompt_columns = batch3_columns_matching(batch3_report_audit_df, ["prompt"])
seed_columns = batch3_columns_matching(batch3_report_audit_df, ["seed"])
runtime_columns = batch3_columns_matching(batch3_report_audit_df, ["runtime", "seconds", "duration", "elapsed"])
memory_columns = batch3_columns_matching(batch3_report_audit_df, ["memory", "vram", "gpu_memory", "allocated", "reserved"])

status_columns = batch3_status_columns(batch3_report_audit_df)

path_columns = batch3_columns_matching(
    batch3_report_audit_df,
    ["path", "file", "uri", "url", "panel", "figure", "image", "heatmap", "overlay", "difference", "diff"],
)

difference_columns = [
    column for column in path_columns
    if batch3_visual_type_from_text(column, column) == "difference_map"
]

restored_columns = [
    column for column in path_columns
    if batch3_visual_type_from_text(column, column) == "restored_candidate"
]

metric_columns = batch3_numeric_metric_columns(batch3_report_audit_df)

batch3_report_audit_df["batch3_candidate_status"] = batch3_report_audit_df.apply(
    lambda row: batch3_candidate_status(row, status_columns),
    axis=1,
)

batch3_report_audit_df["batch3_has_restored_candidate"] = batch3_report_audit_df.apply(
    lambda row: batch3_has_visual_type(row, path_columns, "restored_candidate"),
    axis=1,
)

batch3_report_audit_df["batch3_has_difference_map"] = batch3_report_audit_df.apply(
    lambda row: batch3_has_visual_type(row, path_columns, "difference_map"),
    axis=1,
)

print(f"Batch 2 audit rows loaded: {len(batch3_report_audit_df):,}")
print(f"Metric columns detected: {len(metric_columns):,}")
print(f"Restored-image columns detected: {len(restored_columns):,}")
print(f"Difference-map columns detected: {len(difference_columns):,}")
print(f"Strict status columns detected: {len(status_columns):,}")

display(
    pd.DataFrame(
        [
            {"column_group": "status_strict", "column_count": len(status_columns), "columns": status_columns},
            {"column_group": "metrics_strict", "column_count": len(metric_columns), "columns": metric_columns[:30]},
            {"column_group": "restored_paths", "column_count": len(restored_columns), "columns": restored_columns[:20]},
            {"column_group": "difference_maps", "column_count": len(difference_columns), "columns": difference_columns[:20]},
            {"column_group": "paths", "column_count": len(path_columns), "columns": path_columns[:20]},
        ]
    )
)

display(
    batch3_report_audit_df["batch3_candidate_status"]
    .value_counts(dropna=False)
    .rename_axis("batch3_candidate_status")
    .reset_index(name="candidate_count")
)

Batch 2 audit rows loaded: 945
Metric columns detected: 24
Restored-image columns detected: 2
Difference-map columns detected: 60
Strict status columns detected: 4


,column_group,column_count,columns
0,status_strict,4,"[status, source_row_status, absolute_pixel_err..."
1,metrics_strict,24,"[classical__damaged_mae, classical__damaged_ms..."
2,restored_paths,2,"[restored_path, difference_map__restored_path]"
3,difference_maps,60,"[difference_map__damaged_error_map_path, diffe..."
4,paths,97,"[difference_map__damaged_error_map_path, diffe..."


,batch3_candidate_status,candidate_count
0,successful,945


In [19]:
# Batch 3 / Cell 3 - Build compact summary
summary_rows = []

def batch3_add_summary(section: str, item: str, value: Any, note: str = ""):
    summary_rows.append(
        {
            "section": section,
            "item": item,
            "value": value,
            "note": note,
        }
    )

batch3_add_summary("audit_shape", "candidate_rows", int(len(batch3_report_audit_df)))
batch3_add_summary("audit_shape", "column_count", int(len(batch3_report_audit_df.columns)))
batch3_add_summary("evidence", "metric_columns", int(len(metric_columns)))
batch3_add_summary("evidence", "difference_map_columns", int(len(difference_columns)))
batch3_add_summary("evidence", "prompt_columns", int(len(prompt_columns)))
batch3_add_summary("evidence", "seed_columns", int(len(seed_columns)))
batch3_add_summary("evidence", "runtime_columns", int(len(runtime_columns)))
batch3_add_summary("evidence", "memory_columns", int(len(memory_columns)))

status_counts = (
    batch3_report_audit_df["batch3_candidate_status"]
    .value_counts(dropna=False)
    .reset_index()
)
status_counts.columns = ["status", "candidate_count"]

for _, row in status_counts.iterrows():
    batch3_add_summary("candidate_status", row["status"], int(row["candidate_count"]))

for column in runtime_columns[:5]:
    values = pd.to_numeric(batch3_report_audit_df[column], errors="coerce")
    if int(values.notna().sum()) > 0:
        batch3_add_summary("runtime", f"{column}__non_null", int(values.notna().sum()))
        batch3_add_summary("runtime", f"{column}__median", float(values.median()))
        batch3_add_summary("runtime", f"{column}__p90", float(values.quantile(0.90)))

for column in memory_columns[:5]:
    values = pd.to_numeric(batch3_report_audit_df[column], errors="coerce")
    if int(values.notna().sum()) > 0:
        batch3_add_summary("memory", f"{column}__non_null", int(values.notna().sum()))
        batch3_add_summary("memory", f"{column}__median", float(values.median()))
        batch3_add_summary("memory", f"{column}__p90", float(values.quantile(0.90)))

for column in metric_columns:
    values = pd.to_numeric(batch3_report_audit_df[column], errors="coerce")
    non_null_count = int(values.notna().sum())

    if non_null_count == 0:
        continue

    batch3_add_summary("metric", f"{column}__non_null", non_null_count)
    batch3_add_summary("metric", f"{column}__median", float(values.median()))
    batch3_add_summary("metric", f"{column}__best_observed", float(values.max() if batch3_higher_is_better(column) else values.min()))
    batch3_add_summary(
        "metric_direction",
        column,
        "higher_is_better" if batch3_higher_is_better(column) else "lower_is_better",
    )

batch3_summary_df = pd.DataFrame(summary_rows)

print(f"Compact summary rows: {len(batch3_summary_df):,}")
display(batch3_summary_df.head(30))

Compact summary rows: 132


,section,item,value,note
0,audit_shape,candidate_rows,945,
1,audit_shape,column_count,417,
2,evidence,metric_columns,24,
3,evidence,difference_map_columns,60,
4,evidence,prompt_columns,18,
5,evidence,seed_columns,15,
6,evidence,runtime_columns,6,
7,evidence,memory_columns,6,
8,candidate_status,successful,945,
9,runtime,full_generation_duration_seconds__non_null,143,


In [20]:
# Batch 3 / Cell 4 - Build representative candidate policy
working_df = batch3_report_audit_df.copy()

for column in metric_columns:
    working_df[f"__rank__{column}"] = batch3_rank_series(
        working_df[column],
        higher_is_better=batch3_higher_is_better(column),
    )

rank_columns = [column for column in working_df.columns if column.startswith("__rank__")]

working_df["batch3_metric_rank_mean"] = working_df[rank_columns].mean(axis=1, skipna=True) if rank_columns else np.nan
working_df["batch3_metric_rank_count"] = working_df[rank_columns].notna().sum(axis=1) if rank_columns else 0

runtime_numeric_columns = []
for column in runtime_columns:
    values = pd.to_numeric(working_df[column], errors="coerce")
    if int(values.notna().sum()) > 0:
        numeric_column = f"__runtime_numeric__{batch3_clean_label(column)}"
        working_df[numeric_column] = values
        runtime_numeric_columns.append(numeric_column)

memory_numeric_columns = []
for column in memory_columns:
    values = pd.to_numeric(working_df[column], errors="coerce")
    if int(values.notna().sum()) > 0:
        numeric_column = f"__memory_numeric__{batch3_clean_label(column)}"
        working_df[numeric_column] = values
        memory_numeric_columns.append(numeric_column)

working_df["batch3_runtime_reference"] = working_df[runtime_numeric_columns].mean(axis=1, skipna=True) if runtime_numeric_columns else np.nan
working_df["batch3_memory_reference"] = working_df[memory_numeric_columns].mean(axis=1, skipna=True) if memory_numeric_columns else np.nan

representative_specs = [
    ("overall_high_metric_evidence", "Highest mean rank across valid restoration metrics.", lambda df: df["batch3_metric_rank_count"] > 0, ["batch3_metric_rank_mean", "batch3_metric_rank_count"], [False, False], None),
    ("overall_low_metric_evidence", "Lowest mean rank across valid restoration metrics.", lambda df: df["batch3_metric_rank_count"] > 0, ["batch3_metric_rank_mean", "batch3_metric_rank_count"], [True, False], None),
    ("median_metric_evidence", "Closest to median mean rank across valid metrics.", lambda df: df["batch3_metric_rank_count"] > 0, [], [], "median_rank"),
    ("fast_successful_candidate", "Lowest runtime among successful or unknown candidates.", lambda df: df["batch3_candidate_status"].isin(["successful", "unknown"]) & df["batch3_runtime_reference"].notna(), ["batch3_runtime_reference"], [True], None),
    ("slow_or_expensive_candidate", "Highest runtime among candidates with runtime evidence.", lambda df: df["batch3_runtime_reference"].notna(), ["batch3_runtime_reference"], [False], None),
    ("high_memory_candidate", "Highest memory reference among candidates with memory evidence.", lambda df: df["batch3_memory_reference"].notna(), ["batch3_memory_reference"], [False], None),
    ("difference_map_available_candidate", "Candidate with difference-map evidence available.", lambda df: df["batch3_has_difference_map"], ["batch3_metric_rank_mean"], [False], None),
    ("failed_or_warning_candidate", "Failed/warning candidate retained only if a restored visual exists.", lambda df: df["batch3_candidate_status"].eq("failed_or_warning"), ["batch3_metric_rank_count"], [False], None),
]

representative_rows = []
used_candidate_ids = set()

for role, description, filter_fn, sort_columns, ascending, custom_select in representative_specs:
    candidate_pool = working_df.loc[filter_fn(working_df) & working_df["batch3_has_restored_candidate"]].copy()

    if candidate_pool.empty:
        representative_rows.append(
            {
                "representative_role": role,
                "selection_status": "not_available",
                "selection_reason": description,
                "report_candidate_id": "",
                "candidate_status": "",
                "metric_rank_mean": np.nan,
                "metric_rank_count": 0,
                "runtime_reference": np.nan,
                "memory_reference": np.nan,
                "has_restored_candidate": False,
                "has_difference_map": False,
            }
        )
        continue

    if custom_select == "median_rank":
        median_rank = candidate_pool["batch3_metric_rank_mean"].median()
        candidate_pool["__median_distance"] = (candidate_pool["batch3_metric_rank_mean"] - median_rank).abs()
        selected_row = candidate_pool.sort_values(
            ["__median_distance", "batch3_metric_rank_count", candidate_id_column],
            ascending=[True, False, True],
            kind="stable",
        ).iloc[0]
    else:
        selected_row = candidate_pool.sort_values(
            sort_columns + [candidate_id_column],
            ascending=ascending + [True],
            kind="stable",
        ).iloc[0]

    selected_candidate_id = batch3_clean_text(selected_row[candidate_id_column])
    duplicate_policy_note = "unique_selection" if selected_candidate_id not in used_candidate_ids else "reused_candidate_because_it_best_matches_this_role"
    used_candidate_ids.add(selected_candidate_id)

    representative_row = {
        "representative_role": role,
        "selection_status": "selected",
        "selection_reason": description,
        "duplicate_policy_note": duplicate_policy_note,
        "report_candidate_id": selected_candidate_id,
        "candidate_status": selected_row["batch3_candidate_status"],
        "metric_rank_mean": selected_row["batch3_metric_rank_mean"],
        "metric_rank_count": int(selected_row["batch3_metric_rank_count"]),
        "runtime_reference": selected_row["batch3_runtime_reference"],
        "memory_reference": selected_row["batch3_memory_reference"],
        "has_restored_candidate": bool(selected_row["batch3_has_restored_candidate"]),
        "has_difference_map": bool(selected_row["batch3_has_difference_map"]),
    }

    for column in identity_columns:
        representative_row[column] = selected_row.get(column, np.nan)

    representative_row["prompt_preview"] = batch3_first_existing(selected_row, prompt_columns)
    representative_row["primary_path"] = batch3_first_existing(selected_row, restored_columns or path_columns)
    representative_row["primary_restored_candidate_path"] = batch3_first_existing(selected_row, restored_columns)
    representative_row["primary_difference_map_path"] = batch3_first_existing(selected_row, difference_columns)

    representative_rows.append(representative_row)

batch3_representative_candidates_df = pd.DataFrame(representative_rows)

print(f"Representative policy rows: {len(batch3_representative_candidates_df):,}")
display(batch3_representative_candidates_df)

Representative policy rows: 8


,representative_role,selection_status,selection_reason,duplicate_policy_note,report_candidate_id,candidate_status,metric_rank_mean,metric_rank_count,runtime_reference,memory_reference,...,has_difference_map,report_candidate_key,case_id,source_case_id,mask_id,mask_type,prompt_preview,primary_path,primary_restored_candidate_path,primary_difference_map_path
0,overall_high_metric_evidence,selected,Highest mean rank across valid restoration met...,unique_selection,sd_candidate_189f26cdeed04200,successful,0.615741,24,3.427751,2.753668e+09,...,True,sd__can__p005__mixed_damage__p00_generic__s202...,p005_mixed_damage,p005_mixed_damage,p005_mixed_damage,mixed_damage,False,data/processed/restored/stable_diffusion/canon...,data/processed/restored/stable_diffusion/canon...,outputs/23_stable_diffusion_difference_maps/ma...
1,overall_low_metric_evidence,selected,Lowest mean rank across valid restoration metr...,unique_selection,sd_candidate_2a8339724e8533c4,successful,0.339175,24,3.364456,2.753668e+09,...,True,sd__syn__p001__partial_transparency__p03_artis...,p001__partial_transparency__severe,p001__partial_transparency__severe,NaN,partial_transparency,True,data/processed/restored/stable_diffusion/synth...,data/processed/restored/stable_diffusion/synth...,outputs/23_stable_diffusion_difference_maps/ma...
2,median_metric_evidence,selected,Closest to median mean rank across valid metrics.,unique_selection,sd_candidate_e0253d35510ef0c0,successful,0.512235,24,3.240105,2.753668e+09,...,True,sd__syn__p018__water_stain__p03_artist_style_p...,p018__water_stain__severe,p018__water_stain__severe,NaN,water_stain,True,data/processed/restored/stable_diffusion/synth...,data/processed/restored/stable_diffusion/synth...,outputs/23_stable_diffusion_difference_maps/ma...
3,fast_successful_candidate,selected,Lowest runtime among successful or unknown can...,unique_selection,sd_candidate_72d917e63e54e65c,successful,0.539947,24,0.092364,2.753668e+09,...,True,sd__can__p014__zero__p04_full_context__s2026__...,p014_zero_control,p014_zero_control,p014_zero_control,zero_control,True,data/processed/restored/stable_diffusion/canon...,data/processed/restored/stable_diffusion/canon...,outputs/23_stable_diffusion_difference_maps/ma...
4,slow_or_expensive_candidate,selected,Highest runtime among candidates with runtime ...,unique_selection,sd_candidate_a38d731e796e4a49,successful,0.427271,24,51.396320,2.753668e+09,...,True,sd__mrob__p001__scratch_thin__p00_generic__s20...,p001__scratch_thin__variant_05,p001__scratch_thin__variant_05,NaN,scratch_thin,True,data/processed/restored/stable_diffusion/mask_...,data/processed/restored/stable_diffusion/mask_...,outputs/23_stable_diffusion_difference_maps/ma...
5,high_memory_candidate,selected,Highest memory reference among candidates with...,unique_selection,sd_candidate_000c396305a705f7,successful,0.440498,24,3.595277,2.753668e+09,...,True,sd__syn__p026__fading__p00_generic__s2026__278...,p026__fading__severe,p026__fading__severe,NaN,fading,True,data/processed/restored/stable_diffusion/synth...,data/processed/restored/stable_diffusion/synth...,outputs/23_stable_diffusion_difference_maps/ma...
6,difference_map_available_candidate,selected,Candidate with difference-map evidence available.,reused_candidate_because_it_best_matches_this_...,sd_candidate_189f26cdeed04200,successful,0.615741,24,3.427751,2.753668e+09,...,True,sd__can__p005__mixed_damage__p00_generic__s202...,p005_mixed_damage,p005_mixed_damage,p005_mixed_damage,mixed_damage,False,data/processed/restored/stable_diffusion/canon...,data/processed/restored/stable_diffusion/canon...,outputs/23_stable_diffusion_difference_maps/ma...
7,failed_or_warning_candidate,not_available,Failed/warning candidate retained only if a re...,NaN,,,NaN,0,NaN,NaN,...,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [21]:
# Batch 3 / Cell 5 - Write outputs and validate
BATCH3_SUMMARY_PATH.parent.mkdir(parents=True, exist_ok=True)
BATCH3_REPRESENTATIVE_PATH.parent.mkdir(parents=True, exist_ok=True)

batch3_summary_df.to_csv(BATCH3_SUMMARY_PATH, index=False)
batch3_representative_candidates_df.to_csv(BATCH3_REPRESENTATIVE_PATH, index=False)

batch3_summary_shape = csv_shape(BATCH3_SUMMARY_PATH)
batch3_representative_shape = csv_shape(BATCH3_REPRESENTATIVE_PATH)

batch3_validation_rows = [
    validation_row(
        "summary_written",
        rel(BATCH3_SUMMARY_PATH),
        "file exists",
        BATCH3_SUMMARY_PATH.is_file(),
        "Batch 3 summary CSV was not written.",
    ),
    validation_row(
        "representative_candidates_written",
        rel(BATCH3_REPRESENTATIVE_PATH),
        "file exists",
        BATCH3_REPRESENTATIVE_PATH.is_file(),
        "Batch 3 representative candidate CSV was not written.",
    ),
    validation_row(
        "summary_has_rows",
        int(batch3_summary_shape[0] or 0),
        "> 0",
        int(batch3_summary_shape[0] or 0) > 0,
        "Batch 3 summary CSV has no rows.",
    ),
    validation_row(
        "representative_policy_has_rows",
        int(batch3_representative_shape[0] or 0),
        "> 0",
        int(batch3_representative_shape[0] or 0) > 0,
        "Batch 3 representative candidate CSV has no rows.",
    ),
    validation_row(
        "at_least_one_representative_selected",
        int(batch3_representative_candidates_df["selection_status"].eq("selected").sum()),
        "> 0",
        int(batch3_representative_candidates_df["selection_status"].eq("selected").sum()) > 0,
        "No representative candidates were selected.",
    ),
    validation_row(
        "metric_evidence_available_for_policy",
        int(len(metric_columns)),
        "> 0",
        int(len(metric_columns)) > 0,
        "No metric columns were available for representative policy selection.",
    ),
]

batch3_validation_df = pd.DataFrame(batch3_validation_rows)
batch3_passed = bool(bool_series(batch3_validation_df["passed"]).all())

stage_manifest = read_json_if_exists(STAGE_MANIFEST_PATH)
stage_manifest.update(
    {
        "notebook_id": NOTEBOOK_ID,
        "notebook_title": NOTEBOOK_TITLE,
        "stage": "batch3_summary_and_representative_policy",
        "stage_status": "passed" if batch3_passed else "failed",
        "updated_at_utc": utc_now_iso(),
        "batch3": {
            "status": "passed" if batch3_passed else "failed",
            "summary_csv": rel(BATCH3_SUMMARY_PATH),
            "representative_candidates_csv": rel(BATCH3_REPRESENTATIVE_PATH),
            "summary_rows": int(batch3_summary_shape[0] or 0),
            "representative_policy_rows": int(batch3_representative_shape[0] or 0),
            "selected_representatives": int(batch3_representative_candidates_df["selection_status"].eq("selected").sum()),
            "metric_columns_used": int(len(metric_columns)),
            "difference_map_columns_used": int(len(difference_columns)),
        },
    }
)
write_json(STAGE_MANIFEST_PATH, stage_manifest)

print(f"Saved compact summary: {rel(BATCH3_SUMMARY_PATH)}")
print(f"Saved representative candidates: {rel(BATCH3_REPRESENTATIVE_PATH)}")
print(f"Batch 3 checks passed: {int(bool_series(batch3_validation_df['passed']).sum())} / {len(batch3_validation_df)}")

display(batch3_validation_df)

if not batch3_passed:
    display(
        batch3_validation_df.loc[
            ~bool_series(batch3_validation_df["passed"]),
            ["check_name", "actual", "expected", "failure_message"],
        ]
    )
    raise RuntimeError("Batch 3 validation failed. Fix summary or representative-policy generation before running Batch 4.")

display(batch3_summary_df.head(25))
display(batch3_representative_candidates_df)

print("Batch 3 passed. Compact summary and representative candidate policy are ready for report drafting.")

Saved compact summary: outputs/26_stable_diffusion_report_generation/analysis/stable_diffusion_report_summary.csv
Saved representative candidates: outputs/26_stable_diffusion_report_generation/analysis/stable_diffusion_report_representative_candidates.csv
Batch 3 checks passed: 6 / 6


,check_name,severity,actual,expected,passed,failure_message,checked_at_utc
0,summary_written,error,"""outputs/26_stable_diffusion_report_generation...","""file exists""",True,,2026-08-10T11:29:01.386509+00:00
1,representative_candidates_written,error,"""outputs/26_stable_diffusion_report_generation...","""file exists""",True,,2026-08-10T11:29:01.386509+00:00
2,summary_has_rows,error,132,"""> 0""",True,,2026-08-10T11:29:01.386509+00:00
3,representative_policy_has_rows,error,8,"""> 0""",True,,2026-08-10T11:29:01.386509+00:00
4,at_least_one_representative_selected,error,7,"""> 0""",True,,2026-08-10T11:29:01.393085+00:00
5,metric_evidence_available_for_policy,error,24,"""> 0""",True,,2026-08-10T11:29:01.393085+00:00


,section,item,value,note
0,audit_shape,candidate_rows,945,
1,audit_shape,column_count,417,
2,evidence,metric_columns,24,
3,evidence,difference_map_columns,60,
4,evidence,prompt_columns,18,
5,evidence,seed_columns,15,
6,evidence,runtime_columns,6,
7,evidence,memory_columns,6,
8,candidate_status,successful,945,
9,runtime,full_generation_duration_seconds__non_null,143,


,representative_role,selection_status,selection_reason,duplicate_policy_note,report_candidate_id,candidate_status,metric_rank_mean,metric_rank_count,runtime_reference,memory_reference,...,has_difference_map,report_candidate_key,case_id,source_case_id,mask_id,mask_type,prompt_preview,primary_path,primary_restored_candidate_path,primary_difference_map_path
0,overall_high_metric_evidence,selected,Highest mean rank across valid restoration met...,unique_selection,sd_candidate_189f26cdeed04200,successful,0.615741,24,3.427751,2.753668e+09,...,True,sd__can__p005__mixed_damage__p00_generic__s202...,p005_mixed_damage,p005_mixed_damage,p005_mixed_damage,mixed_damage,False,data/processed/restored/stable_diffusion/canon...,data/processed/restored/stable_diffusion/canon...,outputs/23_stable_diffusion_difference_maps/ma...
1,overall_low_metric_evidence,selected,Lowest mean rank across valid restoration metr...,unique_selection,sd_candidate_2a8339724e8533c4,successful,0.339175,24,3.364456,2.753668e+09,...,True,sd__syn__p001__partial_transparency__p03_artis...,p001__partial_transparency__severe,p001__partial_transparency__severe,NaN,partial_transparency,True,data/processed/restored/stable_diffusion/synth...,data/processed/restored/stable_diffusion/synth...,outputs/23_stable_diffusion_difference_maps/ma...
2,median_metric_evidence,selected,Closest to median mean rank across valid metrics.,unique_selection,sd_candidate_e0253d35510ef0c0,successful,0.512235,24,3.240105,2.753668e+09,...,True,sd__syn__p018__water_stain__p03_artist_style_p...,p018__water_stain__severe,p018__water_stain__severe,NaN,water_stain,True,data/processed/restored/stable_diffusion/synth...,data/processed/restored/stable_diffusion/synth...,outputs/23_stable_diffusion_difference_maps/ma...
3,fast_successful_candidate,selected,Lowest runtime among successful or unknown can...,unique_selection,sd_candidate_72d917e63e54e65c,successful,0.539947,24,0.092364,2.753668e+09,...,True,sd__can__p014__zero__p04_full_context__s2026__...,p014_zero_control,p014_zero_control,p014_zero_control,zero_control,True,data/processed/restored/stable_diffusion/canon...,data/processed/restored/stable_diffusion/canon...,outputs/23_stable_diffusion_difference_maps/ma...
4,slow_or_expensive_candidate,selected,Highest runtime among candidates with runtime ...,unique_selection,sd_candidate_a38d731e796e4a49,successful,0.427271,24,51.396320,2.753668e+09,...,True,sd__mrob__p001__scratch_thin__p00_generic__s20...,p001__scratch_thin__variant_05,p001__scratch_thin__variant_05,NaN,scratch_thin,True,data/processed/restored/stable_diffusion/mask_...,data/processed/restored/stable_diffusion/mask_...,outputs/23_stable_diffusion_difference_maps/ma...
5,high_memory_candidate,selected,Highest memory reference among candidates with...,unique_selection,sd_candidate_000c396305a705f7,successful,0.440498,24,3.595277,2.753668e+09,...,True,sd__syn__p026__fading__p00_generic__s2026__278...,p026__fading__severe,p026__fading__severe,NaN,fading,True,data/processed/restored/stable_diffusion/synth...,data/processed/restored/stable_diffusion/synth...,outputs/23_stable_diffusion_difference_maps/ma...
6,difference_map_available_candidate,selected,Candidate with difference-map evidence available.,reused_candidate_because_it_best_matches_this_...,sd_candidate_189f26cdeed04200,successful,0.615741,24,3.427751,2.753668e+09,...,True,sd__can__p005__mixed_damage__p00_generic__s202...,p005_mixed_damage,p005_mixed_damage,p005_mixed_damage,mixed_damage,False,data/processed/restored/stable_diffusion/canon...,data/processed/restored/stable_diffusion/canon...,outputs/23_stable_diffusion_difference_maps/ma...
7,failed_or_warning_candidate,not_available,Failed/warning candidate retained only if a re...,NaN,,,NaN,0,NaN,NaN,...,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Batch 3 passed. Compact summary and representative candidate policy are ready for report drafting.


In [22]:
# Batch 4 / Cell 1 - Setup and strict asset helpers
from pathlib import Path
from typing import Any
import math
import re
import textwrap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps, ImageDraw, ImageFont

BATCH4_FIGURES_DIR = OUTPUT_DIRS["figures"] / "stable_diffusion_report"
BATCH4_AGGREGATE_DIR = BATCH4_FIGURES_DIR / "aggregate_plots"
BATCH4_PANEL_DIR = BATCH4_FIGURES_DIR / "candidate_panels"
BATCH4_FIGURE_MANIFEST_PATH = OUTPUT_DIRS["figures"] / "stable_diffusion_report_figure_manifest.csv"

for directory in [BATCH4_FIGURES_DIR, BATCH4_AGGREGATE_DIR, BATCH4_PANEL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

BATCH4_IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".webp", ".bmp", ".tif", ".tiff"}
BATCH4_MIN_PNG_BYTES = 2_000
BATCH4_MIN_WIDTH = 640
BATCH4_MIN_HEIGHT = 360

try:
    BATCH4_RESAMPLE = Image.Resampling.LANCZOS
except AttributeError:
    BATCH4_RESAMPLE = Image.LANCZOS

batch4_manifest_rows = []

def batch4_make_unique_columns(columns: list[str]) -> list[str]:
    counts = {}
    unique_columns = []

    for column in columns:
        column = str(column)
        count = counts.get(column, 0)
        unique_columns.append(column if count == 0 else f"{column}__duplicate_{count}")
        counts[column] = count + 1

    return unique_columns

def batch4_clean_text(value: Any) -> str:
    if value is None:
        return ""

    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass

    return str(value).strip()

def batch4_slug(value: Any, fallback: str = "asset") -> str:
    text_value = batch4_clean_text(value).lower()
    text_value = re.sub(r"[^a-z0-9]+", "_", text_value)
    text_value = re.sub(r"_+", "_", text_value).strip("_")
    return (text_value or fallback)[:120]

def batch4_columns_matching(df: pd.DataFrame, patterns: list[str], exclude: list[str] | None = None) -> list[str]:
    exclude = exclude or []
    matched_columns = []

    for column in df.columns:
        lowered = column.lower()
        if any(pattern in lowered for pattern in patterns) and not any(pattern in lowered for pattern in exclude):
            matched_columns.append(column)

    return matched_columns

def batch4_higher_is_better(column: str) -> bool:
    lowered = column.lower()

    if any(token in lowered for token in ["lpips", "mae", "rmse", "mse", "distance", "error"]):
        return False

    if any(token in lowered for token in ["improvement", "ssim", "psnr", "clip", "dino", "similarity", "score"]):
        return True

    return True

def batch4_rank_series(values: pd.Series, higher_is_better: bool) -> pd.Series:
    numeric_values = pd.to_numeric(values, errors="coerce")

    if int(numeric_values.notna().sum()) < 2:
        return pd.Series(np.nan, index=values.index)

    ranks = numeric_values.rank(pct=True, ascending=True)

    return ranks if higher_is_better else 1.0 - ranks

def batch4_metric_family(column: str) -> str:
    lowered = column.lower()

    if "lpips" in lowered:
        return "lpips"
    if "feature" in lowered or "clip" in lowered or "dino" in lowered:
        return "feature"
    if "classical" in lowered or any(token in lowered for token in ["ssim", "psnr", "mae", "rmse"]):
        return "classical"

    return "other_metric"

def batch4_candidate_base_dirs() -> list[Path]:
    base_dirs = []

    for name in ["PROJECT_ROOT", "ROOT_DIR", "REPO_ROOT", "BASE_DIR", "OUTPUT_ROOT", "DATA_ROOT"]:
        value = globals().get(name)
        if value is not None:
            try:
                base_dirs.append(Path(value))
            except Exception:
                pass

    base_dirs.extend([Path.cwd(), BATCH4_FIGURES_DIR.parent.parent])

    unique_dirs = []
    seen = set()

    for base_dir in base_dirs:
        try:
            resolved = base_dir.resolve()
        except Exception:
            resolved = base_dir

        if str(resolved) not in seen:
            unique_dirs.append(resolved)
            seen.add(str(resolved))

    return unique_dirs

def batch4_resolve_local_path(value: Any) -> tuple[Path | None, str]:
    text_value = batch4_clean_text(value)

    if not text_value:
        return None, "empty"

    if text_value.startswith(("http://", "https://", "s3://", "gs://")):
        return None, "external_uri_not_local_file"

    path_value = Path(text_value)

    candidates = [path_value] if path_value.is_absolute() else [
        base_dir / path_value for base_dir in batch4_candidate_base_dirs()
    ]

    for candidate in candidates:
        try:
            if candidate.is_file():
                return candidate.resolve(), "found"
        except Exception:
            pass

    return None, "not_found"

def batch4_is_image_like_path(value: Any) -> bool:
    text_value = batch4_clean_text(value)
    if not text_value:
        return False
    return Path(text_value).suffix.lower() in BATCH4_IMAGE_EXTENSIONS

def batch4_load_image(path: Path, max_side: int = 900) -> Image.Image:
    with Image.open(path) as image:
        image = ImageOps.exif_transpose(image).convert("RGB")
        image.thumbnail((max_side, max_side), BATCH4_RESAMPLE)
        return image.copy()

def batch4_validate_png(path: Path) -> dict:
    record = {
        "file_exists": path.is_file(),
        "is_png": path.suffix.lower() == ".png",
        "file_size_bytes": 0,
        "width_px": 0,
        "height_px": 0,
        "png_readable": False,
        "html_ready": False,
        "validation_message": "",
    }

    if not record["file_exists"]:
        record["validation_message"] = "file does not exist"
        return record

    record["file_size_bytes"] = int(path.stat().st_size)

    try:
        with Image.open(path) as image:
            image.verify()

        with Image.open(path) as image:
            record["width_px"], record["height_px"] = image.size

        record["png_readable"] = True
    except Exception as exc:
        record["validation_message"] = f"png unreadable: {exc}"
        return record

    record["html_ready"] = bool(
        record["is_png"]
        and record["png_readable"]
        and record["file_size_bytes"] >= BATCH4_MIN_PNG_BYTES
        and record["width_px"] >= BATCH4_MIN_WIDTH
        and record["height_px"] >= BATCH4_MIN_HEIGHT
    )

    if not record["html_ready"]:
        record["validation_message"] = "png exists but failed size/readiness thresholds"

    return record

def batch4_rel_path(path: Path) -> str:
    try:
        return rel(path)
    except Exception:
        return str(path)

def batch4_add_manifest_row(
    figure_id: str,
    figure_group: str,
    title: str,
    caption: str,
    path: Path,
    source_table: str,
    source_rows: int,
    candidate_rows: int,
    representative_role: str = "",
    report_candidate_id: str = "",
    evidence_columns: list[str] | None = None,
    notes: str = "",
):
    validation = batch4_validate_png(path)

    batch4_manifest_rows.append(
        {
            "figure_id": figure_id,
            "figure_group": figure_group,
            "title": title,
            "caption": caption,
            "file_path": str(path),
            "relative_path": batch4_rel_path(path),
            "source_table": source_table,
            "source_rows": int(source_rows),
            "candidate_rows": int(candidate_rows),
            "representative_role": representative_role,
            "report_candidate_id": report_candidate_id,
            "evidence_columns": ";".join(evidence_columns or []),
            "notes": notes,
            **validation,
        }
    )

def batch4_save_matplotlib_figure(
    fig,
    path: Path,
    figure_id: str,
    figure_group: str,
    title: str,
    caption: str,
    source_table: str,
    source_rows: int,
    candidate_rows: int,
    evidence_columns: list[str] | None = None,
    notes: str = "",
):
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=180, bbox_inches="tight", facecolor="white")
    plt.close(fig)

    batch4_add_manifest_row(
        figure_id=figure_id,
        figure_group=figure_group,
        title=title,
        caption=caption,
        path=path,
        source_table=source_table,
        source_rows=source_rows,
        candidate_rows=candidate_rows,
        evidence_columns=evidence_columns,
        notes=notes,
    )

print(f"Batch 4 figure manifest path: {rel(BATCH4_FIGURE_MANIFEST_PATH)}")
print(f"Batch 4 aggregate plot directory: {rel(BATCH4_AGGREGATE_DIR)}")
print(f"Batch 4 candidate panel directory: {rel(BATCH4_PANEL_DIR)}")

Batch 4 figure manifest path: outputs/26_stable_diffusion_report_generation/figures/stable_diffusion_report_figure_manifest.csv
Batch 4 aggregate plot directory: outputs/26_stable_diffusion_report_generation/figures/stable_diffusion_report/aggregate_plots
Batch 4 candidate panel directory: outputs/26_stable_diffusion_report_generation/figures/stable_diffusion_report/candidate_panels


In [23]:
# Batch 4 / Cell 2 - Load Batch 3 evidence and recompute report scores
if "batch3_report_audit_df" in globals() and isinstance(batch3_report_audit_df, pd.DataFrame) and not batch3_report_audit_df.empty:
    batch4_report_audit_df = batch3_report_audit_df.copy()
else:
    batch4_report_audit_df = pd.read_csv(BATCH2_REPORT_AUDIT_PATH)

if "batch3_representative_candidates_df" in globals() and isinstance(batch3_representative_candidates_df, pd.DataFrame) and not batch3_representative_candidates_df.empty:
    batch4_representative_candidates_df = batch3_representative_candidates_df.copy()
else:
    batch4_representative_candidates_df = pd.read_csv(BATCH3_REPRESENTATIVE_PATH)

batch4_report_audit_df.columns = batch4_make_unique_columns(list(batch4_report_audit_df.columns))
batch4_representative_candidates_df.columns = batch4_make_unique_columns(list(batch4_representative_candidates_df.columns))

candidate_id_column = "report_candidate_id"

if candidate_id_column not in batch4_report_audit_df.columns:
    raise RuntimeError("Batch 4 requires report_candidate_id in the report audit.")

metric_columns = batch3_numeric_metric_columns(batch4_report_audit_df) if "batch3_numeric_metric_columns" in globals() else [
    column for column in batch4_report_audit_df.columns
    if int(pd.to_numeric(batch4_report_audit_df[column], errors="coerce").notna().sum()) > 0
]

path_columns = batch4_columns_matching(
    batch4_report_audit_df,
    ["path", "file", "uri", "url", "panel", "figure", "image", "heatmap", "overlay", "difference", "diff"],
)

difference_columns = [
    column for column in path_columns
    if "batch3_visual_type_from_text" in globals() and batch3_visual_type_from_text(column, column) == "difference_map"
]

runtime_columns = batch4_columns_matching(batch4_report_audit_df, ["runtime", "seconds", "duration", "elapsed"])
memory_columns = batch4_columns_matching(batch4_report_audit_df, ["memory", "vram", "gpu_memory", "allocated", "reserved"])

if "batch3_candidate_status" not in batch4_report_audit_df.columns:
    status_columns = batch3_status_columns(batch4_report_audit_df)
    batch4_report_audit_df["batch3_candidate_status"] = batch4_report_audit_df.apply(
        lambda row: batch3_candidate_status(row, status_columns),
        axis=1,
    )

for column in metric_columns:
    higher = batch3_higher_is_better(column) if "batch3_higher_is_better" in globals() else batch4_higher_is_better(column)
    batch4_report_audit_df[f"batch4_rank__{batch4_slug(column)}"] = batch4_rank_series(
        batch4_report_audit_df[column],
        higher_is_better=higher,
    )

rank_columns = [column for column in batch4_report_audit_df.columns if column.startswith("batch4_rank__")]

batch4_report_audit_df["batch4_metric_rank_mean"] = batch4_report_audit_df[rank_columns].mean(axis=1, skipna=True) if rank_columns else np.nan
batch4_report_audit_df["batch4_metric_rank_count"] = batch4_report_audit_df[rank_columns].notna().sum(axis=1) if rank_columns else 0

runtime_numeric_columns = []
for column in runtime_columns:
    values = pd.to_numeric(batch4_report_audit_df[column], errors="coerce")
    if int(values.notna().sum()) > 0:
        runtime_numeric_column = f"batch4_runtime_numeric__{batch4_slug(column)}"
        batch4_report_audit_df[runtime_numeric_column] = values
        runtime_numeric_columns.append(runtime_numeric_column)

memory_numeric_columns = []
for column in memory_columns:
    values = pd.to_numeric(batch4_report_audit_df[column], errors="coerce")
    if int(values.notna().sum()) > 0:
        memory_numeric_column = f"batch4_memory_numeric__{batch4_slug(column)}"
        batch4_report_audit_df[memory_numeric_column] = values
        memory_numeric_columns.append(memory_numeric_column)

batch4_report_audit_df["batch4_runtime_reference"] = batch4_report_audit_df[runtime_numeric_columns].mean(axis=1, skipna=True) if runtime_numeric_columns else np.nan
batch4_report_audit_df["batch4_memory_reference"] = batch4_report_audit_df[memory_numeric_columns].mean(axis=1, skipna=True) if memory_numeric_columns else np.nan

selected_representatives_df = batch4_representative_candidates_df.loc[
    batch4_representative_candidates_df["selection_status"].eq("selected")
].copy()

if selected_representatives_df.empty:
    raise RuntimeError("Batch 4 strict check failed: no selected representative candidates are available.")

if not metric_columns:
    raise RuntimeError("Batch 4 strict check failed: no strict numeric metric columns are available.")

print(f"Report audit rows: {len(batch4_report_audit_df):,}")
print(f"Selected representative candidates: {len(selected_representatives_df):,}")
print(f"Strict metric columns: {len(metric_columns):,}")
print(f"Path columns: {len(path_columns):,}")

display(
    pd.DataFrame(
        [
            {"evidence_group": "metrics_strict", "count": len(metric_columns), "sample_columns": metric_columns[:20]},
            {"evidence_group": "difference_columns", "count": len(difference_columns), "sample_columns": difference_columns[:10]},
            {"evidence_group": "path_columns", "count": len(path_columns), "sample_columns": path_columns[:10]},
        ]
    )
)

Report audit rows: 945
Selected representative candidates: 7
Strict metric columns: 24
Path columns: 98


,evidence_group,count,sample_columns
0,metrics_strict,24,"[classical__damaged_mae, classical__damaged_ms..."
1,difference_columns,61,"[difference_map__damaged_error_map_path, diffe..."
2,path_columns,98,"[difference_map__damaged_error_map_path, diffe..."


In [24]:
# Batch 4 / Cell 3 - Aggregate plots
plt.style.use("default")

candidate_count = int(len(batch4_report_audit_df))

# 1) Candidate status distribution
status_counts = (
    batch4_report_audit_df["batch3_candidate_status"]
    .value_counts(dropna=False)
    .rename_axis("candidate_status")
    .reset_index(name="candidate_count")
)

fig, ax = plt.subplots(figsize=(8, 4.8))
ax.bar(status_counts["candidate_status"], status_counts["candidate_count"], color="#3b6ea8")
ax.set_title("Stable Diffusion Candidate Status Distribution")
ax.set_xlabel("Candidate status")
ax.set_ylabel("Candidate count")
ax.tick_params(axis="x", rotation=25)
ax.grid(axis="y", alpha=0.25)

for index, row in status_counts.iterrows():
    ax.text(index, row["candidate_count"], str(int(row["candidate_count"])), ha="center", va="bottom")

batch4_save_matplotlib_figure(
    fig=fig,
    path=BATCH4_AGGREGATE_DIR / "candidate_status_distribution.png",
    figure_id="sd_candidate_status_distribution",
    figure_group="aggregate_plot",
    title="Stable Diffusion Candidate Status Distribution",
    caption="Counts of candidate-level restoration statuses used to identify successful, warning, and unknown cases.",
    source_table="batch4_report_audit_df",
    source_rows=candidate_count,
    candidate_rows=candidate_count,
    evidence_columns=["batch3_candidate_status"],
)

# 2) Metric evidence coverage
metric_coverage_df = pd.DataFrame(
    [
        {
            "metric_column": column,
            "metric_family": batch4_metric_family(column),
            "non_null_count": int(pd.to_numeric(batch4_report_audit_df[column], errors="coerce").notna().sum()),
            "coverage_percent": float(pd.to_numeric(batch4_report_audit_df[column], errors="coerce").notna().mean() * 100.0),
            "direction": "higher_is_better" if batch4_higher_is_better(column) else "lower_is_better",
        }
        for column in metric_columns
    ]
).sort_values(["non_null_count", "metric_column"], ascending=[True, True])

plot_coverage_df = metric_coverage_df.tail(25).copy()

fig, ax = plt.subplots(figsize=(11, max(5, 0.28 * len(plot_coverage_df))))
ax.barh(plot_coverage_df["metric_column"], plot_coverage_df["coverage_percent"], color="#4f8a5b")
ax.set_title("Metric Evidence Coverage")
ax.set_xlabel("Non-null coverage (%)")
ax.set_ylabel("Metric column")
ax.set_xlim(0, 105)
ax.grid(axis="x", alpha=0.25)

batch4_save_matplotlib_figure(
    fig=fig,
    path=BATCH4_AGGREGATE_DIR / "metric_evidence_coverage.png",
    figure_id="sd_metric_evidence_coverage",
    figure_group="aggregate_plot",
    title="Stable Diffusion Metric Evidence Coverage",
    caption="Coverage of numeric metric columns after Batch 2 consolidation. Low-coverage metrics are visible instead of silently dropped.",
    source_table="batch4_report_audit_df",
    source_rows=candidate_count,
    candidate_rows=candidate_count,
    evidence_columns=metric_columns,
)

# 3) Metric family coverage
family_coverage_df = (
    metric_coverage_df
    .groupby(["metric_family", "direction"], as_index=False)
    .agg(metric_columns=("metric_column", "count"), median_coverage_percent=("coverage_percent", "median"))
)

fig, ax = plt.subplots(figsize=(8.5, 4.8))
x = np.arange(len(family_coverage_df))
ax.bar(x, family_coverage_df["metric_columns"], color="#8a5a9e")
ax.set_title("Metric Families Available for Report Analysis")
ax.set_xlabel("Metric family and direction")
ax.set_ylabel("Metric column count")
ax.set_xticks(x)
ax.set_xticklabels(
    family_coverage_df["metric_family"] + "\n" + family_coverage_df["direction"],
    rotation=0,
)
ax.grid(axis="y", alpha=0.25)

for index, row in family_coverage_df.iterrows():
    ax.text(index, row["metric_columns"], str(int(row["metric_columns"])), ha="center", va="bottom")

batch4_save_matplotlib_figure(
    fig=fig,
    path=BATCH4_AGGREGATE_DIR / "metric_family_coverage.png",
    figure_id="sd_metric_family_coverage",
    figure_group="aggregate_plot",
    title="Stable Diffusion Metric Family Coverage",
    caption="Compact overview of classical, LPIPS, and feature-metric evidence available for report discussion.",
    source_table="batch4_report_audit_df",
    source_rows=candidate_count,
    candidate_rows=candidate_count,
    evidence_columns=metric_columns,
)

# 4) Mean metric-rank distribution
rank_values = pd.to_numeric(batch4_report_audit_df["batch4_metric_rank_mean"], errors="coerce").dropna()

fig, ax = plt.subplots(figsize=(8.5, 4.8))
ax.hist(rank_values, bins=20, color="#d18335", edgecolor="white")
ax.axvline(rank_values.median(), color="black", linestyle="--", linewidth=1.5, label=f"median={rank_values.median():.3f}")
ax.set_title("Candidate Mean Metric-Rank Distribution")
ax.set_xlabel("Mean percentile rank across available metrics")
ax.set_ylabel("Candidate count")
ax.legend()
ax.grid(axis="y", alpha=0.25)

batch4_save_matplotlib_figure(
    fig=fig,
    path=BATCH4_AGGREGATE_DIR / "candidate_metric_rank_distribution.png",
    figure_id="sd_candidate_metric_rank_distribution",
    figure_group="aggregate_plot",
    title="Stable Diffusion Candidate Mean Metric-Rank Distribution",
    caption="Distribution of candidate-level mean percentile ranks across available selected-region metrics.",
    source_table="batch4_report_audit_df",
    source_rows=candidate_count,
    candidate_rows=int(rank_values.shape[0]),
    evidence_columns=rank_columns,
)

# 5) Runtime vs metric rank, if available
if runtime_numeric_columns and int(batch4_report_audit_df["batch4_runtime_reference"].notna().sum()) > 0:
    plot_df = batch4_report_audit_df.loc[
        batch4_report_audit_df["batch4_runtime_reference"].notna()
        & batch4_report_audit_df["batch4_metric_rank_mean"].notna()
    ].copy()

    fig, ax = plt.subplots(figsize=(8.5, 5.2))
    ax.scatter(
        plot_df["batch4_runtime_reference"],
        plot_df["batch4_metric_rank_mean"],
        alpha=0.72,
        color="#3b6ea8",
        edgecolor="white",
        linewidth=0.4,
    )
    ax.set_title("Runtime Versus Metric Evidence")
    ax.set_xlabel("Runtime reference")
    ax.set_ylabel("Mean metric percentile rank")
    ax.grid(alpha=0.25)

    batch4_save_matplotlib_figure(
        fig=fig,
        path=BATCH4_AGGREGATE_DIR / "runtime_vs_metric_rank.png",
        figure_id="sd_runtime_vs_metric_rank",
        figure_group="aggregate_plot",
        title="Stable Diffusion Runtime Versus Metric Evidence",
        caption="Candidate runtime reference plotted against mean metric rank to expose expensive candidates and possible tradeoffs.",
        source_table="batch4_report_audit_df",
        source_rows=candidate_count,
        candidate_rows=int(len(plot_df)),
        evidence_columns=runtime_numeric_columns + ["batch4_metric_rank_mean"],
    )

# 6) Representative policy overview
rep_plot_df = selected_representatives_df.copy()

if "metric_rank_mean" in rep_plot_df.columns:
    rep_plot_df["metric_rank_mean_numeric"] = pd.to_numeric(rep_plot_df["metric_rank_mean"], errors="coerce")
else:
    audit_rank_lookup = batch4_report_audit_df.set_index(candidate_id_column)["batch4_metric_rank_mean"].to_dict()
    rep_plot_df["metric_rank_mean_numeric"] = rep_plot_df[candidate_id_column].map(audit_rank_lookup)

fig, ax = plt.subplots(figsize=(10, max(4.8, 0.42 * len(rep_plot_df))))
plot_df = rep_plot_df.sort_values("metric_rank_mean_numeric", ascending=True).copy()
ax.barh(plot_df["representative_role"], plot_df["metric_rank_mean_numeric"], color="#5f6f94")
ax.set_title("Representative Candidate Policy Overview")
ax.set_xlabel("Mean metric percentile rank")
ax.set_ylabel("Representative role")
ax.set_xlim(0, 1.05)
ax.grid(axis="x", alpha=0.25)

batch4_save_matplotlib_figure(
    fig=fig,
    path=BATCH4_AGGREGATE_DIR / "representative_candidate_policy_overview.png",
    figure_id="sd_representative_candidate_policy_overview",
    figure_group="aggregate_plot",
    title="Stable Diffusion Representative Candidate Policy Overview",
    caption="Metric-rank positions of selected representative candidates. Roles are policy selections, not claims of universal restoration quality.",
    source_table="batch4_representative_candidates_df",
    source_rows=int(len(batch4_representative_candidates_df)),
    candidate_rows=int(len(selected_representatives_df)),
    evidence_columns=["representative_role", "metric_rank_mean"],
)

print(f"Aggregate plots created so far: {len(batch4_manifest_rows)}")
display(pd.DataFrame(batch4_manifest_rows))

Aggregate plots created so far: 6


,figure_id,figure_group,title,caption,file_path,relative_path,source_table,source_rows,candidate_rows,representative_role,...,evidence_columns,notes,file_exists,is_png,file_size_bytes,width_px,height_px,png_readable,html_ready,validation_message
0,sd_candidate_status_distribution,aggregate_plot,Stable Diffusion Candidate Status Distribution,Counts of candidate-level restoration statuses...,D:\Masters\FH\Thesis\painting-restoration-eval...,outputs/26_stable_diffusion_report_generation/...,batch4_report_audit_df,945,945,,...,batch3_candidate_status,,True,True,34922,1252,869,True,True,
1,sd_metric_evidence_coverage,aggregate_plot,Stable Diffusion Metric Evidence Coverage,Coverage of numeric metric columns after Batch...,D:\Masters\FH\Thesis\painting-restoration-eval...,outputs/26_stable_diffusion_report_generation/...,batch4_report_audit_df,945,945,,...,classical__damaged_mae;classical__damaged_mse;...,,True,True,188741,2449,1082,True,True,
2,sd_metric_family_coverage,aggregate_plot,Stable Diffusion Metric Family Coverage,"Compact overview of classical, LPIPS, and feat...",D:\Masters\FH\Thesis\painting-restoration-eval...,outputs/26_stable_diffusion_report_generation/...,batch4_report_audit_df,945,945,,...,classical__damaged_mae;classical__damaged_mse;...,,True,True,52612,1290,851,True,True,
3,sd_candidate_metric_rank_distribution,aggregate_plot,Stable Diffusion Candidate Mean Metric-Rank Di...,Distribution of candidate-level mean percentil...,D:\Masters\FH\Thesis\painting-restoration-eval...,outputs/26_stable_diffusion_report_generation/...,batch4_report_audit_df,945,945,,...,batch4_rank__classical_damaged_mae;batch4_rank...,,True,True,50486,1322,816,True,True,
4,sd_runtime_vs_metric_rank,aggregate_plot,Stable Diffusion Runtime Versus Metric Evidence,Candidate runtime reference plotted against me...,D:\Masters\FH\Thesis\painting-restoration-eval...,outputs/26_stable_diffusion_report_generation/...,batch4_report_audit_df,945,945,,...,batch4_runtime_numeric__full_generation_durati...,,True,True,109475,1330,872,True,True,
5,sd_representative_candidate_policy_overview,aggregate_plot,Stable Diffusion Representative Candidate Poli...,Metric-rank positions of selected representati...,D:\Masters\FH\Thesis\painting-restoration-eval...,outputs/26_stable_diffusion_report_generation/...,batch4_representative_candidates_df,8,7,,...,representative_role;metric_rank_mean,,True,True,72809,1937,816,True,True,


In [25]:
# Batch 4 / Cell 4 - Candidate panel helpers
def batch4_visual_type_from_candidate(column: str, value: Any, resolved_path: Path | None = None) -> str:
    column_text = batch4_clean_text(column).lower()
    value_text = batch4_clean_text(value).lower()
    path_text = str(resolved_path or "").lower()
    text = f"{column_text} {value_text} {path_text}"

    is_actual_restored_path = (
        column_text in {"restored_path", "restored_image_path", "difference_map__restored_path"}
        or "data/processed/restored/stable_diffusion" in text
    )

    is_visual_diagnostic = any(
        token in text
        for token in [
            "error_map",
            "signed_improvement",
            "improvement_map",
            "heatmap",
            "overlay",
            "difference_map__damaged_error",
            "difference_map__restored_error",
        ]
    )

    if is_visual_diagnostic:
        return "difference_map"

    if is_actual_restored_path:
        return "restored_candidate"

    if column_text in {"damaged_path", "difference_map__damaged_path"}:
        return "damaged_input"

    if column_text in {"clean_path", "difference_map__clean_path"}:
        return "clean_reference"

    if column_text in {"mask_path", "difference_map__mask_path"}:
        return "other_visual"

    if any(token in text for token in ["difference", "diff"]):
        return "difference_map"

    return "other_visual"

def batch4_visual_score(visual_type: str) -> int:
    return {
        "damaged_input": 100,
        "restored_candidate": 95,
        "clean_reference": 90,
        "difference_map": 85,
        "other_visual": 10,
        "unresolved": -1,
    }.get(visual_type, 0)

def batch4_all_candidate_visual_paths(combined_row: dict) -> list[dict]:
    candidates = []

    for column, value in combined_row.items():
        if not batch4_is_image_like_path(value):
            continue

        resolved_path, resolve_status = batch4_resolve_local_path(value)
        visual_type = batch4_visual_type_from_candidate(column, value, resolved_path) if resolved_path is not None else "unresolved"

        candidates.append(
            {
                "column": column,
                "raw_path": batch4_clean_text(value),
                "resolved_path": resolved_path,
                "resolve_status": resolve_status,
                "visual_type": visual_type,
                "score": batch4_visual_score(visual_type),
            }
        )

    return candidates

def batch4_select_panel_visuals(combined_row: dict) -> tuple[list[dict], list[dict]]:
    candidates = batch4_all_candidate_visual_paths(combined_row)
    found_candidates = [candidate for candidate in candidates if candidate["resolved_path"] is not None]

    selected_visuals = []
    used_paths = set()

    preferred_order = ["damaged_input", "restored_candidate", "clean_reference", "difference_map", "other_visual"]

    for visual_type in preferred_order:
        visual_pool = [
            candidate for candidate in found_candidates
            if candidate["visual_type"] == visual_type
            and str(candidate["resolved_path"]) not in used_paths
        ]

        if not visual_pool:
            continue

        chosen = sorted(visual_pool, key=lambda item: (-item["score"], item["column"], str(item["resolved_path"])))[0]
        selected_visuals.append(chosen)
        used_paths.add(str(chosen["resolved_path"]))

        if len(selected_visuals) >= 4:
            break

    return selected_visuals, candidates

def batch4_draw_wrapped_text(draw: ImageDraw.ImageDraw, xy: tuple[int, int], text: str, font, fill: str, width_chars: int, line_spacing: int = 4) -> int:
    x, y = xy
    wrapped_lines = []
    for paragraph in batch4_clean_text(text).splitlines():
        wrapped_lines.extend(textwrap.wrap(paragraph, width=width_chars) if paragraph.strip() else [""])
    for line in wrapped_lines:
        draw.text((x, y), line, font=font, fill=fill)
        y += 15 + line_spacing
    return y

def batch4_paste_contained(canvas: Image.Image, image: Image.Image, box: tuple[int, int, int, int]):
    x0, y0, x1, y1 = box
    working_image = image.copy()
    working_image.thumbnail((x1 - x0, y1 - y0), BATCH4_RESAMPLE)
    paste_x = x0 + ((x1 - x0) - working_image.width) // 2
    paste_y = y0 + ((y1 - y0) - working_image.height) // 2
    canvas.paste(working_image, (paste_x, paste_y))

def batch4_create_text_only_panel(path: Path, title: str, body: str):
    canvas = Image.new("RGB", (1200, 680), "white")
    draw = ImageDraw.Draw(canvas)
    font = ImageFont.load_default()
    draw.rectangle((0, 0, 1200, 86), fill="#263238")
    draw.text((28, 28), title[:150], font=font, fill="white")
    batch4_draw_wrapped_text(draw, (36, 120), body, font, "#222222", width_chars=135, line_spacing=7)
    canvas.save(path, format="PNG")

def batch4_create_candidate_panel(rep_row: pd.Series, audit_row: pd.Series | None) -> dict:
    report_candidate_id = batch4_clean_text(rep_row.get(candidate_id_column))
    role = batch4_clean_text(rep_row.get("representative_role"))
    figure_id = f"sd_candidate_panel__{batch4_slug(role)}__{batch4_slug(report_candidate_id)}"
    output_path = BATCH4_PANEL_DIR / f"{figure_id}.png"

    combined_row = {}
    if audit_row is not None:
        combined_row.update(audit_row.to_dict())
    combined_row.update(rep_row.to_dict())

    selected_visuals, all_visual_candidates = batch4_select_panel_visuals(combined_row)
    selected_types = [visual["visual_type"] for visual in selected_visuals]

    status = batch4_clean_text(rep_row.get("candidate_status"))
    rank_mean = batch4_clean_text(rep_row.get("metric_rank_mean"))
    rank_count = batch4_clean_text(rep_row.get("metric_rank_count"))
    runtime_reference = batch4_clean_text(rep_row.get("runtime_reference"))
    memory_reference = batch4_clean_text(rep_row.get("memory_reference"))
    reason = batch4_clean_text(rep_row.get("selection_reason"))
    prompt_preview = batch4_clean_text(rep_row.get("prompt_preview"))

    title = f"{role} | {report_candidate_id}"
    body = (
        f"Candidate status: {status}\n"
        f"Mean metric rank: {rank_mean} across {rank_count} metrics\n"
        f"Runtime reference: {runtime_reference}\n"
        f"Memory reference: {memory_reference}\n"
        f"Selection reason: {reason}\n"
        f"Prompt preview: {prompt_preview[:450]}\n"
        f"Selected visual types: {selected_types}\n"
    )

    if not selected_visuals:
        batch4_create_text_only_panel(output_path, title, body + "\nNo local image paths were resolvable.")
    else:
        tile_width = 360
        tile_height = 330
        margin = 28
        header_height = 140
        footer_height = 155
        panel_count = len(selected_visuals)

        canvas_width = max(1200, margin * 2 + panel_count * tile_width + (panel_count - 1) * 18)
        canvas_height = header_height + tile_height + footer_height + margin

        canvas = Image.new("RGB", (canvas_width, canvas_height), "white")
        draw = ImageDraw.Draw(canvas)
        font = ImageFont.load_default()

        draw.rectangle((0, 0, canvas_width, 86), fill="#263238")
        draw.text((margin, 28), title[:180], font=font, fill="white")

        header_text = (
            f"Status: {status}    Metric rank mean: {rank_mean}    "
            f"Metric count: {rank_count}    Runtime: {runtime_reference}    Memory: {memory_reference}"
        )
        draw.text((margin, 104), header_text[:220], font=font, fill="#222222")

        y0 = header_height
        x = margin

        for visual in selected_visuals:
            image = batch4_load_image(visual["resolved_path"], max_side=900)
            draw.rectangle((x, y0, x + tile_width, y0 + tile_height), outline="#d0d0d0", width=2)
            draw.rectangle((x, y0, x + tile_width, y0 + 28), fill="#eeeeee")
            draw.text((x + 8, y0 + 8), visual["visual_type"], font=font, fill="#111111")
            batch4_paste_contained(canvas, image, (x + 8, y0 + 38, x + tile_width - 8, y0 + tile_height - 36))
            source_label = f"{visual['column']} | {batch4_rel_path(visual['resolved_path'])}"
            draw.text((x + 8, y0 + tile_height - 24), source_label[:52], font=font, fill="#444444")
            x += tile_width + 18

        footer_text = f"Selection reason: {reason}\nPrompt preview: {prompt_preview[:520]}"
        batch4_draw_wrapped_text(draw, (margin, y0 + tile_height + 24), footer_text, font, "#222222", width_chars=145, line_spacing=5)
        canvas.save(output_path, format="PNG")

    return {
        "figure_id": figure_id,
        "path": output_path,
        "report_candidate_id": report_candidate_id,
        "representative_role": role,
        "selected_visual_count": int(len(selected_visuals)),
        "selected_visual_types": ";".join(selected_types),
        "contains_damaged_input": "damaged_input" in selected_types,
        "contains_restored_candidate": "restored_candidate" in selected_types,
        "contains_clean_reference": "clean_reference" in selected_types,
        "uses_difference_map": "difference_map" in selected_types,
        "panel_status": "built_with_restored_candidate" if "restored_candidate" in selected_types else "built_missing_restored_candidate",
        "unresolved_visual_candidates": int(sum(candidate["resolved_path"] is None for candidate in all_visual_candidates)),
    }

print("Candidate panel helpers ready with restored-candidate enforcement.")

Candidate panel helpers ready with restored-candidate enforcement.


In [26]:
# Batch 4 / Cell 5 - Generate representative candidate panels
audit_lookup = {
    batch4_clean_text(row[candidate_id_column]): row
    for _, row in batch4_report_audit_df.iterrows()
}

batch4_panel_records = []

for _, rep_row in selected_representatives_df.iterrows():
    report_candidate_id = batch4_clean_text(rep_row[candidate_id_column])
    audit_row = audit_lookup.get(report_candidate_id)

    panel_record = batch4_create_candidate_panel(rep_row, audit_row)
    batch4_panel_records.append(panel_record)

    batch4_add_manifest_row(
        figure_id=panel_record["figure_id"],
        figure_group="representative_candidate_panel",
        title=f"Representative Candidate Panel: {panel_record['representative_role']}",
        caption=(
            "Selected Stable Diffusion candidate panel combining available visual evidence, "
            "metric-policy context, and difference-map evidence where available."
        ),
        path=panel_record["path"],
        source_table="batch4_representative_candidates_df + batch4_report_audit_df",
        source_rows=int(len(batch4_report_audit_df)),
        candidate_rows=1,
        representative_role=panel_record["representative_role"],
        report_candidate_id=panel_record["report_candidate_id"],
        evidence_columns=path_columns + metric_columns,
        notes=panel_record["panel_status"],
    )

batch4_panel_records_df = pd.DataFrame(batch4_panel_records)

print(f"Candidate panels generated: {len(batch4_panel_records_df):,}")
display(batch4_panel_records_df)

Candidate panels generated: 7


,figure_id,path,report_candidate_id,representative_role,selected_visual_count,selected_visual_types,contains_damaged_input,contains_restored_candidate,contains_clean_reference,uses_difference_map,panel_status,unresolved_visual_candidates
0,sd_candidate_panel__overall_high_metric_eviden...,D:\Masters\FH\Thesis\painting-restoration-eval...,sd_candidate_189f26cdeed04200,overall_high_metric_evidence,4,damaged_input;restored_candidate;clean_referen...,True,True,True,True,built_with_restored_candidate,11
1,sd_candidate_panel__overall_low_metric_evidenc...,D:\Masters\FH\Thesis\painting-restoration-eval...,sd_candidate_2a8339724e8533c4,overall_low_metric_evidence,4,damaged_input;restored_candidate;clean_referen...,True,True,True,True,built_with_restored_candidate,10
2,sd_candidate_panel__median_metric_evidence__sd...,D:\Masters\FH\Thesis\painting-restoration-eval...,sd_candidate_e0253d35510ef0c0,median_metric_evidence,4,damaged_input;restored_candidate;clean_referen...,True,True,True,True,built_with_restored_candidate,10
3,sd_candidate_panel__fast_successful_candidate_...,D:\Masters\FH\Thesis\painting-restoration-eval...,sd_candidate_72d917e63e54e65c,fast_successful_candidate,4,damaged_input;restored_candidate;clean_referen...,True,True,True,True,built_with_restored_candidate,11
4,sd_candidate_panel__slow_or_expensive_candidat...,D:\Masters\FH\Thesis\painting-restoration-eval...,sd_candidate_a38d731e796e4a49,slow_or_expensive_candidate,4,damaged_input;restored_candidate;clean_referen...,True,True,True,True,built_with_restored_candidate,10
5,sd_candidate_panel__high_memory_candidate__sd_...,D:\Masters\FH\Thesis\painting-restoration-eval...,sd_candidate_000c396305a705f7,high_memory_candidate,4,damaged_input;restored_candidate;clean_referen...,True,True,True,True,built_with_restored_candidate,10
6,sd_candidate_panel__difference_map_available_c...,D:\Masters\FH\Thesis\painting-restoration-eval...,sd_candidate_189f26cdeed04200,difference_map_available_candidate,4,damaged_input;restored_candidate;clean_referen...,True,True,True,True,built_with_restored_candidate,11


In [27]:
# Batch 4 / Cell 6 - Write strict figure manifest and validate
batch4_figure_manifest_df = pd.DataFrame(batch4_manifest_rows)

if batch4_figure_manifest_df.empty:
    raise RuntimeError("Batch 4 strict check failed: no figure manifest rows were created.")

batch4_figure_manifest_df = batch4_figure_manifest_df.sort_values(
    ["figure_group", "figure_id"],
    kind="stable",
).reset_index(drop=True)

batch4_figure_manifest_df.insert(0, "figure_order", np.arange(1, len(batch4_figure_manifest_df) + 1))

BATCH4_FIGURE_MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
batch4_figure_manifest_df.to_csv(BATCH4_FIGURE_MANIFEST_PATH, index=False)

batch4_manifest_shape = csv_shape(BATCH4_FIGURE_MANIFEST_PATH)

aggregate_plot_count = int(batch4_figure_manifest_df["figure_group"].eq("aggregate_plot").sum())
candidate_panel_count = int(batch4_figure_manifest_df["figure_group"].eq("representative_candidate_panel").sum())
selected_representative_count = int(len(selected_representatives_df))
html_ready_count = int(bool_series(batch4_figure_manifest_df["html_ready"]).sum())

restored_panel_count = int(batch4_panel_records_df["contains_restored_candidate"].sum()) if not batch4_panel_records_df.empty else 0
damaged_panel_count = int(batch4_panel_records_df["contains_damaged_input"].sum()) if not batch4_panel_records_df.empty else 0
clean_panel_count = int(batch4_panel_records_df["contains_clean_reference"].sum()) if not batch4_panel_records_df.empty else 0
difference_panel_count = int(batch4_panel_records_df["uses_difference_map"].sum()) if not batch4_panel_records_df.empty else 0
real_image_panel_count = int((batch4_panel_records_df["selected_visual_count"] > 0).sum()) if not batch4_panel_records_df.empty else 0

batch4_validation_rows = [
    validation_row("figure_manifest_written", rel(BATCH4_FIGURE_MANIFEST_PATH), "file exists", BATCH4_FIGURE_MANIFEST_PATH.is_file(), "Batch 4 figure manifest CSV was not written."),
    validation_row("figure_manifest_has_rows", int(batch4_manifest_shape[0] or 0), "> 0", int(batch4_manifest_shape[0] or 0) > 0, "Batch 4 figure manifest has no rows."),
    validation_row("all_manifest_assets_html_ready", html_ready_count, int(len(batch4_figure_manifest_df)), html_ready_count == int(len(batch4_figure_manifest_df)), "One or more manifest PNG assets are missing, unreadable, too small, or not HTML-ready."),
    validation_row("figure_ids_unique", int(batch4_figure_manifest_df["figure_id"].nunique(dropna=False)), int(len(batch4_figure_manifest_df)), int(batch4_figure_manifest_df["figure_id"].nunique(dropna=False)) == int(len(batch4_figure_manifest_df)), "Figure IDs are not unique."),
    validation_row("figure_paths_unique", int(batch4_figure_manifest_df["relative_path"].nunique(dropna=False)), int(len(batch4_figure_manifest_df)), int(batch4_figure_manifest_df["relative_path"].nunique(dropna=False)) == int(len(batch4_figure_manifest_df)), "Figure paths are not unique."),
    validation_row("aggregate_plot_count_strict_minimum", aggregate_plot_count, ">= 4", aggregate_plot_count >= 4, "Too few aggregate plots were created."),
    validation_row("candidate_panel_for_each_selected_representative", candidate_panel_count, selected_representative_count, candidate_panel_count == selected_representative_count, "Not every selected representative candidate has a generated panel."),
    validation_row("candidate_panels_include_real_images", real_image_panel_count, selected_representative_count, real_image_panel_count == selected_representative_count, "Not every representative panel includes resolved local images."),
    validation_row("candidate_panels_include_restored_candidate", restored_panel_count, selected_representative_count, restored_panel_count == selected_representative_count, "Not every representative panel includes a restored candidate image."),
    validation_row("candidate_panels_include_damaged_input", damaged_panel_count, selected_representative_count, damaged_panel_count == selected_representative_count, "Not every representative panel includes a damaged input image."),
    validation_row("candidate_panels_include_clean_reference", clean_panel_count, selected_representative_count, clean_panel_count == selected_representative_count, "Not every representative panel includes a clean reference image."),
    validation_row("candidate_panels_include_difference_map", difference_panel_count, "> 0", difference_panel_count > 0, "No representative candidate panel includes a difference-map image."),
    validation_row("manifest_titles_non_empty", int(batch4_figure_manifest_df["title"].apply(lambda value: bool(batch4_clean_text(value))).sum()), int(len(batch4_figure_manifest_df)), int(batch4_figure_manifest_df["title"].apply(lambda value: bool(batch4_clean_text(value))).sum()) == int(len(batch4_figure_manifest_df)), "One or more figure manifest rows have empty titles."),
    validation_row("manifest_captions_non_empty", int(batch4_figure_manifest_df["caption"].apply(lambda value: bool(batch4_clean_text(value))).sum()), int(len(batch4_figure_manifest_df)), int(batch4_figure_manifest_df["caption"].apply(lambda value: bool(batch4_clean_text(value))).sum()) == int(len(batch4_figure_manifest_df)), "One or more figure manifest rows have empty captions."),
]

batch4_validation_df = pd.DataFrame(batch4_validation_rows)
batch4_passed = bool(bool_series(batch4_validation_df["passed"]).all())

stage_manifest = read_json_if_exists(STAGE_MANIFEST_PATH)
stage_manifest.update(
    {
        "notebook_id": NOTEBOOK_ID,
        "notebook_title": NOTEBOOK_TITLE,
        "stage": "batch4_standardized_report_assets",
        "stage_status": "passed" if batch4_passed else "failed",
        "updated_at_utc": utc_now_iso(),
        "batch4": {
            "status": "passed" if batch4_passed else "failed",
            "figure_manifest_csv": rel(BATCH4_FIGURE_MANIFEST_PATH),
            "figure_manifest_rows": int(batch4_manifest_shape[0] or 0),
            "aggregate_plot_count": aggregate_plot_count,
            "candidate_panel_count": candidate_panel_count,
            "selected_representative_count": selected_representative_count,
            "html_ready_assets": html_ready_count,
            "restored_candidate_panels": restored_panel_count,
            "damaged_input_panels": damaged_panel_count,
            "clean_reference_panels": clean_panel_count,
            "difference_map_candidate_panels": difference_panel_count,
            "metric_columns_used": int(len(metric_columns)),
            "path_columns_scanned": int(len(path_columns)),
        },
    }
)
write_json(STAGE_MANIFEST_PATH, stage_manifest)

print(f"Saved figure manifest: {rel(BATCH4_FIGURE_MANIFEST_PATH)}")
print(f"Aggregate plots: {aggregate_plot_count}")
print(f"Candidate panels: {candidate_panel_count}")
print(f"Restored candidate panels: {restored_panel_count} / {selected_representative_count}")
print(f"Batch 4 checks passed: {int(bool_series(batch4_validation_df['passed']).sum())} / {len(batch4_validation_df)}")

display(batch4_validation_df)
display(batch4_panel_records_df)

if not batch4_passed:
    display(batch4_validation_df.loc[~bool_series(batch4_validation_df["passed"]), ["check_name", "actual", "expected", "failure_message"]])
    raise RuntimeError("Batch 4 validation failed. Fix figure generation or path resolution before final HTML report generation.")

display(batch4_figure_manifest_df)

print("Batch 4 passed. Standardized report assets and strict figure manifest are ready.")

Saved figure manifest: outputs/26_stable_diffusion_report_generation/figures/stable_diffusion_report_figure_manifest.csv
Aggregate plots: 6
Candidate panels: 7
Restored candidate panels: 7 / 7
Batch 4 checks passed: 14 / 14


,check_name,severity,actual,expected,passed,failure_message,checked_at_utc
0,figure_manifest_written,error,"""outputs/26_stable_diffusion_report_generation...","""file exists""",True,,2026-08-10T11:29:08.150282+00:00
1,figure_manifest_has_rows,error,13,"""> 0""",True,,2026-08-10T11:29:08.150282+00:00
2,all_manifest_assets_html_ready,error,13,13,True,,2026-08-10T11:29:08.150282+00:00
3,figure_ids_unique,error,13,13,True,,2026-08-10T11:29:08.150282+00:00
4,figure_paths_unique,error,13,13,True,,2026-08-10T11:29:08.150282+00:00
5,aggregate_plot_count_strict_minimum,error,6,""">= 4""",True,,2026-08-10T11:29:08.150282+00:00
6,candidate_panel_for_each_selected_representative,error,7,7,True,,2026-08-10T11:29:08.150282+00:00
7,candidate_panels_include_real_images,error,7,7,True,,2026-08-10T11:29:08.150282+00:00
8,candidate_panels_include_restored_candidate,error,7,7,True,,2026-08-10T11:29:08.150282+00:00
9,candidate_panels_include_damaged_input,error,7,7,True,,2026-08-10T11:29:08.150282+00:00


,figure_id,path,report_candidate_id,representative_role,selected_visual_count,selected_visual_types,contains_damaged_input,contains_restored_candidate,contains_clean_reference,uses_difference_map,panel_status,unresolved_visual_candidates
0,sd_candidate_panel__overall_high_metric_eviden...,D:\Masters\FH\Thesis\painting-restoration-eval...,sd_candidate_189f26cdeed04200,overall_high_metric_evidence,4,damaged_input;restored_candidate;clean_referen...,True,True,True,True,built_with_restored_candidate,11
1,sd_candidate_panel__overall_low_metric_evidenc...,D:\Masters\FH\Thesis\painting-restoration-eval...,sd_candidate_2a8339724e8533c4,overall_low_metric_evidence,4,damaged_input;restored_candidate;clean_referen...,True,True,True,True,built_with_restored_candidate,10
2,sd_candidate_panel__median_metric_evidence__sd...,D:\Masters\FH\Thesis\painting-restoration-eval...,sd_candidate_e0253d35510ef0c0,median_metric_evidence,4,damaged_input;restored_candidate;clean_referen...,True,True,True,True,built_with_restored_candidate,10
3,sd_candidate_panel__fast_successful_candidate_...,D:\Masters\FH\Thesis\painting-restoration-eval...,sd_candidate_72d917e63e54e65c,fast_successful_candidate,4,damaged_input;restored_candidate;clean_referen...,True,True,True,True,built_with_restored_candidate,11
4,sd_candidate_panel__slow_or_expensive_candidat...,D:\Masters\FH\Thesis\painting-restoration-eval...,sd_candidate_a38d731e796e4a49,slow_or_expensive_candidate,4,damaged_input;restored_candidate;clean_referen...,True,True,True,True,built_with_restored_candidate,10
5,sd_candidate_panel__high_memory_candidate__sd_...,D:\Masters\FH\Thesis\painting-restoration-eval...,sd_candidate_000c396305a705f7,high_memory_candidate,4,damaged_input;restored_candidate;clean_referen...,True,True,True,True,built_with_restored_candidate,10
6,sd_candidate_panel__difference_map_available_c...,D:\Masters\FH\Thesis\painting-restoration-eval...,sd_candidate_189f26cdeed04200,difference_map_available_candidate,4,damaged_input;restored_candidate;clean_referen...,True,True,True,True,built_with_restored_candidate,11


,figure_order,figure_id,figure_group,title,caption,file_path,relative_path,source_table,source_rows,candidate_rows,...,evidence_columns,notes,file_exists,is_png,file_size_bytes,width_px,height_px,png_readable,html_ready,validation_message
0,1,sd_candidate_metric_rank_distribution,aggregate_plot,Stable Diffusion Candidate Mean Metric-Rank Di...,Distribution of candidate-level mean percentil...,D:\Masters\FH\Thesis\painting-restoration-eval...,outputs/26_stable_diffusion_report_generation/...,batch4_report_audit_df,945,945,...,batch4_rank__classical_damaged_mae;batch4_rank...,,True,True,50486,1322,816,True,True,
1,2,sd_candidate_status_distribution,aggregate_plot,Stable Diffusion Candidate Status Distribution,Counts of candidate-level restoration statuses...,D:\Masters\FH\Thesis\painting-restoration-eval...,outputs/26_stable_diffusion_report_generation/...,batch4_report_audit_df,945,945,...,batch3_candidate_status,,True,True,34922,1252,869,True,True,
2,3,sd_metric_evidence_coverage,aggregate_plot,Stable Diffusion Metric Evidence Coverage,Coverage of numeric metric columns after Batch...,D:\Masters\FH\Thesis\painting-restoration-eval...,outputs/26_stable_diffusion_report_generation/...,batch4_report_audit_df,945,945,...,classical__damaged_mae;classical__damaged_mse;...,,True,True,188741,2449,1082,True,True,
3,4,sd_metric_family_coverage,aggregate_plot,Stable Diffusion Metric Family Coverage,"Compact overview of classical, LPIPS, and feat...",D:\Masters\FH\Thesis\painting-restoration-eval...,outputs/26_stable_diffusion_report_generation/...,batch4_report_audit_df,945,945,...,classical__damaged_mae;classical__damaged_mse;...,,True,True,52612,1290,851,True,True,
4,5,sd_representative_candidate_policy_overview,aggregate_plot,Stable Diffusion Representative Candidate Poli...,Metric-rank positions of selected representati...,D:\Masters\FH\Thesis\painting-restoration-eval...,outputs/26_stable_diffusion_report_generation/...,batch4_representative_candidates_df,8,7,...,representative_role;metric_rank_mean,,True,True,72809,1937,816,True,True,
5,6,sd_runtime_vs_metric_rank,aggregate_plot,Stable Diffusion Runtime Versus Metric Evidence,Candidate runtime reference plotted against me...,D:\Masters\FH\Thesis\painting-restoration-eval...,outputs/26_stable_diffusion_report_generation/...,batch4_report_audit_df,945,945,...,batch4_runtime_numeric__full_generation_durati...,,True,True,109475,1330,872,True,True,
6,7,sd_candidate_panel__difference_map_available_c...,representative_candidate_panel,Representative Candidate Panel: difference_map...,Selected Stable Diffusion candidate panel comb...,D:\Masters\FH\Thesis\painting-restoration-eval...,outputs/26_stable_diffusion_report_generation/...,batch4_representative_candidates_df + batch4_r...,945,1,...,difference_map__damaged_error_map_path;differe...,built_with_restored_candidate,True,True,172853,1550,653,True,True,
7,8,sd_candidate_panel__fast_successful_candidate_...,representative_candidate_panel,Representative Candidate Panel: fast_successfu...,Selected Stable Diffusion candidate panel comb...,D:\Masters\FH\Thesis\painting-restoration-eval...,outputs/26_stable_diffusion_report_generation/...,batch4_representative_candidates_df + batch4_r...,945,1,...,difference_map__damaged_error_map_path;differe...,built_with_restored_candidate,True,True,71421,1550,653,True,True,
8,9,sd_candidate_panel__high_memory_candidate__sd_...,representative_candidate_panel,Representative Candidate Panel: high_memory_ca...,Selected Stable Diffusion candidate panel comb...,D:\Masters\FH\Thesis\painting-restoration-eval...,outputs/26_stable_diffusion_report_generation/...,batch4_representative_candidates_df + batch4_r...,945,1,...,difference_map__damaged_error_map_path;differe...,built_with_restored_candidate,True,True,252996,1550,653,True,True,
9,10,sd_candidate_panel__median_metric_evidence__sd...,representative_candidate_panel,Representative Candidate Panel: median_metric_...,Selected Stable Diffusion candidate panel comb.

Batch 4 passed. Standardized report assets and strict figure manifest are ready.


In [28]:
# Batch 5 / Cell 1 - Setup and HTML helpers
from pathlib import Path
from typing import Any
import base64
import html
import json
import re
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from PIL import Image

BATCH5_REPORTS_DIR = OUTPUT_DIRS["reports"]
BATCH5_REPORT_PATH = BATCH5_REPORTS_DIR / "stable_diffusion_report.html"
BATCH5_REPORTS_DIR.mkdir(parents=True, exist_ok=True)

if "BATCH4_FIGURE_MANIFEST_PATH" not in globals():
    BATCH4_FIGURE_MANIFEST_PATH = OUTPUT_DIRS["figures"] / "stable_diffusion_report_figure_manifest.csv"

BATCH5_MIN_HTML_BYTES = 25_000

def batch5_clean_text(value: Any) -> str:
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass
    return str(value).strip()

def batch5_escape(value: Any) -> str:
    return html.escape(batch5_clean_text(value), quote=True)

def batch5_rel(path: Path | str) -> str:
    path = Path(path)
    if "rel" in globals():
        try:
            return rel(path)
        except Exception:
            pass
    return str(path)

def batch5_utc_now_iso() -> str:
    if "utc_now_iso" in globals():
        return utc_now_iso()
    return datetime.now(timezone.utc).isoformat(timespec="seconds").replace("+00:00", "Z")

def batch5_bool_series(series: pd.Series) -> pd.Series:
    if "bool_series" in globals():
        return bool_series(series)
    return series.fillna(False).astype(bool)

def batch5_read_json_if_exists(path: Path) -> dict:
    if "read_json_if_exists" in globals():
        return read_json_if_exists(path)
    if path.is_file():
        return json.loads(path.read_text(encoding="utf-8"))
    return {}

def batch5_write_json(path: Path, payload: dict):
    if "write_json" in globals():
        write_json(path, payload)
    else:
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")

def batch5_validation_row(check_name, actual, expected, passed, failure_message):
    if "validation_row" in globals():
        return validation_row(check_name, actual, expected, passed, failure_message)
    return {
        "check_name": check_name,
        "actual": actual,
        "expected": expected,
        "passed": bool(passed),
        "failure_message": failure_message,
    }

def batch5_resolve_path(value: Any) -> Path | None:
    text_value = batch5_clean_text(value)
    if not text_value:
        return None

    path = Path(text_value)
    candidates = [path] if path.is_absolute() else [Path.cwd() / path]

    for candidate in candidates:
        if candidate.is_file():
            return candidate.resolve()

    return None

def batch5_image_to_data_uri(path: Path) -> tuple[str, dict]:
    validation = {
        "path": str(path),
        "exists": path.is_file(),
        "readable": False,
        "width_px": 0,
        "height_px": 0,
        "file_size_bytes": 0,
        "data_uri_ready": False,
        "message": "",
    }

    if not path.is_file():
        validation["message"] = "image file missing"
        return "", validation

    validation["file_size_bytes"] = int(path.stat().st_size)

    try:
        with Image.open(path) as image:
            validation["width_px"], validation["height_px"] = image.size
            image.verify()
        validation["readable"] = True
    except Exception as exc:
        validation["message"] = f"image unreadable: {exc}"
        return "", validation

    encoded = base64.b64encode(path.read_bytes()).decode("ascii")
    validation["data_uri_ready"] = True
    return f"data:image/png;base64,{encoded}", validation

def batch5_format_value(value: Any) -> str:
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass

    if isinstance(value, float):
        if np.isfinite(value):
            return f"{value:.4f}"
        return ""

    text_value = str(value)
    if len(text_value) > 180:
        return text_value[:177] + "..."
    return text_value

def batch5_table_html(
    df: pd.DataFrame,
    max_rows: int = 25,
    columns: list[str] | None = None,
    table_class: str = "data-table",
) -> str:
    if df is None or df.empty:
        return '<p class="muted">No rows available.</p>'

    table_df = df.copy()
    if columns:
        table_df = table_df[[column for column in columns if column in table_df.columns]]

    table_df = table_df.head(max_rows).copy()

    header_html = "".join(f"<th>{batch5_escape(column)}</th>" for column in table_df.columns)

    row_html = []
    for _, row in table_df.iterrows():
        cells = "".join(
            f"<td>{batch5_escape(batch5_format_value(row[column]))}</td>"
            for column in table_df.columns
        )
        row_html.append(f"<tr>{cells}</tr>")

    note = ""
    if len(df) > max_rows:
        note = f'<p class="table-note">Showing {max_rows:,} of {len(df):,} rows.</p>'

    return (
        f'{note}<div class="table-wrap"><table class="{table_class}">'
        f"<thead><tr>{header_html}</tr></thead>"
        f"<tbody>{''.join(row_html)}</tbody></table></div>"
    )

print(f"Batch 5 report path: {batch5_rel(BATCH5_REPORT_PATH)}")

Batch 5 report path: outputs/26_stable_diffusion_report_generation/reports/stable_diffusion_report.html


In [29]:
# Batch 5 / Cell 2 - Load report inputs strictly
if not Path(BATCH4_FIGURE_MANIFEST_PATH).is_file():
    raise RuntimeError(f"Batch 5 requires Batch 4 figure manifest: {BATCH4_FIGURE_MANIFEST_PATH}")

batch5_figure_manifest_df = pd.read_csv(BATCH4_FIGURE_MANIFEST_PATH)

if "batch4_report_audit_df" in globals() and isinstance(batch4_report_audit_df, pd.DataFrame) and not batch4_report_audit_df.empty:
    batch5_report_audit_df = batch4_report_audit_df.copy()
elif "batch3_report_audit_df" in globals() and isinstance(batch3_report_audit_df, pd.DataFrame) and not batch3_report_audit_df.empty:
    batch5_report_audit_df = batch3_report_audit_df.copy()
else:
    batch5_report_audit_df = pd.read_csv(BATCH2_REPORT_AUDIT_PATH)

if "batch4_representative_candidates_df" in globals() and isinstance(batch4_representative_candidates_df, pd.DataFrame) and not batch4_representative_candidates_df.empty:
    batch5_representative_candidates_df = batch4_representative_candidates_df.copy()
elif "batch3_representative_candidates_df" in globals() and isinstance(batch3_representative_candidates_df, pd.DataFrame) and not batch3_representative_candidates_df.empty:
    batch5_representative_candidates_df = batch3_representative_candidates_df.copy()
else:
    batch5_representative_candidates_df = pd.read_csv(BATCH3_REPRESENTATIVE_PATH)

if "batch3_summary_df" in globals() and isinstance(batch3_summary_df, pd.DataFrame) and not batch3_summary_df.empty:
    batch5_summary_df = batch3_summary_df.copy()
else:
    batch5_summary_df = pd.read_csv(BATCH3_SUMMARY_PATH)

required_manifest_columns = [
    "figure_id",
    "figure_group",
    "title",
    "caption",
    "file_path",
    "relative_path",
    "html_ready",
]

missing_manifest_columns = [
    column for column in required_manifest_columns
    if column not in batch5_figure_manifest_df.columns
]

if missing_manifest_columns:
    raise RuntimeError(f"Batch 5 missing figure manifest columns: {missing_manifest_columns}")

batch5_selected_representatives_df = batch5_representative_candidates_df.loc[
    batch5_representative_candidates_df["selection_status"].eq("selected")
].copy()

if batch5_figure_manifest_df.empty:
    raise RuntimeError("Batch 5 requires a non-empty figure manifest.")

if batch5_report_audit_df.empty:
    raise RuntimeError("Batch 5 requires a non-empty report audit table.")

if batch5_selected_representatives_df.empty:
    raise RuntimeError("Batch 5 requires selected representative candidates.")

batch5_figure_manifest_df["resolved_file_path"] = batch5_figure_manifest_df["file_path"].apply(batch5_resolve_path)
batch5_figure_manifest_df["resolved_file_exists"] = batch5_figure_manifest_df["resolved_file_path"].apply(lambda value: value is not None and Path(value).is_file())

if not batch5_bool_series(batch5_figure_manifest_df["html_ready"]).all():
    display(
        batch5_figure_manifest_df.loc[
            ~batch5_bool_series(batch5_figure_manifest_df["html_ready"]),
            ["figure_id", "figure_group", "relative_path", "html_ready", "validation_message"],
        ]
    )
    raise RuntimeError("Batch 5 strict check failed: Batch 4 manifest contains non-HTML-ready assets.")

if not batch5_figure_manifest_df["resolved_file_exists"].all():
    display(
        batch5_figure_manifest_df.loc[
            ~batch5_figure_manifest_df["resolved_file_exists"],
            ["figure_id", "figure_group", "file_path", "relative_path"],
        ]
    )
    raise RuntimeError("Batch 5 strict check failed: one or more figure files cannot be resolved.")

print(f"Figure manifest rows: {len(batch5_figure_manifest_df):,}")
print(f"Report audit rows: {len(batch5_report_audit_df):,}")
print(f"Representative rows: {len(batch5_selected_representatives_df):,}")
print(f"Summary rows: {len(batch5_summary_df):,}")

display(
    pd.DataFrame(
        [
            {"input": "figure_manifest", "rows": len(batch5_figure_manifest_df), "path": batch5_rel(BATCH4_FIGURE_MANIFEST_PATH)},
            {"input": "report_audit", "rows": len(batch5_report_audit_df), "path": batch5_clean_text(globals().get("BATCH2_REPORT_AUDIT_PATH", ""))},
            {"input": "batch3_summary", "rows": len(batch5_summary_df), "path": batch5_clean_text(globals().get("BATCH3_SUMMARY_PATH", ""))},
            {"input": "representatives", "rows": len(batch5_selected_representatives_df), "path": batch5_clean_text(globals().get("BATCH3_REPRESENTATIVE_PATH", ""))},
        ]
    )
)

Figure manifest rows: 13
Report audit rows: 945
Representative rows: 7
Summary rows: 132


,input,rows,path
0,figure_manifest,13,outputs/26_stable_diffusion_report_generation/...
1,report_audit,945,D:\Masters\FH\Thesis\painting-restoration-eval...
2,batch3_summary,132,D:\Masters\FH\Thesis\painting-restoration-eval...
3,representatives,7,D:\Masters\FH\Thesis\painting-restoration-eval...


In [30]:
# Batch 5 / Cell 3 - Build metric, status, and candidate evidence tables
candidate_id_column = "report_candidate_id"

metric_columns = batch3_numeric_metric_columns(batch5_report_audit_df) if "batch3_numeric_metric_columns" in globals() else []

def batch5_metric_family(column: str) -> str:
    lowered = column.lower()
    if "lpips" in lowered:
        return "lpips"
    if "feature" in lowered or "clip" in lowered or "dino" in lowered:
        return "feature"
    if any(token in lowered for token in ["ssim", "psnr", "mae", "mse", "rmse"]):
        return "classical"
    return "other"

def batch5_higher_is_better(column: str) -> bool:
    if "batch3_higher_is_better" in globals():
        return batch3_higher_is_better(column)

    lowered = column.lower()
    if "improvement" in lowered:
        return True
    if any(token in lowered for token in ["ssim", "psnr", "clip", "dino", "similarity"]):
        return True
    if any(token in lowered for token in ["lpips", "mae", "rmse", "mse", "distance", "error"]):
        return False
    return True

metric_summary_rows = []

for column in metric_columns:
    values = pd.to_numeric(batch5_report_audit_df[column], errors="coerce")
    non_null = values.dropna()

    metric_summary_rows.append(
        {
            "metric_column": column,
            "metric_family": batch5_metric_family(column),
            "direction": "higher is better" if batch5_higher_is_better(column) else "lower is better",
            "non_null_count": int(non_null.shape[0]),
            "coverage_percent": float(values.notna().mean() * 100.0),
            "mean": float(non_null.mean()) if not non_null.empty else np.nan,
            "median": float(non_null.median()) if not non_null.empty else np.nan,
            "std": float(non_null.std()) if non_null.shape[0] > 1 else np.nan,
            "min": float(non_null.min()) if not non_null.empty else np.nan,
            "max": float(non_null.max()) if not non_null.empty else np.nan,
        }
    )

batch5_metric_summary_df = pd.DataFrame(metric_summary_rows).sort_values(
    ["metric_family", "metric_column"],
    kind="stable",
)

status_column = "batch3_candidate_status" if "batch3_candidate_status" in batch5_report_audit_df.columns else None

batch5_status_summary_df = (
    batch5_report_audit_df[status_column]
    .fillna("unknown")
    .value_counts()
    .rename_axis("candidate_status")
    .reset_index(name="candidate_count")
) if status_column else pd.DataFrame(
    [{"candidate_status": "status column unavailable", "candidate_count": len(batch5_report_audit_df)}]
)

rank_column = (
    "batch4_metric_rank_mean"
    if "batch4_metric_rank_mean" in batch5_report_audit_df.columns
    else "batch3_metric_rank_mean"
    if "batch3_metric_rank_mean" in batch5_report_audit_df.columns
    else None
)

if rank_column:
    candidate_rank_df = batch5_report_audit_df.copy()
    candidate_rank_df[rank_column] = pd.to_numeric(candidate_rank_df[rank_column], errors="coerce")
    candidate_rank_df = candidate_rank_df.loc[candidate_rank_df[rank_column].notna()].copy()
    top_candidate_df = candidate_rank_df.sort_values(rank_column, ascending=False, kind="stable").head(15)
    low_candidate_df = candidate_rank_df.sort_values(rank_column, ascending=True, kind="stable").head(15)
else:
    top_candidate_df = pd.DataFrame()
    low_candidate_df = pd.DataFrame()

representative_display_columns = [
    "representative_role", "selection_reason", "report_candidate_id", "candidate_status",
    "metric_rank_mean", "metric_rank_count", "runtime_reference", "memory_reference",
    "has_restored_candidate", "has_difference_map", "duplicate_policy_note",
    "prompt_preview", "primary_restored_candidate_path", "primary_difference_map_path",
]

candidate_display_columns = [
    column for column in [
        candidate_id_column, rank_column, "batch3_candidate_status",
        "batch4_metric_rank_count", "batch4_runtime_reference", "batch4_memory_reference",
    ]
    if column and column in batch5_report_audit_df.columns
]

figure_group_summary_df = (
    batch5_figure_manifest_df
    .groupby("figure_group", as_index=False)
    .agg(
        figure_count=("figure_id", "count"),
        html_ready_count=("html_ready", lambda values: int(batch5_bool_series(values).sum())),
    )
)

print(f"Strict metric columns included in report: {len(metric_columns):,}")
print(f"Rank column used: {rank_column or 'none'}")

display(figure_group_summary_df)
display(batch5_status_summary_df)
display(batch5_metric_summary_df.head(30))

Strict metric columns included in report: 48
Rank column used: batch4_metric_rank_mean


D:\Masters\FH\Thesis\painting-restoration-eval\.venv\Lib\site-packages\pandas\core\nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
D:\Masters\FH\Thesis\painting-restoration-eval\.venv\Lib\site-packages\pandas\core\nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


,figure_group,figure_count,html_ready_count
0,aggregate_plot,6,6
1,representative_candidate_panel,7,7


,candidate_status,candidate_count
0,successful,945


,metric_column,metric_family,direction,non_null_count,coverage_percent,mean,median,std,min,max
0,batch4_rank__classical_damaged_mae,classical,lower is better,945,100.0,0.499471,0.499471,2.886834e-01,0.000000,9.497354e-01
1,batch4_rank__classical_damaged_mse,classical,lower is better,945,100.0,0.499471,0.501587,2.886834e-01,0.000000,9.497354e-01
2,batch4_rank__classical_damaged_psnr,classical,higher is better,945,100.0,0.500529,0.500529,1.110811e-16,0.500529,5.005291e-01
3,batch4_rank__classical_damaged_ssim,classical,higher is better,945,100.0,0.500529,0.500529,2.886834e-01,0.001058,9.507937e-01
4,batch4_rank__classical_mae_improvement,classical,higher is better,945,100.0,0.500529,0.500529,2.886857e-01,0.001058,1.000000e+00
5,batch4_rank__classical_mse_improvement,classical,higher is better,945,100.0,0.500529,0.500529,2.886857e-01,0.001058,1.000000e+00
6,batch4_rank__classical_psnr_improvement,classical,higher is better,945,100.0,0.500529,0.500529,2.886857e-01,0.001058,1.000000e+00
7,batch4_rank__classical_restored_mae,classical,lower is better,945,100.0,0.499471,0.499471,2.886857e-01,0.000000,9.497354e-01
8,batch4_rank__classical_restored_mse,classical,lower is better,945,100.0,0.499471,0.499471,2.886857e-01,0.000000,9.497354e-01
9,batch4_rank__classical_restored_psnr,classical,higher is better,945,100.0,0.500529,0.500529,1.110811e-16,0.500529,5.005291e-01


In [31]:
# Batch 5 / Cell 4 - Embed figures and compose HTML sections
batch5_image_records = []
batch5_embedded_manifest_df = batch5_figure_manifest_df.copy()

def batch5_figure_card(row: pd.Series) -> str:
    path = Path(row["resolved_file_path"])
    data_uri, image_validation = batch5_image_to_data_uri(path)
    batch5_image_records.append(
        {
            "figure_id": row["figure_id"],
            "figure_group": row["figure_group"],
            "relative_path": row["relative_path"],
            **image_validation,
        }
    )

    if not data_uri:
        return f"""
        <article class="figure-card figure-error">
            <h3>{batch5_escape(row["title"])}</h3>
            <p>{batch5_escape(row["caption"])}</p>
            <p class="error">Image could not be embedded: {batch5_escape(image_validation["message"])}</p>
        </article>
        """

    return f"""
    <article class="figure-card">
        <h3>{batch5_escape(row["title"])}</h3>
        <img src="{data_uri}" alt="{batch5_escape(row["title"])}" loading="lazy">
        <p>{batch5_escape(row["caption"])}</p>
        <p class="asset-path">{batch5_escape(row["relative_path"])}</p>
    </article>
    """

def batch5_gallery_html(group_name: str, empty_message: str) -> str:
    sort_column = "figure_order" if "figure_order" in batch5_embedded_manifest_df.columns else "figure_id"

    gallery_df = batch5_embedded_manifest_df.loc[
        batch5_embedded_manifest_df["figure_group"].eq(group_name)
    ].sort_values(sort_column, kind="stable")

    if gallery_df.empty:
        return f'<p class="muted">{batch5_escape(empty_message)}</p>'

    return "\n".join(batch5_figure_card(row) for _, row in gallery_df.iterrows())

aggregate_gallery_html = batch5_gallery_html(
    "aggregate_plot",
    "No aggregate plots were found in the Batch 4 manifest.",
)

representative_gallery_html = batch5_gallery_html(
    "representative_candidate_panel",
    "No representative candidate panels were found in the Batch 4 manifest.",
)

status_count = int(len(batch5_status_summary_df))
candidate_count = int(len(batch5_report_audit_df))
metric_count = int(len(metric_columns))
figure_count = int(len(batch5_figure_manifest_df))
aggregate_count = int(batch5_figure_manifest_df["figure_group"].eq("aggregate_plot").sum())
panel_count = int(batch5_figure_manifest_df["figure_group"].eq("representative_candidate_panel").sum())
representative_count = int(len(batch5_selected_representatives_df))

if rank_column and not top_candidate_df.empty:
    rank_median = float(pd.to_numeric(batch5_report_audit_df[rank_column], errors="coerce").median())
    rank_text = f"{rank_median:.4f}"
else:
    rank_text = "unavailable"

report_title = "Stable Diffusion Restoration Report"

batch5_provenance_df = pd.DataFrame(
    [
        {"artifact": "Final HTML report", "path": batch5_rel(BATCH5_REPORT_PATH), "role": "Standalone report output"},
        {"artifact": "Figure manifest", "path": batch5_rel(BATCH4_FIGURE_MANIFEST_PATH), "role": "Strict source of report figures"},
        {"artifact": "Batch 3 summary", "path": batch5_clean_text(globals().get("BATCH3_SUMMARY_PATH", "")), "role": "Compact candidate summary"},
        {"artifact": "Representative candidates", "path": batch5_clean_text(globals().get("BATCH3_REPRESENTATIVE_PATH", "")), "role": "Policy-selected report examples"},
        {"artifact": "Report audit", "path": batch5_clean_text(globals().get("BATCH2_REPORT_AUDIT_PATH", "")), "role": "Candidate-level consolidated evidence"},
    ]
)

batch5_audit_columns_df = pd.DataFrame(
    {"column_name": list(batch5_report_audit_df.columns)}
)

provenance_table_html = batch5_table_html(batch5_provenance_df, max_rows=10)
status_summary_table_html = batch5_table_html(batch5_status_summary_df, max_rows=20)
representative_policy_table_html = batch5_table_html(
    batch5_selected_representatives_df,
    max_rows=50,
    columns=representative_display_columns,
)
metric_summary_table_html = batch5_table_html(batch5_metric_summary_df, max_rows=80)
top_candidate_table_html = batch5_table_html(top_candidate_df, max_rows=15, columns=candidate_display_columns)
low_candidate_table_html = batch5_table_html(low_candidate_df, max_rows=15, columns=candidate_display_columns)
batch3_summary_table_html = batch5_table_html(batch5_summary_df, max_rows=60)
figure_manifest_table_html = batch5_table_html(
    batch5_figure_manifest_df.drop(columns=["resolved_file_path"], errors="ignore"),
    max_rows=200,
)
audit_columns_table_html = batch5_table_html(batch5_audit_columns_df, max_rows=300)

batch5_css = """
:root {
    --bg: #f7f7f4;
    --paper: #ffffff;
    --ink: #1d252c;
    --muted: #5b6770;
    --line: #d8ddd8;
    --accent: #284f70;
    --accent-soft: #e8f0f6;
    --bad: #a33131;
}

* {
    box-sizing: border-box;
}

body {
    margin: 0;
    background: var(--bg);
    color: var(--ink);
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
    line-height: 1.48;
}

header {
    background: #1f2a33;
    color: white;
    padding: 34px 44px;
}

header h1 {
    margin: 0 0 10px;
    font-size: 32px;
}

header p {
    max-width: 1120px;
    margin: 0;
    color: #dce5eb;
}

main {
    max-width: 1280px;
    margin: 0 auto;
    padding: 28px 28px 64px;
}

section {
    background: var(--paper);
    border: 1px solid var(--line);
    border-radius: 8px;
    padding: 24px;
    margin: 0 0 22px;
}

h2 {
    margin: 0 0 14px;
    font-size: 23px;
}

h3 {
    margin: 0 0 10px;
    font-size: 17px;
}

p {
    margin: 0 0 12px;
}

.card-grid {
    display: grid;
    grid-template-columns: repeat(auto-fit, minmax(185px, 1fr));
    gap: 12px;
    margin: 16px 0 4px;
}

.metric-card {
    border: 1px solid var(--line);
    border-radius: 8px;
    padding: 14px;
    background: #fbfcfb;
}

.metric-card .value {
    display: block;
    font-size: 27px;
    font-weight: 700;
    color: var(--accent);
    margin-bottom: 3px;
}

.metric-card .label {
    color: var(--muted);
    font-size: 13px;
}

.figure-grid {
    display: grid;
    grid-template-columns: repeat(auto-fit, minmax(360px, 1fr));
    gap: 16px;
}

.figure-card {
    border: 1px solid var(--line);
    border-radius: 8px;
    padding: 14px;
    background: #fbfcfb;
}

.figure-card img {
    width: 100%;
    height: auto;
    display: block;
    border: 1px solid var(--line);
    border-radius: 6px;
    background: white;
    margin: 10px 0;
}

.figure-card p {
    color: var(--muted);
    font-size: 13px;
}

.figure-card .asset-path {
    font-family: ui-monospace, SFMono-Regular, Consolas, monospace;
    font-size: 11px;
    color: #69767d;
    overflow-wrap: anywhere;
}

.figure-error {
    border-color: var(--bad);
}

.error {
    color: var(--bad);
    font-weight: 600;
}

.muted,
.table-note {
    color: var(--muted);
}

.callout {
    background: var(--accent-soft);
    border-left: 4px solid var(--accent);
    padding: 14px 16px;
    border-radius: 6px;
}

.table-wrap {
    overflow-x: auto;
    border: 1px solid var(--line);
    border-radius: 8px;
}

.data-table {
    width: 100%;
    border-collapse: collapse;
    font-size: 13px;
}

.data-table th,
.data-table td {
    border-bottom: 1px solid var(--line);
    padding: 8px 10px;
    text-align: left;
    vertical-align: top;
}

.data-table th {
    background: #eef2f0;
    font-weight: 650;
}

.two-col {
    display: grid;
    grid-template-columns: minmax(0, 1fr) minmax(0, 1fr);
    gap: 18px;
}

@media (max-width: 860px) {
    header {
        padding: 26px 22px;
    }

    main {
        padding: 18px;
    }

    section {
        padding: 18px;
    }

    .two-col {
        grid-template-columns: 1fr;
    }

    .figure-grid {
        grid-template-columns: 1fr;
    }
}
"""

overview_cards_html = f"""
<div class="card-grid">
    <div class="metric-card"><span class="value">{candidate_count:,}</span><span class="label">candidate audit rows</span></div>
    <div class="metric-card"><span class="value">{representative_count:,}</span><span class="label">selected representative cases</span></div>
    <div class="metric-card"><span class="value">{metric_count:,}</span><span class="label">numeric metric columns</span></div>
    <div class="metric-card"><span class="value">{figure_count:,}</span><span class="label">embedded report figures</span></div>
    <div class="metric-card"><span class="value">{aggregate_count:,}</span><span class="label">aggregate plots</span></div>
    <div class="metric-card"><span class="value">{panel_count:,}</span><span class="label">candidate panels</span></div>
    <div class="metric-card"><span class="value">{rank_text}</span><span class="label">median metric-rank evidence</span></div>
</div>
"""

method_notes_html = """
<p>
This report is an evidence-oriented Stable Diffusion restoration summary. It does not treat any single
metric as a final truth signal. Instead, it combines candidate status, metric coverage, rank-based
comparisons, visual panels, and difference-map evidence where available.
</p>
<p>
The representative examples are policy selections from Batch 3. They are useful for report discussion,
failure inspection, and qualitative sanity checks, but they should not be described as globally optimal
without the later cross-method comparison batches.
</p>
"""

batch5_html = f"""<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>{batch5_escape(report_title)}</title>
<style>
{batch5_css}
</style>
</head>
<body>
<header>
    <h1>{batch5_escape(report_title)}</h1>
    <p>
        Standalone HTML report generated by Notebook 15 Stable Diffusion reporting workflow.
        Generated at {batch5_escape(batch5_utc_now_iso())}.
    </p>
</header>

<main>
<section id="overview">
    <h2>Overview</h2>
    <div class="callout">
        {method_notes_html}
    </div>
    {overview_cards_html}
</section>

<section id="provenance">
    <h2>Input and Output Provenance</h2>
    {provenance_table_html}
</section>

<section id="status-summary">
    <h2>Candidate Status Summary</h2>
    <p class="muted">This table shows the candidate status distribution carried into the report.</p>
    {status_summary_table_html}
</section>

<section id="aggregate-plots">
    <h2>Aggregate Plots</h2>
    <p class="muted">
        These figures summarize status distribution, metric coverage, metric families,
        rank distribution, runtime evidence where available, and representative-policy evidence.
    </p>
    <div class="figure-grid">
        {aggregate_gallery_html}
    </div>
</section>

<section id="representative-panels">
    <h2>Representative Candidate Panels</h2>
    <p class="muted">
        These panels combine available painting outputs, damaged/restored views, heatmaps or difference maps,
        prompt previews, runtime references, and metric-policy context.
    </p>
    <div class="figure-grid">
        {representative_gallery_html}
    </div>
</section>

<section id="representative-policy">
    <h2>Representative Selection Policy</h2>
    <p class="muted">
        Selected representatives are kept explicit so the final thesis text can explain why each visual example appears.
    </p>
    {representative_policy_table_html}
</section>

<section id="metric-evidence">
    <h2>Metric Evidence Summary</h2>
    <p class="muted">
        Coverage and descriptive statistics for numeric metric evidence. Direction is inferred from metric names
        and should be checked before making final thesis claims.
    </p>
    {metric_summary_table_html}
</section>

<section id="candidate-rank-examples">
    <h2>Candidate Rank Examples</h2>
    <div class="two-col">
        <div>
            <h3>Higher Metric-Rank Candidates</h3>
            {top_candidate_table_html}
        </div>
        <div>
            <h3>Lower Metric-Rank Candidates</h3>
            {low_candidate_table_html}
        </div>
    </div>
</section>

<section id="batch3-summary">
    <h2>Compact Batch 3 Summary</h2>
    <p class="muted">Compact summary generated before report asset construction.</p>
    {batch3_summary_table_html}
</section>

<section id="figure-manifest">
    <h2>Figure Manifest Appendix</h2>
    <p class="muted">Every figure embedded in this report comes from this strict Batch 4 manifest.</p>
    {figure_manifest_table_html}
</section>

<section id="audit-columns">
    <h2>Audit Column Appendix</h2>
    <p class="muted">
        Available report-audit columns are listed to make downstream report expansion easier without guessing.
    </p>
    {audit_columns_table_html}
</section>
</main>
</body>
</html>
"""

print(f"HTML sections composed. Embedded image attempts: {len(batch5_image_records):,}")

HTML sections composed. Embedded image attempts: 13


In [32]:
# Batch 5 / Cell 5 - Write HTML, validate standalone report, update stage manifest
BATCH5_REPORT_PATH.write_text(batch5_html, encoding="utf-8")

batch5_image_validation_df = pd.DataFrame(batch5_image_records)
batch5_report_text = BATCH5_REPORT_PATH.read_text(encoding="utf-8")
batch5_report_size = int(BATCH5_REPORT_PATH.stat().st_size)

external_img_refs = re.findall(r'<img[^>]+src=["\'](?!data:image/)([^"\']+)["\']', batch5_report_text)
missing_sections = [
    section_id for section_id in [
        "overview",
        "provenance",
        "status-summary",
        "aggregate-plots",
        "representative-panels",
        "representative-policy",
        "metric-evidence",
        "candidate-rank-examples",
        "figure-manifest",
    ]
    if f'id="{section_id}"' not in batch5_report_text
]

embedded_image_count = batch5_report_text.count("data:image/png;base64,")
expected_image_count = int(len(batch5_figure_manifest_df))
embedded_image_ready_count = (
    int(batch5_image_validation_df["data_uri_ready"].sum())
    if not batch5_image_validation_df.empty
    else 0
)

batch5_validation_rows = [
    batch5_validation_row(
        "html_report_written",
        batch5_rel(BATCH5_REPORT_PATH),
        "file exists",
        BATCH5_REPORT_PATH.is_file(),
        "Stable Diffusion HTML report was not written.",
    ),
    batch5_validation_row(
        "html_report_large_enough",
        batch5_report_size,
        f">= {BATCH5_MIN_HTML_BYTES}",
        batch5_report_size >= BATCH5_MIN_HTML_BYTES,
        "HTML report is suspiciously small and may be missing embedded assets or sections.",
    ),
    batch5_validation_row(
        "all_manifest_images_embedded",
        embedded_image_count,
        expected_image_count,
        embedded_image_count == expected_image_count,
        "Not every Batch 4 manifest image was embedded into the standalone HTML.",
    ),
    batch5_validation_row(
        "all_image_embeddings_ready",
        embedded_image_ready_count,
        expected_image_count,
        embedded_image_ready_count == expected_image_count,
        "One or more image files could not be encoded into the standalone HTML.",
    ),
    batch5_validation_row(
        "no_external_image_references",
        len(external_img_refs),
        0,
        len(external_img_refs) == 0,
        "Standalone report contains external or relative image references.",
    ),
    batch5_validation_row(
        "all_required_sections_present",
        len(missing_sections),
        0,
        len(missing_sections) == 0,
        f"Missing required HTML sections: {missing_sections}",
    ),
    batch5_validation_row(
        "representative_panels_present",
        int(batch5_figure_manifest_df["figure_group"].eq("representative_candidate_panel").sum()),
        int(len(batch5_selected_representatives_df)),
        int(batch5_figure_manifest_df["figure_group"].eq("representative_candidate_panel").sum()) == int(len(batch5_selected_representatives_df)),
        "Representative panel count does not match selected representative candidate count.",
    ),
    batch5_validation_row(
        "aggregate_plots_present",
        int(batch5_figure_manifest_df["figure_group"].eq("aggregate_plot").sum()),
        ">= 4",
        int(batch5_figure_manifest_df["figure_group"].eq("aggregate_plot").sum()) >= 4,
        "Too few aggregate plots are available in the final report.",
    ),
    batch5_validation_row(
        "metric_summary_available",
        int(len(batch5_metric_summary_df)),
        "> 0",
        int(len(batch5_metric_summary_df)) > 0,
        "Metric summary table is empty.",
    ),
]

batch5_validation_df = pd.DataFrame(batch5_validation_rows)
batch5_passed = bool(batch5_bool_series(batch5_validation_df["passed"]).all())

stage_manifest = batch5_read_json_if_exists(STAGE_MANIFEST_PATH)
stage_manifest.update(
    {
        "notebook_id": NOTEBOOK_ID,
        "notebook_title": NOTEBOOK_TITLE,
        "stage": "batch5_standalone_html_report",
        "stage_status": "passed" if batch5_passed else "failed",
        "updated_at_utc": batch5_utc_now_iso(),
        "batch5": {
            "status": "passed" if batch5_passed else "failed",
            "html_report": batch5_rel(BATCH5_REPORT_PATH),
            "html_report_size_bytes": batch5_report_size,
            "embedded_png_images": embedded_image_count,
            "expected_manifest_images": expected_image_count,
            "candidate_audit_rows": int(len(batch5_report_audit_df)),
            "selected_representatives": int(len(batch5_selected_representatives_df)),
            "metric_columns_used": int(len(metric_columns)),
            "aggregate_plots": int(batch5_figure_manifest_df["figure_group"].eq("aggregate_plot").sum()),
            "representative_candidate_panels": int(batch5_figure_manifest_df["figure_group"].eq("representative_candidate_panel").sum()),
        },
    }
)
batch5_write_json(STAGE_MANIFEST_PATH, stage_manifest)

print(f"Saved standalone HTML report: {batch5_rel(BATCH5_REPORT_PATH)}")
print(f"HTML size: {batch5_report_size:,} bytes")
print(f"Embedded PNG images: {embedded_image_count:,} / {expected_image_count:,}")
print(f"Batch 5 checks passed: {int(batch5_bool_series(batch5_validation_df['passed']).sum())} / {len(batch5_validation_df)}")

display(batch5_validation_df)

if not batch5_image_validation_df.empty:
    display(batch5_image_validation_df)

if not batch5_passed:
    display(
        batch5_validation_df.loc[
            ~batch5_bool_series(batch5_validation_df["passed"]),
            ["check_name", "actual", "expected", "failure_message"],
        ]
    )
    raise RuntimeError("Batch 5 validation failed. Fix report construction before treating the HTML as final.")

print("Batch 5 passed. Standalone Stable Diffusion HTML report is ready.")

Saved standalone HTML report: outputs/26_stable_diffusion_report_generation/reports/stable_diffusion_report.html
HTML size: 2,426,923 bytes
Embedded PNG images: 13 / 13
Batch 5 checks passed: 9 / 9


,check_name,severity,actual,expected,passed,failure_message,checked_at_utc
0,html_report_written,error,"""outputs/26_stable_diffusion_report_generation...","""file exists""",True,,2026-08-10T11:29:08.914595+00:00
1,html_report_large_enough,error,2426923,""">= 25000""",True,,2026-08-10T11:29:08.914595+00:00
2,all_manifest_images_embedded,error,13,13,True,,2026-08-10T11:29:08.914595+00:00
3,all_image_embeddings_ready,error,13,13,True,,2026-08-10T11:29:08.914595+00:00
4,no_external_image_references,error,0,0,True,,2026-08-10T11:29:08.914595+00:00
5,all_required_sections_present,error,0,0,True,,2026-08-10T11:29:08.914595+00:00
6,representative_panels_present,error,7,7,True,,2026-08-10T11:29:08.916609+00:00
7,aggregate_plots_present,error,6,""">= 4""",True,,2026-08-10T11:29:08.916609+00:00
8,metric_summary_available,error,48,"""> 0""",True,,2026-08-10T11:29:08.916609+00:00


,figure_id,figure_group,relative_path,path,exists,readable,width_px,height_px,file_size_bytes,data_uri_ready,message
0,sd_candidate_metric_rank_distribution,aggregate_plot,outputs/26_stable_diffusion_report_generation/...,D:\Masters\FH\Thesis\painting-restoration-eval...,True,True,1322,816,50486,True,
1,sd_candidate_status_distribution,aggregate_plot,outputs/26_stable_diffusion_report_generation/...,D:\Masters\FH\Thesis\painting-restoration-eval...,True,True,1252,869,34922,True,
2,sd_metric_evidence_coverage,aggregate_plot,outputs/26_stable_diffusion_report_generation/...,D:\Masters\FH\Thesis\painting-restoration-eval...,True,True,2449,1082,188741,True,
3,sd_metric_family_coverage,aggregate_plot,outputs/26_stable_diffusion_report_generation/...,D:\Masters\FH\Thesis\painting-restoration-eval...,True,True,1290,851,52612,True,
4,sd_representative_candidate_policy_overview,aggregate_plot,outputs/26_stable_diffusion_report_generation/...,D:\Masters\FH\Thesis\painting-restoration-eval...,True,True,1937,816,72809,True,
5,sd_runtime_vs_metric_rank,aggregate_plot,outputs/26_stable_diffusion_report_generation/...,D:\Masters\FH\Thesis\painting-restoration-eval...,True,True,1330,872,109475,True,
6,sd_candidate_panel__difference_map_available_c...,representative_candidate_panel,outputs/26_stable_diffusion_report_generation/...,D:\Masters\FH\Thesis\painting-restoration-eval...,True,True,1550,653,172853,True,
7,sd_candidate_panel__fast_successful_candidate_...,representative_candidate_panel,outputs/26_stable_diffusion_report_generation/...,D:\Masters\FH\Thesis\painting-restoration-eval...,True,True,1550,653,71421,True,
8,sd_candidate_panel__high_memory_candidate__sd_...,representative_candidate_panel,outputs/26_stable_diffusion_report_generation/...,D:\Masters\FH\Thesis\painting-restoration-eval...,True,True,1550,653,252996,True,
9,sd_candidate_panel__median_metric_evidence__sd...,representative_candidate_panel,outputs/26_stable_diffusion_report_generation/...,D:\Masters\FH\Thesis\painting-restoration-eval...,True,True,1550,653,134002,True,


Batch 5 passed. Standalone Stable Diffusion HTML report is ready.


In [33]:
# Batch 6 / Cell 1 - Setup final validation paths and helpers
from pathlib import Path
from typing import Any
from datetime import datetime, timezone
import hashlib
import json
import os
import re

import numpy as np
import pandas as pd

BATCH6_NAME = "final_validation_artifact_index_handoff"

def batch6_utc_now_iso() -> str:
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()

def batch6_clean_text(value: Any) -> str:
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass
    return str(value).strip()

def batch6_bool_series(values: pd.Series) -> pd.Series:
    if values is None:
        return pd.Series(dtype=bool)

    if values.dtype == bool:
        return values.fillna(False)

    return (
        values.astype(str)
        .str.strip()
        .str.lower()
        .isin(["true", "1", "yes", "y", "passed", "pass"])
    )

def batch6_path_from_global(name: str) -> Path | None:
    value = globals().get(name)
    text_value = batch6_clean_text(value)
    if not text_value:
        return None

    try:
        return Path(text_value)
    except Exception:
        return None

def batch6_resolve_path(value: Any) -> Path | None:
    text_value = batch6_clean_text(value)
    if not text_value:
        return None

    path = Path(text_value)
    candidates = [path] if path.is_absolute() else [Path.cwd() / path]

    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()

    return (Path.cwd() / path).resolve() if not path.is_absolute() else path

def batch6_rel(path: Any) -> str:
    resolved = batch6_resolve_path(path)
    if resolved is None:
        return ""

    try:
        return str(resolved.relative_to(Path.cwd()))
    except Exception:
        return str(resolved)

def batch6_read_json_if_exists(path: Path) -> dict:
    if "read_json_if_exists" in globals():
        return read_json_if_exists(path)

    if path.is_file():
        return json.loads(path.read_text(encoding="utf-8"))

    return {}

def batch6_write_json(path: Path, payload: dict):
    if "write_json" in globals():
        write_json(path, payload)
    else:
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")

def batch6_validation_row(check_name, actual, expected, passed, failure_message):
    if "validation_row" in globals():
        return validation_row(check_name, actual, expected, passed, failure_message)

    return {
        "check_name": check_name,
        "actual": actual,
        "expected": expected,
        "passed": bool(passed),
        "failure_message": "" if passed else failure_message,
    }

def batch6_sha256(path: Path) -> str:
    if not path.is_file():
        return ""

    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()

def batch6_csv_row_count(path: Path) -> int | None:
    if not path.is_file() or path.suffix.lower() != ".csv":
        return None

    try:
        return int(pd.read_csv(path).shape[0])
    except Exception:
        return None

def batch6_file_type(path: Path) -> str:
    suffix = path.suffix.lower().lstrip(".")
    return suffix or "directory" if path.is_dir() else "unknown"

stage_manifest_candidate = batch6_path_from_global("STAGE_MANIFEST_PATH")
figure_manifest_candidate = batch6_path_from_global("BATCH4_FIGURE_MANIFEST_PATH")
html_report_candidate = batch6_path_from_global("BATCH5_REPORT_PATH")

if stage_manifest_candidate is not None:
    BATCH6_MANIFEST_DIR = batch6_resolve_path(stage_manifest_candidate).parent
elif figure_manifest_candidate is not None:
    BATCH6_MANIFEST_DIR = batch6_resolve_path(figure_manifest_candidate).parent
elif html_report_candidate is not None:
    BATCH6_MANIFEST_DIR = batch6_resolve_path(html_report_candidate).parent / "manifests"
else:
    BATCH6_MANIFEST_DIR = Path.cwd() / "manifests"

BATCH6_REPORT_ROOT_DIR = BATCH6_MANIFEST_DIR.parent
BATCH6_VALIDATION_DIR = BATCH6_REPORT_ROOT_DIR / "validation"

BATCH6_MANIFEST_DIR.mkdir(parents=True, exist_ok=True)
BATCH6_VALIDATION_DIR.mkdir(parents=True, exist_ok=True)

BATCH6_FINAL_VALIDATION_PATH = BATCH6_VALIDATION_DIR / "stable_diffusion_report_final_validation.csv"
BATCH6_ARTIFACT_INDEX_PATH = BATCH6_MANIFEST_DIR / "stable_diffusion_report_artifact_index.csv"
BATCH6_HANDOFF_MANIFEST_PATH = BATCH6_MANIFEST_DIR / "stable_diffusion_report_handoff_manifest.json"
BATCH6_STAGE_MANIFEST_PATH = BATCH6_MANIFEST_DIR / "stable_diffusion_report_stage_manifest.json"

STAGE_MANIFEST_PATH = BATCH6_STAGE_MANIFEST_PATH

print(f"Batch 6 validation path: {batch6_rel(BATCH6_FINAL_VALIDATION_PATH)}")
print(f"Batch 6 artifact index path: {batch6_rel(BATCH6_ARTIFACT_INDEX_PATH)}")
print(f"Batch 6 handoff manifest path: {batch6_rel(BATCH6_HANDOFF_MANIFEST_PATH)}")
print(f"Batch 6 stage manifest path: {batch6_rel(BATCH6_STAGE_MANIFEST_PATH)}")

Batch 6 validation path: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\26_stable_diffusion_report_generation\validation\stable_diffusion_report_final_validation.csv
Batch 6 artifact index path: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\26_stable_diffusion_report_generation\manifests\stable_diffusion_report_artifact_index.csv
Batch 6 handoff manifest path: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\26_stable_diffusion_report_generation\manifests\stable_diffusion_report_handoff_manifest.json
Batch 6 stage manifest path: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\26_stable_diffusion_report_generation\manifests\stable_diffusion_report_stage_manifest.json


In [34]:
# Batch 6 / Cell 2 - Load final inputs
BATCH6_SOURCE_PATHS = {
    "final_html_report": batch6_path_from_global("BATCH5_REPORT_PATH"),
    "figure_manifest": batch6_path_from_global("BATCH4_FIGURE_MANIFEST_PATH"),
    "report_audit": batch6_path_from_global("BATCH2_REPORT_AUDIT_PATH"),
    "batch3_summary": batch6_path_from_global("BATCH3_SUMMARY_PATH"),
    "representative_candidates": batch6_path_from_global("BATCH3_REPRESENTATIVE_PATH"),
    "stage_manifest": BATCH6_STAGE_MANIFEST_PATH,
}

BATCH6_HTML_PATH = batch6_resolve_path(BATCH6_SOURCE_PATHS["final_html_report"])
BATCH6_FIGURE_MANIFEST_PATH = batch6_resolve_path(BATCH6_SOURCE_PATHS["figure_manifest"])
BATCH6_REPORT_AUDIT_PATH = batch6_resolve_path(BATCH6_SOURCE_PATHS["report_audit"])
BATCH6_SUMMARY_PATH = batch6_resolve_path(BATCH6_SOURCE_PATHS["batch3_summary"])
BATCH6_REPRESENTATIVE_PATH = batch6_resolve_path(BATCH6_SOURCE_PATHS["representative_candidates"])

batch6_html_text = ""
if BATCH6_HTML_PATH is not None and BATCH6_HTML_PATH.is_file():
    batch6_html_text = BATCH6_HTML_PATH.read_text(encoding="utf-8")

if BATCH6_FIGURE_MANIFEST_PATH is not None and BATCH6_FIGURE_MANIFEST_PATH.is_file():
    batch6_figure_manifest_df = pd.read_csv(BATCH6_FIGURE_MANIFEST_PATH)
else:
    batch6_figure_manifest_df = pd.DataFrame()

if "batch5_report_audit_df" in globals() and isinstance(batch5_report_audit_df, pd.DataFrame):
    batch6_report_audit_df = batch5_report_audit_df.copy()
elif BATCH6_REPORT_AUDIT_PATH is not None and BATCH6_REPORT_AUDIT_PATH.is_file():
    batch6_report_audit_df = pd.read_csv(BATCH6_REPORT_AUDIT_PATH)
else:
    batch6_report_audit_df = pd.DataFrame()

if "batch5_summary_df" in globals() and isinstance(batch5_summary_df, pd.DataFrame):
    batch6_summary_df = batch5_summary_df.copy()
elif BATCH6_SUMMARY_PATH is not None and BATCH6_SUMMARY_PATH.is_file():
    batch6_summary_df = pd.read_csv(BATCH6_SUMMARY_PATH)
else:
    batch6_summary_df = pd.DataFrame()

if "batch5_selected_representatives_df" in globals() and isinstance(batch5_selected_representatives_df, pd.DataFrame):
    batch6_selected_representatives_df = batch5_selected_representatives_df.copy()
elif BATCH6_REPRESENTATIVE_PATH is not None and BATCH6_REPRESENTATIVE_PATH.is_file():
    batch6_representative_df = pd.read_csv(BATCH6_REPRESENTATIVE_PATH)
    if "selection_status" in batch6_representative_df.columns:
        batch6_selected_representatives_df = batch6_representative_df.loc[
            batch6_representative_df["selection_status"].astype(str).str.lower().eq("selected")
        ].copy()
    else:
        batch6_selected_representatives_df = batch6_representative_df.copy()
else:
    batch6_selected_representatives_df = pd.DataFrame()

batch6_stage_manifest_existing = batch6_read_json_if_exists(BATCH6_STAGE_MANIFEST_PATH)

batch6_source_status_df = pd.DataFrame(
    [
        {
            "artifact_id": artifact_id,
            "path": batch6_rel(path),
            "exists": bool(batch6_resolve_path(path) is not None and batch6_resolve_path(path).exists()),
            "is_file": bool(batch6_resolve_path(path) is not None and batch6_resolve_path(path).is_file()),
        }
        for artifact_id, path in BATCH6_SOURCE_PATHS.items()
    ]
)

display(batch6_source_status_df)

print(f"HTML bytes loaded: {len(batch6_html_text.encode('utf-8')):,}")
print(f"Figure manifest rows: {len(batch6_figure_manifest_df):,}")
print(f"Report audit rows: {len(batch6_report_audit_df):,}")
print(f"Summary rows: {len(batch6_summary_df):,}")
print(f"Selected representatives: {len(batch6_selected_representatives_df):,}")

,artifact_id,path,exists,is_file
0,final_html_report,D:\Masters\FH\Thesis\painting-restoration-eval...,True,True
1,figure_manifest,D:\Masters\FH\Thesis\painting-restoration-eval...,True,True
2,report_audit,D:\Masters\FH\Thesis\painting-restoration-eval...,True,True
3,batch3_summary,D:\Masters\FH\Thesis\painting-restoration-eval...,True,True
4,representative_candidates,D:\Masters\FH\Thesis\painting-restoration-eval...,True,True
5,stage_manifest,D:\Masters\FH\Thesis\painting-restoration-eval...,True,True


HTML bytes loaded: 2,426,482
Figure manifest rows: 13
Report audit rows: 945
Summary rows: 132
Selected representatives: 7


In [35]:
# Batch 6 / Cell 3 - Run final validation and save CSV
required_html_sections = [
    "overview",
    "provenance",
    "status-summary",
    "aggregate-plots",
    "representative-panels",
    "representative-policy",
    "metric-evidence",
    "candidate-rank-examples",
    "figure-manifest",
    "audit-columns",
]

missing_html_sections = [
    section_id
    for section_id in required_html_sections
    if f'id="{section_id}"' not in batch6_html_text
]

external_img_refs = re.findall(
    r'<img[^>]+src=["\'](?!data:image/)([^"\']+)["\']',
    batch6_html_text,
)

embedded_png_count = batch6_html_text.count("data:image/png;base64,")
expected_png_count = int(len(batch6_figure_manifest_df))

required_figure_manifest_columns = [
    "figure_id",
    "figure_group",
    "title",
    "caption",
    "file_path",
    "relative_path",
    "html_ready",
]

missing_figure_manifest_columns = [
    column for column in required_figure_manifest_columns
    if column not in batch6_figure_manifest_df.columns
]

if not batch6_figure_manifest_df.empty and "file_path" in batch6_figure_manifest_df.columns:
    batch6_figure_manifest_df["batch6_resolved_file_path"] = batch6_figure_manifest_df["file_path"].apply(batch6_resolve_path)
    batch6_figure_manifest_df["batch6_file_exists"] = batch6_figure_manifest_df["batch6_resolved_file_path"].apply(
        lambda value: bool(value is not None and Path(value).is_file())
    )
else:
    batch6_figure_manifest_df["batch6_file_exists"] = False

if not batch6_figure_manifest_df.empty and "html_ready" in batch6_figure_manifest_df.columns:
    all_figures_html_ready = bool(batch6_bool_series(batch6_figure_manifest_df["html_ready"]).all())
else:
    all_figures_html_ready = False

all_figure_files_exist = bool(
    not batch6_figure_manifest_df.empty
    and batch6_figure_manifest_df["batch6_file_exists"].all()
)

batch5_recorded_status = batch6_clean_text(
    batch6_stage_manifest_existing.get("batch5", {}).get("status", "")
).lower()

batch5_inferred_passed = bool(
    BATCH6_HTML_PATH is not None
    and BATCH6_HTML_PATH.is_file()
    and expected_png_count > 0
    and embedded_png_count == expected_png_count
    and len(external_img_refs) == 0
    and len(missing_html_sections) == 0
)

batch5_recorded_or_inferred_passed = (
    batch5_recorded_status == "passed"
    or batch6_stage_manifest_existing.get("stage_status", "").lower() == "passed"
    or batch5_inferred_passed
)

min_html_bytes = int(globals().get("BATCH5_MIN_HTML_BYTES", 10_000))
html_size_bytes = int(BATCH6_HTML_PATH.stat().st_size) if BATCH6_HTML_PATH is not None and BATCH6_HTML_PATH.is_file() else 0

batch6_validation_rows = [
    batch6_validation_row(
        "output_directories_created",
        f"{batch6_rel(BATCH6_VALIDATION_DIR)} | {batch6_rel(BATCH6_MANIFEST_DIR)}",
        "directories exist",
        BATCH6_VALIDATION_DIR.is_dir() and BATCH6_MANIFEST_DIR.is_dir(),
        "Batch 6 output directories could not be created.",
    ),
    batch6_validation_row(
        "final_html_report_exists",
        batch6_rel(BATCH6_HTML_PATH),
        "file exists",
        BATCH6_HTML_PATH is not None and BATCH6_HTML_PATH.is_file(),
        "Batch 5 standalone HTML report is missing.",
    ),
    batch6_validation_row(
        "final_html_report_large_enough",
        html_size_bytes,
        f">= {min_html_bytes}",
        html_size_bytes >= min_html_bytes,
        "Standalone HTML report is suspiciously small.",
    ),
    batch6_validation_row(
        "required_html_sections_present",
        len(missing_html_sections),
        0,
        len(missing_html_sections) == 0,
        f"Missing HTML sections: {missing_html_sections}",
    ),
    batch6_validation_row(
        "no_external_image_references",
        len(external_img_refs),
        0,
        len(external_img_refs) == 0,
        "Standalone report contains non-embedded image references.",
    ),
    batch6_validation_row(
        "figure_manifest_exists",
        batch6_rel(BATCH6_FIGURE_MANIFEST_PATH),
        "file exists",
        BATCH6_FIGURE_MANIFEST_PATH is not None and BATCH6_FIGURE_MANIFEST_PATH.is_file(),
        "Batch 4 figure manifest is missing.",
    ),
    batch6_validation_row(
        "figure_manifest_nonempty",
        len(batch6_figure_manifest_df),
        "> 0",
        len(batch6_figure_manifest_df) > 0,
        "Figure manifest is empty.",
    ),
    batch6_validation_row(
        "figure_manifest_required_columns_present",
        missing_figure_manifest_columns,
        [],
        len(missing_figure_manifest_columns) == 0,
        f"Missing figure manifest columns: {missing_figure_manifest_columns}",
    ),
    batch6_validation_row(
        "all_manifest_figure_files_exist",
        int(batch6_figure_manifest_df["batch6_file_exists"].sum()) if not batch6_figure_manifest_df.empty else 0,
        expected_png_count,
        all_figure_files_exist,
        "One or more figure manifest files are missing.",
    ),
    batch6_validation_row(
        "all_manifest_figures_html_ready",
        bool(all_figures_html_ready),
        True,
        all_figures_html_ready,
        "One or more figure manifest rows are not marked html_ready.",
    ),
    batch6_validation_row(
        "embedded_image_count_matches_manifest",
        embedded_png_count,
        expected_png_count,
        expected_png_count > 0 and embedded_png_count == expected_png_count,
        "Embedded PNG count does not match Batch 4 figure manifest rows.",
    ),
    batch6_validation_row(
        "report_audit_exists",
        batch6_rel(BATCH6_REPORT_AUDIT_PATH),
        "file exists",
        BATCH6_REPORT_AUDIT_PATH is not None and BATCH6_REPORT_AUDIT_PATH.is_file(),
        "Report audit CSV is missing.",
    ),
    batch6_validation_row(
        "report_audit_nonempty",
        len(batch6_report_audit_df),
        "> 0",
        len(batch6_report_audit_df) > 0,
        "Report audit table is empty.",
    ),
    batch6_validation_row(
        "summary_nonempty",
        len(batch6_summary_df),
        "> 0",
        len(batch6_summary_df) > 0,
        "Batch 3 summary table is empty or missing.",
    ),
    batch6_validation_row(
        "selected_representatives_nonempty",
        len(batch6_selected_representatives_df),
        "> 0",
        len(batch6_selected_representatives_df) > 0,
        "No selected representative candidates are available.",
    ),
    batch6_validation_row(
        "batch5_recorded_or_inferred_passed",
        batch5_recorded_status or "inferred_from_html_checks",
        "passed",
        batch5_recorded_or_inferred_passed,
        "Batch 5 was not recorded as passed and could not be inferred from report checks.",
    ),
]

batch6_final_validation_df = pd.DataFrame(batch6_validation_rows)
batch6_final_validation_df.to_csv(BATCH6_FINAL_VALIDATION_PATH, index=False)

batch6_validation_passed = bool(batch6_bool_series(batch6_final_validation_df["passed"]).all())

print(f"Saved final validation CSV: {batch6_rel(BATCH6_FINAL_VALIDATION_PATH)}")
print(f"Batch 6 validation checks passed: {int(batch6_bool_series(batch6_final_validation_df['passed']).sum())} / {len(batch6_final_validation_df)}")

display(batch6_final_validation_df)

if not batch6_validation_passed:
    display(
        batch6_final_validation_df.loc[
            ~batch6_bool_series(batch6_final_validation_df["passed"]),
            ["check_name", "actual", "expected", "failure_message"],
        ]
    )

Saved final validation CSV: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\26_stable_diffusion_report_generation\validation\stable_diffusion_report_final_validation.csv
Batch 6 validation checks passed: 16 / 16


,check_name,severity,actual,expected,passed,failure_message,checked_at_utc
0,output_directories_created,error,"""D:\\Masters\\FH\\Thesis\\painting-restoration...","""directories exist""",True,,2026-08-10T11:29:09.183522+00:00
1,final_html_report_exists,error,"""D:\\Masters\\FH\\Thesis\\painting-restoration...","""file exists""",True,,2026-08-10T11:29:09.183522+00:00
2,final_html_report_large_enough,error,2426923,""">= 25000""",True,,2026-08-10T11:29:09.183522+00:00
3,required_html_sections_present,error,0,0,True,,2026-08-10T11:29:09.183522+00:00
4,no_external_image_references,error,0,0,True,,2026-08-10T11:29:09.183522+00:00
5,figure_manifest_exists,error,"""D:\\Masters\\FH\\Thesis\\painting-restoration...","""file exists""",True,,2026-08-10T11:29:09.183522+00:00
6,figure_manifest_nonempty,error,13,"""> 0""",True,,2026-08-10T11:29:09.183522+00:00
7,figure_manifest_required_columns_present,error,[],[],True,,2026-08-10T11:29:09.183522+00:00
8,all_manifest_figure_files_exist,error,13,13,True,,2026-08-10T11:29:09.183522+00:00
9,all_manifest_figures_html_ready,error,true,true,True,,2026-08-10T11:29:09.183522+00:00


In [36]:
# Batch 6 / Cell 4 - Build artifact index, handoff manifest, and stage manifest
def batch6_artifact_record(artifact_id: str, path: Path | None, role: str, required: bool = True) -> dict:
    resolved = batch6_resolve_path(path)
    exists = bool(resolved is not None and resolved.exists())
    is_file = bool(resolved is not None and resolved.is_file())

    omit_hash = artifact_id == "artifact_index_csv"

    return {
        "artifact_id": artifact_id,
        "artifact_role": role,
        "required_for_handoff": bool(required),
        "path": batch6_rel(resolved),
        "absolute_path": str(resolved) if resolved is not None else "",
        "exists": exists,
        "is_file": is_file,
        "file_type": batch6_file_type(resolved) if resolved is not None else "",
        "size_bytes": int(resolved.stat().st_size) if is_file else 0,
        "sha256": "" if omit_hash else batch6_sha256(resolved) if is_file else "",
        "row_count": batch6_csv_row_count(resolved) if is_file else None,
        "hash_note": "self-referential hash omitted" if omit_hash else "",
        "indexed_at_utc": batch6_utc_now_iso(),
    }

batch6_handoff_manifest = {
    "notebook_id": globals().get("NOTEBOOK_ID", "15_stable_diffusion_report"),
    "notebook_title": globals().get("NOTEBOOK_TITLE", "Stable Diffusion Restoration Report"),
    "stage": "batch6_final_validation_artifact_index_handoff",
    "handoff_status": "passed" if batch6_validation_passed else "failed",
    "generated_at_utc": batch6_utc_now_iso(),
    "summary": {
        "html_report": batch6_rel(BATCH6_HTML_PATH),
        "html_report_size_bytes": html_size_bytes,
        "candidate_audit_rows": int(len(batch6_report_audit_df)),
        "figure_manifest_rows": int(len(batch6_figure_manifest_df)),
        "embedded_png_images": int(embedded_png_count),
        "selected_representatives": int(len(batch6_selected_representatives_df)),
        "validation_checks": int(len(batch6_final_validation_df)),
        "validation_checks_passed": int(batch6_bool_series(batch6_final_validation_df["passed"]).sum()),
    },
    "primary_outputs": {
        "final_validation_csv": batch6_rel(BATCH6_FINAL_VALIDATION_PATH),
        "artifact_index_csv": batch6_rel(BATCH6_ARTIFACT_INDEX_PATH),
        "handoff_manifest_json": batch6_rel(BATCH6_HANDOFF_MANIFEST_PATH),
        "stage_manifest_json": batch6_rel(BATCH6_STAGE_MANIFEST_PATH),
        "standalone_html_report": batch6_rel(BATCH6_HTML_PATH),
    },
    "source_artifacts": {
        "figure_manifest": batch6_rel(BATCH6_FIGURE_MANIFEST_PATH),
        "report_audit": batch6_rel(BATCH6_REPORT_AUDIT_PATH),
        "batch3_summary": batch6_rel(BATCH6_SUMMARY_PATH),
        "representative_candidates": batch6_rel(BATCH6_REPRESENTATIVE_PATH),
    },
    "handoff_notes": [
        "The HTML report is standalone and should contain embedded PNG assets only.",
        "Representative examples are evidence selections, not final cross-method winners.",
        "Metric direction is inferred by naming conventions and should be checked before final thesis wording.",
        "This handoff closes the Stable Diffusion report-generation notebook stage.",
    ],
    "failed_checks": batch6_final_validation_df.loc[
        ~batch6_bool_series(batch6_final_validation_df["passed"]),
        ["check_name", "actual", "expected", "failure_message"],
    ].to_dict(orient="records"),
}

batch6_write_json(BATCH6_HANDOFF_MANIFEST_PATH, batch6_handoff_manifest)

batch6_stage_manifest = batch6_read_json_if_exists(BATCH6_STAGE_MANIFEST_PATH)
batch6_stage_manifest.update(
    {
        "notebook_id": globals().get("NOTEBOOK_ID", "15_stable_diffusion_report"),
        "notebook_title": globals().get("NOTEBOOK_TITLE", "Stable Diffusion Restoration Report"),
        "stage": "batch6_final_validation_artifact_index_handoff",
        "stage_status": "passed" if batch6_validation_passed else "failed",
        "updated_at_utc": batch6_utc_now_iso(),
        "batch6": {
            "status": "passed" if batch6_validation_passed else "failed",
            "final_validation_csv": batch6_rel(BATCH6_FINAL_VALIDATION_PATH),
            "artifact_index_csv": batch6_rel(BATCH6_ARTIFACT_INDEX_PATH),
            "handoff_manifest_json": batch6_rel(BATCH6_HANDOFF_MANIFEST_PATH),
            "stage_manifest_json": batch6_rel(BATCH6_STAGE_MANIFEST_PATH),
            "standalone_html_report": batch6_rel(BATCH6_HTML_PATH),
            "validation_checks": int(len(batch6_final_validation_df)),
            "validation_checks_passed": int(batch6_bool_series(batch6_final_validation_df["passed"]).sum()),
            "html_report_size_bytes": int(html_size_bytes),
            "embedded_png_images": int(embedded_png_count),
            "expected_manifest_images": int(expected_png_count),
            "candidate_audit_rows": int(len(batch6_report_audit_df)),
            "selected_representatives": int(len(batch6_selected_representatives_df)),
        },
    }
)

batch6_write_json(BATCH6_STAGE_MANIFEST_PATH, batch6_stage_manifest)

batch6_artifact_specs = [
    ("standalone_html_report", BATCH6_HTML_PATH, "Final standalone Stable Diffusion HTML report", True),
    ("final_validation_csv", BATCH6_FINAL_VALIDATION_PATH, "Batch 6 final validation table", True),
    ("artifact_index_csv", BATCH6_ARTIFACT_INDEX_PATH, "Machine-readable handoff artifact index", True),
    ("handoff_manifest_json", BATCH6_HANDOFF_MANIFEST_PATH, "Compact final handoff manifest", True),
    ("stage_manifest_json", BATCH6_STAGE_MANIFEST_PATH, "Shared stage manifest updated through Batch 6", True),
    ("figure_manifest_csv", BATCH6_FIGURE_MANIFEST_PATH, "Batch 4 strict report figure manifest", True),
    ("report_audit_csv", BATCH6_REPORT_AUDIT_PATH, "Candidate-level consolidated report audit", True),
    ("batch3_summary_csv", BATCH6_SUMMARY_PATH, "Compact Batch 3 summary", True),
    ("representative_candidates_csv", BATCH6_REPRESENTATIVE_PATH, "Representative candidate selection table", True),
]

batch6_artifact_index_df = pd.DataFrame(
    [
        batch6_artifact_record(artifact_id, path, role, required)
        for artifact_id, path, role, required in batch6_artifact_specs
    ]
)

batch6_artifact_index_df.to_csv(BATCH6_ARTIFACT_INDEX_PATH, index=False)

print(f"Saved handoff manifest: {batch6_rel(BATCH6_HANDOFF_MANIFEST_PATH)}")
print(f"Updated stage manifest: {batch6_rel(BATCH6_STAGE_MANIFEST_PATH)}")
print(f"Saved artifact index: {batch6_rel(BATCH6_ARTIFACT_INDEX_PATH)}")

display(batch6_artifact_index_df)

Saved handoff manifest: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\26_stable_diffusion_report_generation\manifests\stable_diffusion_report_handoff_manifest.json
Updated stage manifest: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\26_stable_diffusion_report_generation\manifests\stable_diffusion_report_stage_manifest.json
Saved artifact index: D:\Masters\FH\Thesis\painting-restoration-eval\outputs\26_stable_diffusion_report_generation\manifests\stable_diffusion_report_artifact_index.csv


,artifact_id,artifact_role,required_for_handoff,path,absolute_path,exists,is_file,file_type,size_bytes,sha256,row_count,hash_note,indexed_at_utc
0,standalone_html_report,Final standalone Stable Diffusion HTML report,True,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,True,True,unknown,2426923,164c1649c003348668b0b3b997eb6f481440c3622bb15e...,NaN,,2026-08-10T11:29:09+00:00
1,final_validation_csv,Batch 6 final validation table,True,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,True,True,unknown,2170,e2771e68514914229f0d931b212ae44a3735b15ab744be...,16.0,,2026-08-10T11:29:09+00:00
2,artifact_index_csv,Machine-readable handoff artifact index,True,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,True,True,unknown,4355,,9.0,self-referential hash omitted,2026-08-10T11:29:09+00:00
3,handoff_manifest_json,Compact final handoff manifest,True,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,True,True,unknown,2806,facde123a06796516dab7b927b9e8449109a3177727ce6...,NaN,,2026-08-10T11:29:09+00:00
4,stage_manifest_json,Shared stage manifest updated through Batch 6,True,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,True,True,unknown,4924,5bf31178d5a3dece2a2eeb6c21ff479267df419103d723...,NaN,,2026-08-10T11:29:09+00:00
5,figure_manifest_csv,Batch 4 strict report figure manifest,True,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,True,True,unknown,42011,d902ece74632bfe7a0a1b6001175776d50f28b6bbc06b9...,13.0,,2026-08-10T11:29:09+00:00
6,report_audit_csv,Candidate-level consolidated report audit,True,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,True,True,unknown,5481987,f352896cc14803eab9e2e8fc21b85512fe969070d992f6...,945.0,,2026-08-10T11:29:09+00:00
7,batch3_summary_csv,Compact Batch 3 summary,True,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,True,True,unknown,7939,ff5e4114f58fb043613bd8f199442d81159124456a52b1...,132.0,,2026-08-10T11:29:09+00:00
8,representative_candidates_csv,Representative candidate selection table,True,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,True,True,unknown,6082,a1e7cd3b49ceb901cb279aa79ee23d027c77ef8ae3b0fe...,8.0,,2026-08-10T11:29:09+00:00


In [37]:
# Batch 6 repair - Fix self-referential artifact_index_csv row, then rerun Cell 5
artifact_index_path = BATCH6_ARTIFACT_INDEX_PATH.resolve()

batch6_repaired_artifact_index_df = pd.read_csv(artifact_index_path)

self_mask = batch6_repaired_artifact_index_df["artifact_id"].astype(str).eq("artifact_index_csv")

batch6_repaired_artifact_index_df.loc[self_mask, "path"] = batch6_rel(artifact_index_path)
batch6_repaired_artifact_index_df.loc[self_mask, "absolute_path"] = str(artifact_index_path)
batch6_repaired_artifact_index_df.loc[self_mask, "exists"] = artifact_index_path.is_file()
batch6_repaired_artifact_index_df.loc[self_mask, "is_file"] = artifact_index_path.is_file()
batch6_repaired_artifact_index_df.loc[self_mask, "file_type"] = artifact_index_path.suffix.lower().lstrip(".")
batch6_repaired_artifact_index_df.loc[self_mask, "size_bytes"] = int(artifact_index_path.stat().st_size)
batch6_repaired_artifact_index_df.loc[self_mask, "sha256"] = ""
batch6_repaired_artifact_index_df.loc[self_mask, "row_count"] = int(len(batch6_repaired_artifact_index_df))
batch6_repaired_artifact_index_df.loc[self_mask, "hash_note"] = "self-referential hash omitted"
batch6_repaired_artifact_index_df.loc[self_mask, "indexed_at_utc"] = batch6_utc_now_iso()

batch6_repaired_artifact_index_df.to_csv(artifact_index_path, index=False)

display(batch6_repaired_artifact_index_df)

print("Repaired artifact_index_csv self-row. Now rerun Batch 6 / Cell 5.")

,artifact_id,artifact_role,required_for_handoff,path,absolute_path,exists,is_file,file_type,size_bytes,sha256,row_count,hash_note,indexed_at_utc
0,standalone_html_report,Final standalone Stable Diffusion HTML report,True,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,True,True,unknown,2426923,164c1649c003348668b0b3b997eb6f481440c3622bb15e...,NaN,NaN,2026-08-10T11:29:09+00:00
1,final_validation_csv,Batch 6 final validation table,True,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,True,True,unknown,2170,e2771e68514914229f0d931b212ae44a3735b15ab744be...,16.0,NaN,2026-08-10T11:29:09+00:00
2,artifact_index_csv,Machine-readable handoff artifact index,True,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,True,True,csv,4359,,9.0,self-referential hash omitted,2026-08-10T11:29:09+00:00
3,handoff_manifest_json,Compact final handoff manifest,True,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,True,True,unknown,2806,facde123a06796516dab7b927b9e8449109a3177727ce6...,NaN,NaN,2026-08-10T11:29:09+00:00
4,stage_manifest_json,Shared stage manifest updated through Batch 6,True,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,True,True,unknown,4924,5bf31178d5a3dece2a2eeb6c21ff479267df419103d723...,NaN,NaN,2026-08-10T11:29:09+00:00
5,figure_manifest_csv,Batch 4 strict report figure manifest,True,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,True,True,unknown,42011,d902ece74632bfe7a0a1b6001175776d50f28b6bbc06b9...,13.0,NaN,2026-08-10T11:29:09+00:00
6,report_audit_csv,Candidate-level consolidated report audit,True,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,True,True,unknown,5481987,f352896cc14803eab9e2e8fc21b85512fe969070d992f6...,945.0,NaN,2026-08-10T11:29:09+00:00
7,batch3_summary_csv,Compact Batch 3 summary,True,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,True,True,unknown,7939,ff5e4114f58fb043613bd8f199442d81159124456a52b1...,132.0,NaN,2026-08-10T11:29:09+00:00
8,representative_candidates_csv,Representative candidate selection table,True,D:\Masters\FH\Thesis\painting-restoration-eval...,D:\Masters\FH\Thesis\painting-restoration-eval...,True,True,unknown,6082,a1e7cd3b49ceb901cb279aa79ee23d027c77ef8ae3b0fe...,8.0,NaN,2026-08-10T11:29:09+00:00


Repaired artifact_index_csv self-row. Now rerun Batch 6 / Cell 5.


In [38]:
# Batch 6 / Cell 5 - Final strict handoff check
batch6_reload_validation_df = pd.read_csv(BATCH6_FINAL_VALIDATION_PATH)
batch6_reload_artifact_index_df = pd.read_csv(BATCH6_ARTIFACT_INDEX_PATH)
batch6_reload_handoff_manifest = batch6_read_json_if_exists(BATCH6_HANDOFF_MANIFEST_PATH)
batch6_reload_stage_manifest = batch6_read_json_if_exists(BATCH6_STAGE_MANIFEST_PATH)

required_final_output_paths = [
    BATCH6_FINAL_VALIDATION_PATH,
    BATCH6_ARTIFACT_INDEX_PATH,
    BATCH6_HANDOFF_MANIFEST_PATH,
    BATCH6_STAGE_MANIFEST_PATH,
]

required_artifact_ids = {
    "standalone_html_report",
    "final_validation_csv",
    "artifact_index_csv",
    "handoff_manifest_json",
    "stage_manifest_json",
    "figure_manifest_csv",
    "report_audit_csv",
    "batch3_summary_csv",
    "representative_candidates_csv",
}

artifact_ids_present = set(batch6_reload_artifact_index_df["artifact_id"].astype(str))
missing_artifact_ids = sorted(required_artifact_ids - artifact_ids_present)

batch6_final_handoff_rows = [
    batch6_validation_row(
        "batch6_output_files_exist",
        sum(path.is_file() for path in required_final_output_paths),
        len(required_final_output_paths),
        all(path.is_file() for path in required_final_output_paths),
        "One or more Batch 6 output files are missing.",
    ),
    batch6_validation_row(
        "batch6_validation_all_passed",
        int(batch6_bool_series(batch6_reload_validation_df["passed"]).sum()),
        len(batch6_reload_validation_df),
        bool(batch6_bool_series(batch6_reload_validation_df["passed"]).all()),
        "Final validation CSV contains failed checks.",
    ),
    batch6_validation_row(
        "artifact_index_contains_required_ids",
        missing_artifact_ids,
        [],
        len(missing_artifact_ids) == 0,
        f"Artifact index missing required IDs: {missing_artifact_ids}",
    ),
    batch6_validation_row(
        "artifact_index_required_files_exist",
        int(batch6_bool_series(batch6_reload_artifact_index_df["exists"]).sum()),
        len(batch6_reload_artifact_index_df),
        bool(batch6_bool_series(batch6_reload_artifact_index_df["exists"]).all()),
        "Artifact index contains missing required artifacts.",
    ),
    batch6_validation_row(
        "handoff_manifest_status_passed",
        batch6_reload_handoff_manifest.get("handoff_status", ""),
        "passed",
        batch6_reload_handoff_manifest.get("handoff_status", "") == "passed",
        "Handoff manifest is not marked passed.",
    ),
    batch6_validation_row(
        "stage_manifest_batch6_status_passed",
        batch6_reload_stage_manifest.get("batch6", {}).get("status", ""),
        "passed",
        batch6_reload_stage_manifest.get("batch6", {}).get("status", "") == "passed",
        "Stage manifest Batch 6 status is not marked passed.",
    ),
]

batch6_final_handoff_df = pd.DataFrame(batch6_final_handoff_rows)
batch6_final_handoff_passed = bool(batch6_bool_series(batch6_final_handoff_df["passed"]).all())

print(f"Final handoff checks passed: {int(batch6_bool_series(batch6_final_handoff_df['passed']).sum())} / {len(batch6_final_handoff_df)}")
display(batch6_final_handoff_df)

if not batch6_final_handoff_passed:
    display(
        batch6_final_handoff_df.loc[
            ~batch6_bool_series(batch6_final_handoff_df["passed"]),
            ["check_name", "actual", "expected", "failure_message"],
        ]
    )
    raise RuntimeError("Batch 6 final handoff check failed. Inspect failed rows before treating Notebook 15 as complete.")

print("Batch 6 passed. Stable Diffusion report validation, artifact index, and handoff manifest are complete.")

Final handoff checks passed: 6 / 6


,check_name,severity,actual,expected,passed,failure_message,checked_at_utc
0,batch6_output_files_exist,error,4,4,True,,2026-08-10T11:29:09.717444+00:00
1,batch6_validation_all_passed,error,16,16,True,,2026-08-10T11:29:09.717444+00:00
2,artifact_index_contains_required_ids,error,[],[],True,,2026-08-10T11:29:09.717444+00:00
3,artifact_index_required_files_exist,error,9,9,True,,2026-08-10T11:29:09.719451+00:00
4,handoff_manifest_status_passed,error,"""passed""","""passed""",True,,2026-08-10T11:29:09.719451+00:00
5,stage_manifest_batch6_status_passed,error,"""passed""","""passed""",True,,2026-08-10T11:29:09.719451+00:00


Batch 6 passed. Stable Diffusion report validation, artifact index, and handoff manifest are complete.
